<a href="https://colab.research.google.com/github/Tamim480-hub/hccd-bangla-deblurring/blob/main/ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import re
import random
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

GT_FOLDER = "Koushik_HCCD/HCCD DATASET/Ground Truth/"
BLUR_FOLDER = (
    "Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_level_1/"
)

PATCH_SIZE = 128
BATCH_SIZE = 8
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_names = z.namelist()

def collect_images(folder, prefix):
    result = {}

    for name in all_names:
        if not name.startswith(folder):
            continue

        filename = name.rsplit("/", 1)[-1]
        match = re.fullmatch(
            re.escape(prefix) + r" \((\d+)\)\.jpg",
            filename,
            re.IGNORECASE
        )

        if match:
            result[int(match.group(1))] = name

    return result

gt_images = collect_images(GT_FOLDER, "Ground_Truth")
blur_images = collect_images(BLUR_FOLDER, "Blur_level_1")

pair_ids = sorted(set(gt_images) & set(blur_images))
random.shuffle(pair_ids)

split = int(0.8 * len(pair_ids))
train_ids = pair_ids[:split]
val_ids = pair_ids[split:]

print("Total pairs:", len(pair_ids))
print("Training pairs:", len(train_ids))
print("Validation pairs:", len(val_ids))

to_tensor = transforms.ToTensor()

In [ ]:
from google.colab import drive
import torch
import os

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Koushik_HCCD.zip'

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Dataset ZIP exists:", os.path.exists(zip_path))

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:

import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

if not os.path.exists(ZIP_PATH):
    from google.colab import drive
    drive.mount("/content/drive")

assert os.path.exists(ZIP_PATH), (
    "Dataset ZIP পাওয়া যায়নি। Google Drive path পরীক্ষা করো।"
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

    ground_truth = [
        f for f in files
        if "/Ground Truth/" in f
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    blur_images = [
        f for f in files
        if "/Blur/Blur_level_1/" in f
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

print("GPU:", __import__("torch").cuda.get_device_name(0))
print("Ground Truth:", len(ground_truth))
print("Blur Level 1:", len(blur_images))
print("\nGround Truth examples:")
print(*ground_truth[:3], sep="\n")
print("\nBlur examples:")
print(*blur_images[:3], sep="\n")


In [ ]:

import zipfile
import re
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

    gt_files = {
        int(re.search(r"Ground_Truth \((\d+)\)\.jpg$", f).group(1)): f
        for f in files
        if "/Ground Truth/Ground_Truth (" in f
        and re.search(r"Ground_Truth \((\d+)\)\.jpg$", f)
    }

    blur_files = {
        int(re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f).group(1)): f
        for f in files
        if "/Blur/Blur_level_1/Blur_level_1 (" in f
        and re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f)
    }

    common_ids = sorted(set(gt_files) & set(blur_files))
    gt_only = sorted(set(gt_files) - set(blur_files))
    blur_only = sorted(set(blur_files) - set(gt_files))

    print("Matched pairs:", len(common_ids))
    print("Ground Truth without pair:", len(gt_only))
    print("Blur without pair:", len(blur_only))

    if common_ids:
        sample_ids = common_ids[:3]
        fig, axes = plt.subplots(len(sample_ids), 2, figsize=(10, 4 * len(sample_ids)))

        if len(sample_ids) == 1:
            axes = [axes]

        for row, idx in enumerate(sample_ids):
            blur = Image.open(BytesIO(z.read(blur_files[idx]))).convert("RGB")
            gt = Image.open(BytesIO(z.read(gt_files[idx]))).convert("RGB")

            axes[row][0].imshow(blur)
            axes[row][0].set_title(f"Blur Level 1 — ID {idx}")
            axes[row][0].axis("off")

            axes[row][1].imshow(gt)
            axes[row][1].set_title(f"Ground Truth — ID {idx}")
            axes[row][1].axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:

import zipfile
import re
import random
import torch
import numpy as np
from io import BytesIO
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 128
BATCH_SIZE = 8

class HandwritingDataset(Dataset):
    def __init__(self, zip_path, ids, gt_files, blur_files):
        self.zip_path = zip_path
        self.ids = ids
        self.gt_files = gt_files
        self.blur_files = blur_files
        self.transform = transforms.Compose([
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]

        with zipfile.ZipFile(self.zip_path, "r") as z:
            blur = Image.open(
                BytesIO(z.read(self.blur_files[image_id]))
            ).convert("RGB")
            gt = Image.open(
                BytesIO(z.read(self.gt_files[image_id]))
            ).convert("RGB")

        # Apply identical geometric crop to both images
        from torchvision.transforms import functional as TF
        from torchvision.transforms import RandomCrop

        i, j, h, w = RandomCrop.get_params(
            blur, output_size=(PATCH_SIZE, PATCH_SIZE)
        )
        blur = TF.crop(blur, i, j, h, w)
        gt = TF.crop(gt, i, j, h, w)

        return self.transform(blur), self.transform(gt)

# Create filename-to-ID dictionaries
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

gt_files = {
    int(re.search(r"Ground_Truth \((\d+)\)\.jpg$", f).group(1)): f
    for f in files
    if "/Ground Truth/Ground_Truth (" in f
    and re.search(r"Ground_Truth \((\d+)\)\.jpg$", f)
}

blur_files = {
    int(re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f).group(1)): f
    for f in files
    if "/Blur/Blur_level_1/Blur_level_1 (" in f
    and re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f)
}

paired_ids = sorted(set(gt_files) & set(blur_files))
random.seed(42)
random.shuffle(paired_ids)

split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

train_ds = HandwritingDataset(
    ZIP_PATH, train_ids, gt_files, blur_files
)
val_ds = HandwritingDataset(
    ZIP_PATH, val_ids, gt_files, blur_files
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print("Training pairs:", len(train_ds))
print("Validation pairs:", len(val_ds))

blur_batch, clean_batch = next(iter(train_loader))
print("Blur batch shape:", blur_batch.shape)
print("Clean batch shape:", clean_batch.shape)
print("Device:", torch.cuda.get_device_name(0))


In [ ]:

!pip -q install diffusers accelerate

import torch
from diffusers import UNet2DModel, DDPMScheduler

device = torch.device("cuda")

model = UNet2DModel(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4
)

print("Model initialized successfully!")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

%pip uninstall -y torchao
%pip install -U diffusers accelerate


In [ ]:

import torch
import diffusers

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

from diffusers import UNet2DModel, DDPMScheduler

print("Diffusion model imports successfully!")


In [ ]:

import torch
from diffusers import UNet2DModel, DDPMScheduler

device = torch.device("cuda")
PATCH_SIZE = 128

model = UNet2DModel(
    sample_size=PATCH_SIZE,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

print("Model initialized!")
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))


In [ ]:

blur_batch, clean_batch = next(iter(train_loader))

blur_batch = blur_batch.to(device)
clean_batch = clean_batch.to(device)

timesteps = torch.randint(
    0,
    noise_scheduler.config.num_train_timesteps,
    (clean_batch.shape[0],),
    device=device
).long()

noise = torch.randn_like(clean_batch)

noisy_clean = noise_scheduler.add_noise(
    clean_batch, noise, timesteps
)

model_input = torch.cat(
    [noisy_clean, blur_batch], dim=1
)

with torch.no_grad():
    predicted_noise = model(
        model_input, timesteps
    ).sample

print("Input shape:", model_input.shape)
print("Predicted noise shape:", predicted_noise.shape)
print("Batch test successful!")


In [ ]:

import os
import re
import zipfile
import random
from io import BytesIO

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import RandomCrop

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 128
BATCH_SIZE = 8

assert torch.cuda.is_available(), "আগে T4 GPU চালু করো!"
assert os.path.exists(ZIP_PATH), "Dataset ZIP পাওয়া যায়নি!"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

gt_files = {
    int(re.search(r"Ground_Truth \((\d+)\)\.jpg$", f).group(1)): f
    for f in files
    if "/Ground Truth/Ground_Truth (" in f
    and re.search(r"Ground_Truth \((\d+)\)\.jpg$", f)
}

blur_files = {
    int(re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f).group(1)): f
    for f in files
    if "/Blur/Blur_level_1/Blur_level_1 (" in f
    and re.search(r"Blur_level_1 \((\d+)\)\.jpg$", f)
}

paired_ids = sorted(set(gt_files) & set(blur_files))
assert len(paired_ids) > 0, "কোনো matching image পাওয়া যায়নি!"

random.seed(42)
random.shuffle(paired_ids)

split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

class HandwritingDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        idx = self.ids[index]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur_files[idx]))
            ).convert("RGB")
            gt = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        # Resize both images to the same size first
        blur = transforms.Resize((PATCH_SIZE, PATCH_SIZE))(blur)
        gt = transforms.Resize((PATCH_SIZE, PATCH_SIZE))(gt)

        blur = self.transform(blur)
        gt = self.transform(gt)

        return blur, gt

train_ds = HandwritingDataset(train_ids)
val_ds = HandwritingDataset(val_ids)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print("Matched pairs:", len(paired_ids))
print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))
print("train_loader created successfully!")


In [ ]:

blur_batch, clean_batch = next(iter(train_loader))

device = torch.device("cuda")
blur_batch = blur_batch.to(device)
clean_batch = clean_batch.to(device)

print("Blur batch:", blur_batch.shape)
print("Clean batch:", clean_batch.shape)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
print("Model exists:", "model" in globals())
print("Scheduler exists:", "noise_scheduler" in globals())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:

model.train()

blur, clean = next(iter(train_loader))
blur = blur.to(device)
clean = clean.to(device)

noise = torch.randn_like(clean)

timesteps = torch.randint(
    0,
    noise_scheduler.config.num_train_timesteps,
    (clean.shape[0],),
    device=device
).long()

noisy_clean = noise_scheduler.add_noise(
    clean, noise, timesteps
)

model_input = torch.cat([noisy_clean, blur], dim=1)

with torch.no_grad():
    with torch.autocast("cuda", dtype=torch.float16):
        predicted_noise = model(model_input, timesteps).sample

print("Input:", model_input.shape)
print("Predicted:", predicted_noise.shape)
print("Batch test successful!")
print("GPU memory (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))


In [ ]:

import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Check that all required objects exist
assert "train_loader" in globals(), "আগে Dataset Loader চালাও।"
assert "model" in globals(), "আগে Model initialization চালাও।"
assert "noise_scheduler" in globals(), "Scheduler তৈরি হয়নি।"

device = torch.device("cuda")
model = model.to(device)
model.train()

# Save checkpoints in Google Drive
SAVE_DIR = "/content/drive/MyDrive/handwriting_diffusion"
os.makedirs(SAVE_DIR, exist_ok=True)

EPOCHS = 3
scaler = torch.amp.GradScaler("cuda")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for blur, clean in progress:
        blur = blur.to(device)
        clean = clean.to(device)

        # Add random noise to clean handwriting
        noise = torch.randn_like(clean)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean.shape[0],),
            device=device
        ).long()

        noisy_clean = noise_scheduler.add_noise(
            clean, noise, timesteps
        )

        # Condition on the corresponding blurred image
        model_input = torch.cat([noisy_clean, blur], dim=1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(model_input, timesteps).sample
            loss = F.mse_loss(predicted_noise.float(), noise.float())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} average training loss: {avg_loss:.6f}")

    checkpoint_path = os.path.join(
        SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pt"
    )

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": avg_loss
    }, checkpoint_path)

    print("Checkpoint saved:", checkpoint_path)

print("Initial training completed!")


In [ ]:

import torch
import matplotlib.pyplot as plt

device = torch.device("cuda")
model.eval()

# Select one validation image
blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

# Start the reverse diffusion process from random noise
noise_scheduler.set_timesteps(50, device=device)
generated = torch.randn_like(clean)

with torch.no_grad():
    for t in noise_scheduler.timesteps:
        # Use the blurred image as conditioning
        model_input = torch.cat([generated, blur], dim=1)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, t.expand(1)
            ).sample

        generated = noise_scheduler.step(
            predicted_noise.float(),
            t,
            generated.float()
        ).prev_sample

# Convert tensors from [-1, 1] to [0, 1]
def to_image(tensor):
    return (
        tensor[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

blur_img = to_image(blur)
clean_img = to_image(clean)
restored_img = to_image(generated)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(blur_img)
axes[0].set_title("Blurred Input")

axes[1].imshow(clean_img)
axes[1].set_title("Ground Truth")

axes[2].imshow(restored_img)
axes[2].set_title("Diffusion Restored")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from diffusers import UNet2DModel, DDPMScheduler

device = torch.device("cuda")

model = UNet2DModel(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

ckpt_path = (
    "/content/drive/MyDrive/handwriting_diffusion/"
    "checkpoint_epoch_3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

print("Checkpoint loaded successfully!")

In [ ]:

import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = torch.device("cuda")
model.eval()

# Validation image
blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

# Reverse diffusion
noise_scheduler.set_timesteps(100, device=device)

generated = torch.randn_like(clean)

with torch.no_grad():
    for t in noise_scheduler.timesteps:
        # Noisy generated image + blurred condition
        model_input = torch.cat([generated, blur], dim=1)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, t.expand(1)
            ).sample

        generated = noise_scheduler.step(
            predicted_noise.float(),
            t,
            generated.float()
        ).prev_sample

def to_image(tensor):
    return (
        tensor[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(to_image(blur))
axes[0].set_title("Blurred Input")

axes[1].imshow(to_image(clean))
axes[1].set_title("Ground Truth")

axes[2].imshow(to_image(generated))
axes[2].set_title("Diffusion Restored")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

device = torch.device("cuda")
SAVE_DIR = "/content/drive/MyDrive/handwriting_diffusion"

# Restore optimizer and AMP scaler states
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

checkpoint = torch.load(
    os.path.join(SAVE_DIR, "checkpoint_epoch_3.pt"),
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

if "scaler_state_dict" in checkpoint:
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

model.to(device)
start_epoch = checkpoint["epoch"]

for epoch in range(start_epoch, 15):
    model.train()
    total_loss = 0.0

    for blur, clean in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/15"
    ):
        blur = blur.to(device)
        clean = clean.to(device)

        noise = torch.randn_like(clean)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean.shape[0],),
            device=device
        ).long()

        noisy_clean = noise_scheduler.add_noise(
            clean, noise, timesteps
        )
        model_input = torch.cat([noisy_clean, blur], dim=1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, timesteps
            ).sample
            loss = F.mse_loss(
                predicted_noise.float(), noise.float()
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": avg_loss
    }, os.path.join(
        SAVE_DIR, f"checkpoint_epoch_{epoch + 1}.pt"
    ))

    print(
        f"Epoch {epoch + 1}/15 | "
        f"Average Loss: {avg_loss:.6f}"
    )


In [ ]:

import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

device = torch.device("cuda")
SAVE_DIR = "/content/drive/MyDrive/handwriting_diffusion"

# Restore optimizer and AMP scaler states
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

checkpoint = torch.load(
    os.path.join(SAVE_DIR, "checkpoint_epoch_3.pt"),
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

if "scaler_state_dict" in checkpoint:
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

model.to(device)
start_epoch = checkpoint["epoch"]

for epoch in range(start_epoch, 15):
    model.train()
    total_loss = 0.0

    for blur, clean in tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/15"
    ):
        blur = blur.to(device)
        clean = clean.to(device)

        noise = torch.randn_like(clean)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean.shape[0],),
            device=device
        ).long()

        noisy_clean = noise_scheduler.add_noise(
            clean, noise, timesteps
        )
        model_input = torch.cat([noisy_clean, blur], dim=1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, timesteps
            ).sample
            loss = F.mse_loss(
                predicted_noise.float(), noise.float()
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": avg_loss
    }, os.path.join(
        SAVE_DIR, f"checkpoint_epoch_{epoch + 1}.pt"
    ))

    print(
        f"Epoch {epoch + 1}/15 | "
        f"Average Loss: {avg_loss:.6f}"
    )


In [ ]:

import os
import torch
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda")

# Load latest checkpoint
checkpoint_path = (
    "/content/drive/MyDrive/handwriting_diffusion/"
    "checkpoint_epoch_15.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print("Training loss:", checkpoint["loss"])

# Select validation sample
blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

# Diffusion sampling
noise_scheduler.set_timesteps(100, device=device)
generated = torch.randn_like(clean)

with torch.no_grad():
    for t in noise_scheduler.timesteps:
        model_input = torch.cat([generated, blur], dim=1)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, t.expand(1)
            ).sample

        generated = noise_scheduler.step(
            predicted_noise.float(),
            t,
            generated.float()
        ).prev_sample

def to_image(t):
    return (
        t[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

axes[0].imshow(to_image(blur))
axes[0].set_title("Blurred Input")

axes[1].imshow(to_image(clean))
axes[1].set_title("Ground Truth")

axes[2].imshow(to_image(generated))
axes[2].set_title("Restored Output")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
import torch
import matplotlib.pyplot as plt

SAVE_DIR = "/content/drive/MyDrive/handwriting_diffusion"

epochs = []
losses = []

for epoch in range(1, 16):
    path = os.path.join(
        SAVE_DIR, f"checkpoint_epoch_{epoch}.pt"
    )

    if os.path.exists(path):
        ckpt = torch.load(
            path, map_location="cpu", weights_only=False
        )
        epochs.append(epoch)
        losses.append(ckpt["loss"])

print("Epochs:", epochs)
print("Losses:", [round(x, 5) for x in losses])

plt.figure(figsize=(8, 4))
plt.plot(epochs, losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Conditional Diffusion Training Loss")
plt.grid(True)
plt.show()


In [ ]:

import torch
import matplotlib.pyplot as plt

device = torch.device("cuda")
model.eval()

blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

# Test noise prediction at a fixed timestep
t = torch.tensor([500], device=device)
noise = torch.randn_like(clean)
noisy_clean = noise_scheduler.add_noise(clean, noise, t)

with torch.no_grad():
    inp_correct = torch.cat([noisy_clean, blur], dim=1)
    inp_zero = torch.cat([
        noisy_clean, torch.zeros_like(blur)
    ], dim=1)

    with torch.autocast("cuda", dtype=torch.float16):
        pred_correct = model(inp_correct, t).sample
        pred_zero = model(inp_zero, t).sample

correct_mse = torch.nn.functional.mse_loss(
    pred_correct.float(), noise.float()
).item()

zero_mse = torch.nn.functional.mse_loss(
    pred_zero.float(), noise.float()
).item()

print("MSE with blurred condition:", round(correct_mse, 5))
print("MSE with zero condition:", round(zero_mse, 5))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(blur[0].cpu().permute(1, 2, 0).clamp(-1,1)*0.5+0.5)
plt.title("Blurred condition")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(clean[0].cpu().permute(1, 2, 0).clamp(-1,1)*0.5+0.5)
plt.title("Ground truth")
plt.axis("off")
plt.show()


In [ ]:

import torch
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler

device = torch.device("cuda")
model.eval()

# Use the same noise schedule as training
ddim_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon",
    clip_sample=True
)

blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

ddim_scheduler.set_timesteps(100, device=device)

# Start from Gaussian noise
torch.manual_seed(42)
generated = torch.randn_like(clean)

with torch.no_grad():
    for t in ddim_scheduler.timesteps:
        model_input = torch.cat([generated, blur], dim=1)

        with torch.autocast("cuda", dtype=torch.float16):
            predicted_noise = model(
                model_input, t.expand(1)
            ).sample

        generated = ddim_scheduler.step(
            predicted_noise.float(),
            t,
            generated.float(),
            eta=0.0
        ).prev_sample

def to_image(x):
    return (
        x[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(to_image(blur))
axes[0].set_title("Blurred Input")

axes[1].imshow(to_image(clean))
axes[1].set_title("Ground Truth")

axes[2].imshow(to_image(generated))
axes[2].set_title("DDIM Restored")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import matplotlib.pyplot as plt

device = torch.device("cuda")
model.eval()

blur, clean = val_ds[0]
blur = blur.unsqueeze(0).to(device)
clean = clean.unsqueeze(0).to(device)

# Test at a fixed noise level
t = torch.tensor([500], device=device)
torch.manual_seed(42)
noise = torch.randn_like(clean)

noisy_clean = noise_scheduler.add_noise(clean, noise, t)

with torch.no_grad():
    model_input = torch.cat([noisy_clean, blur], dim=1)

    with torch.autocast("cuda", dtype=torch.float16):
        predicted_noise = model(model_input, t).sample

# Estimate the original clean image from noisy image
alpha_bar = noise_scheduler.alphas_cumprod[t.item()].to(device)
alpha_bar = alpha_bar.float()

estimated_clean = (
    noisy_clean.float()
    - torch.sqrt(1 - alpha_bar) * predicted_noise.float()
) / torch.sqrt(alpha_bar)

def show_image(x):
    return (
        x[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(show_image(blur))
axes[0].set_title("Blur Input")

axes[1].imshow(show_image(noisy_clean))
axes[1].set_title("Noisy Ground Truth")

axes[2].imshow(show_image(estimated_clean))
axes[2].set_title("Estimated Clean")

axes[3].imshow(show_image(clean))
axes[3].set_title("Actual Ground Truth")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print(
    "Estimated-clean MSE:",
    torch.nn.functional.mse_loss(
        estimated_clean.clamp(-1, 1), clean
    ).item()
)


In [ ]:

%pip -q install scikit-image


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def image_array(t):
    return (
        t[0].detach().float().cpu()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

blur_img = image_array(blur)
clean_img = image_array(clean)
estimated_img = image_array(estimated_clean)

def evaluate(name, pred, target):
    psnr = peak_signal_noise_ratio(
        target, pred, data_range=1.0
    )
    ssim = structural_similarity(
        target, pred, data_range=1.0, channel_axis=2
    )
    print(f"{name}: PSNR={psnr:.2f} dB | SSIM={ssim:.4f}")

evaluate("Blur Input", blur_img, clean_img)
evaluate("Estimated Clean", estimated_img, clean_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(
    axes,
    [blur_img, estimated_img, clean_img],
    ["Blur Input", "Estimated Clean", "Ground Truth"]
):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Use the same validation sample
blur, clean = val_ds[0]

def display_tensor(t):
    return (
        t.detach().cpu().float()
        .clamp(-1, 1).add(1).div(2)
        .permute(1, 2, 0).numpy()
    )

blur_img = display_tensor(blur)
clean_img = display_tensor(clean)

print("Blur shape:", blur_img.shape)
print("Clean shape:", clean_img.shape)
print(
    "Mean absolute pixel difference:",
    np.mean(np.abs(blur_img - clean_img))
)
print(
    "PSNR:",
    peak_signal_noise_ratio(clean_img, blur_img, data_range=1.0)
)
print(
    "SSIM:",
    structural_similarity(
        clean_img, blur_img, data_range=1.0, channel_axis=2
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(blur_img)
axes[0].set_title("Blur Level 1")
axes[1].imshow(clean_img)
axes[1].set_title("Ground Truth")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile
import re
from io import BytesIO
from PIL import Image
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

def load_rgb(z, path):
    return np.asarray(
        Image.open(BytesIO(z.read(path)))
        .convert("RGB")
        .resize((128, 128))
    ) / 255.0

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

    # Choose one image ID that exists in ground truth
    gt_path = next(
        p for p in names
        if re.search(r"Ground_Truth \(\d+\)\.jpg$", p)
        and "/Ground Truth/" in p
    )

    image_id = int(
        re.search(r"Ground_Truth \((\d+)\)\.jpg$", gt_path).group(1)
    )
    gt = load_rgb(z, gt_path)

    print("Image ID:", image_id)

    for level in range(1, 6):
        matches = [
            p for p in names
            if f"/Blur/Blur_level_{level}/" in p
            and re.search(
                rf"Blur_level_{level} \({image_id}\)\.jpg$", p
            )
        ]

        if not matches:
            print(f"Blur Level {level}: image not found")
            continue

        blurred = load_rgb(z, matches[0])
        psnr = peak_signal_noise_ratio(gt, blurred, data_range=1.0)
        ssim = structural_similarity(
            gt, blurred, data_range=1.0, channel_axis=2
        )
        mad = np.mean(np.abs(gt - blurred))

        print(
            f"Level {level}: PSNR={psnr:.2f} dB, "
            f"SSIM={ssim:.4f}, MAD={mad:.5f}"
        )


In [ ]:

import zipfile
import re
from io import BytesIO
from PIL import Image
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

    gt_files = {}
    blur_files = {}

    for p in names:
        m = re.search(r"Ground_Truth \((\d+)\)\.jpg$", p)
        if m and "/Ground Truth/" in p:
            gt_files[int(m.group(1))] = p

        m = re.search(r"Blur_level_1 \((\d+)\)\.jpg$", p)
        if m and "/Blur/Blur_level_1/" in p:
            blur_files[int(m.group(1))] = p

    ids = sorted(set(gt_files) & set(blur_files))[:50]

    psnrs, ssims, mads = [], [], []

    for idx in ids:
        gt = np.asarray(
            Image.open(BytesIO(z.read(gt_files[idx])))
            .convert("RGB").resize((128, 128))
        ) / 255.0

        blur = np.asarray(
            Image.open(BytesIO(z.read(blur_files[idx])))
            .convert("RGB").resize((128, 128))
        ) / 255.0

        psnrs.append(
            peak_signal_noise_ratio(gt, blur, data_range=1.0)
        )
        ssims.append(
            structural_similarity(
                gt, blur, data_range=1.0, channel_axis=2
            )
        )
        mads.append(np.mean(np.abs(gt - blur)))

print("Matched images:", len(ids))
print("Mean PSNR:", round(float(np.mean(psnrs)), 3))
print("Mean SSIM:", round(float(np.mean(ssims)), 5))
print("Mean MAD:", round(float(np.mean(mads)), 6))


In [ ]:

import zipfile

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    categories = sorted({
        "/".join(p.split("/")[-4:-1])
        for p in z.namelist()
        if p.lower().endswith((".jpg", ".jpeg", ".png"))
        and any(
            key in p.lower()
            for key in ["blur", "shadow", "illumination"]
        )
    })

for category in categories:
    print(category)


In [ ]:

import zipfile
import re
from io import BytesIO
from PIL import Image
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
SAMPLE_COUNT = 50

def get_id(path, pattern):
    match = re.search(pattern, path, re.IGNORECASE)
    return int(match.group(1)) if match else None

def read_image(z, path):
    return np.asarray(
        Image.open(BytesIO(z.read(path)))
        .convert("RGB")
        .resize((128, 128))
    ) / 255.0

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

    gt = {}
    for p in names:
        if "/Ground Truth/" in p:
            idx = get_id(p, r"Ground_Truth \((\d+)\)\.jpg$")
            if idx is not None:
                gt[idx] = p

    categories = {
        "Blur Level 1": (
            "Synthetic/Blur/Blur_level_1/",
            r"Blur_level_1 \((\d+)\)\.jpg$"
        ),
        "Blur Level 2": (
            "Synthetic/Blur/Blur_level_2/",
            r"Blur_level_2 \((\d+)\)\.jpg$"
        ),
        "Blur Level 3": (
            "Synthetic/Blur/Blur_Level_3/",
            r"Blur_Level_3 \((\d+)\)\.jpg$"
        ),
        "Low Illumination 1": (
            "Synthetic/Low illumination/Low_level_1/",
            r"Low_level_1 \((\d+)\)\.jpg$"
        ),
        "Low Illumination 2": (
            "Synthetic/Low illumination/Low_Level_2/",
            r"Low_Level_2 \((\d+)\)\.jpg$"
        ),
        "Low Illumination 3": (
            "Synthetic/Low illumination/Low_level_3/",
            r"Low_level_3 \((\d+)\)\.jpg$"
        ),
        "Shadow 1": (
            "Synthetic/Shadow/Shadow level 1/",
            r"Shadow level 1 \((\d+)\)\.jpg$"
        ),
        "Shadow 2": (
            "Synthetic/Shadow/Shadow level 2",
            r"Shadow level 2\w* \((\d+)\)\.jpg$"
        )
    }

    for label, (folder, pattern) in categories.items():
        files = {}

        for p in names:
            if folder.lower() in p.lower():
                idx = get_id(p, pattern)
                if idx is not None:
                    files[idx] = p

        ids = sorted(set(gt) & set(files))[:SAMPLE_COUNT]

        if not ids:
            print(f"{label}: no matched images found")
            continue

        psnrs, ssims, mads = [], [], []

        for idx in ids:
            target = read_image(z, gt[idx])
            degraded = read_image(z, files[idx])

            psnrs.append(
                peak_signal_noise_ratio(
                    target, degraded, data_range=1.0
                )
            )
            ssims.append(
                structural_similarity(
                    target, degraded,
                    data_range=1.0, channel_axis=2
                )
            )
            mads.append(np.mean(np.abs(target - degraded)))

        print(
            f"{label} | N={len(ids)} | "
            f"PSNR={np.mean(psnrs):.2f} dB | "
            f"SSIM={np.mean(ssims):.4f} | "
            f"MAD={np.mean(mads):.5f}"
        )


In [ ]:

import re
import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 128
BATCH_SIZE = 8

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

gt_files = {}
blur3_files = {}

for path in names:
    m = re.search(r"Ground_Truth \((\d+)\)\.jpg$", path)
    if m and "/Ground Truth/" in path:
        gt_files[int(m.group(1))] = path

    m = re.search(r"Blur_Level_3 \((\d+)\)\.jpg$", path, re.I)
    if m and "/Blur/Blur_Level_3/" in path:
        blur3_files[int(m.group(1))] = path

paired_ids = sorted(set(gt_files) & set(blur3_files))

print("Ground Truth images:", len(gt_files))
print("Blur Level 3 images:", len(blur3_files))
print("Matched pairs:", len(paired_ids))

if len(paired_ids) == 0:
    raise ValueError("No matching Blur Level 3 pairs found!")

class BlurLevel3Dataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
        self.transform = transforms.Compose([
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        idx = self.ids[index]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")
            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        return self.transform(blur), self.transform(clean)

import random
random.seed(42)
random.shuffle(paired_ids)

split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

train_ds = BlurLevel3Dataset(train_ids)
val_ds = BlurLevel3Dataset(val_ids)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

blur, clean = val_ds[0]

def show_tensor(t):
    return t.permute(1, 2, 0).numpy().clip(-1, 1) * 0.5 + 0.5

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(show_tensor(blur))
ax[0].set_title("Blur Level 3")
ax[1].imshow(show_tensor(clean))
ax[1].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Training pairs:", len(train_ds))
print("Validation pairs:", len(val_ds))


In [ ]:

import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.optim import AdamW
from tqdm.auto import tqdm
import os

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Fresh conditional diffusion model
model = UNet2DModel(
    sample_size=128,
    in_channels=6,   # 3 noisy image channels + 3 blur condition channels
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Mixed precision on GPU
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

EPOCHS = 30
SAVE_DIR = "/content/drive/MyDrive/handwriting_diffusion_level3"
os.makedirs(SAVE_DIR, exist_ok=True)

train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for blur, clean in progress:
        blur = blur.to(device)
        clean = clean.to(device)

        # Add random noise to clean target
        noise = torch.randn_like(clean)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean.shape[0],),
            device=device
        ).long()

        noisy_clean = noise_scheduler.add_noise(
            clean, noise, timesteps
        )

        # Condition on the blurred input
        model_input = torch.cat([noisy_clean, blur], dim=1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            noise_pred = model(
                model_input,
                timesteps
            ).sample

            loss = F.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for blur, clean in val_loader:
            blur = blur.to(device)
            clean = clean.to(device)

            noise = torch.randn_like(clean)
            timesteps = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (clean.shape[0],),
                device=device
            ).long()

            noisy_clean = noise_scheduler.add_noise(
                clean, noise, timesteps
            )

            model_input = torch.cat(
                [noisy_clean, blur], dim=1
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                noise_pred = model(
                    model_input, timesteps
                ).sample

                val_loss = F.mse_loss(noise_pred, noise)

            val_running_loss += val_loss.item()

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Validation Loss: {val_loss:.5f}"
    )

    # Save latest checkpoint every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses
    }

    torch.save(
        checkpoint,
        os.path.join(SAVE_DIR, "latest_checkpoint.pt")
    )

    # Save best validation checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            checkpoint,
            os.path.join(SAVE_DIR, "best_checkpoint.pt")
        )
        print("Best checkpoint saved!")

print("Training complete.")
print("Best validation loss:", best_val_loss)


In [ ]:

import torch
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load best checkpoint (not the latest one)
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/handwriting_diffusion_level3/"
    "best_checkpoint.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

# Use the same scheduler configuration as training
scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

# Get one validation example
blur, clean = val_ds[0]
blur_batch = blur.unsqueeze(0).to(device)

# Conditional diffusion sampling
scheduler.set_timesteps(100, device=device)

generator = torch.Generator(device=device).manual_seed(42)
sample = torch.randn(
    (1, 3, 128, 128),
    device=device,
    generator=generator
)

with torch.no_grad():
    for t in scheduler.timesteps:
        # Combine current noisy sample with blurred condition
        model_input = torch.cat([sample, blur_batch], dim=1)

        timestep_batch = t.expand(1).to(device)
        noise_pred = model(model_input, timestep_batch).sample

        sample = scheduler.step(
            noise_pred,
            t,
            sample,
            generator=generator
        ).prev_sample

restored = sample[0].detach().cpu().clamp(-1, 1)

def display_image(tensor):
    return (
        tensor.detach().cpu()
        .permute(1, 2, 0)
        .numpy()
        .clip(-1, 1) * 0.5 + 0.5
    )

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(display_image(blur))
plt.title("Input: Blur Level 3")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(display_image(restored))
plt.title("Restored: Conditional Diffusion")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(display_image(clean))
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

base_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".pt") and (
            "checkpoint" in file.lower()
            or "handwriting" in root.lower()
        ):
            print(os.path.join(root, file))


In [ ]:

import torch
import os

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "handwriting_diffusion_level3/best_checkpoint.pt"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded successfully!")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Device:", device)


In [ ]:

from diffusers import DDPMScheduler
import matplotlib.pyplot as plt
import torch

# Load the trained model weights
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()

scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

# Validation image
blur, clean = val_ds[0]
blur_batch = blur.unsqueeze(0).to(device)

# Sampling
scheduler.set_timesteps(100, device=device)

generator = torch.Generator(device=device).manual_seed(42)

sample = torch.randn(
    (1, 3, 128, 128),
    device=device,
    generator=generator
)

with torch.no_grad():
    for t in scheduler.timesteps:
        model_input = torch.cat([sample, blur_batch], dim=1)
        t_batch = t.expand(1).to(device)

        noise_pred = model(model_input, t_batch).sample

        sample = scheduler.step(
            noise_pred, t, sample, generator=generator
        ).prev_sample

restored = sample[0].cpu().clamp(-1, 1)

def to_image(t):
    return (
        t.detach().cpu().permute(1, 2, 0).numpy()
        .clip(-1, 1) * 0.5 + 0.5
    )

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(to_image(blur))
plt.title("Blur Level 3 Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(to_image(restored))
plt.title("Diffusion Restored")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(to_image(clean))
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
import torch
import matplotlib.pyplot as plt
from diffusers import UNet2DModel, DDPMScheduler

# 1. Device and checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "handwriting_diffusion_level3/best_checkpoint.pt"
)

assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

# 2. Re-create the SAME architecture used during training
model = UNet2DModel(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully!")

# 3. Ensure validation dataset exists
assert "val_ds" in globals(), (
    "val_ds is missing. Dataset setup must be run again."
)

blur, clean = val_ds[0]
blur_batch = blur.unsqueeze(0).to(device)

# 4. Conditional DDPM sampling
scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

scheduler.set_timesteps(100, device=device)

generator = torch.Generator(device=device).manual_seed(42)

sample = torch.randn(
    (1, 3, 128, 128),
    device=device,
    generator=generator
)

with torch.no_grad():
    for t in scheduler.timesteps:
        model_input = torch.cat(
            [sample, blur_batch], dim=1
        )

        t_batch = t.expand(1).to(device)
        noise_pred = model(model_input, t_batch).sample

        sample = scheduler.step(
            noise_pred,
            t,
            sample,
            generator=generator
        ).prev_sample

restored = sample[0].detach().cpu().clamp(-1, 1)

# 5. Display images
def to_image(tensor):
    return (
        tensor.detach().cpu()
        .permute(1, 2, 0)
        .numpy().clip(-1, 1) * 0.5 + 0.5
    )

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(to_image(blur))
plt.title("Blur Level 3 Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(to_image(restored))
plt.title("Conditional Diffusion Output")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(to_image(clean))
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

%pip uninstall -y torchao
%pip install -U diffusers accelerate


In [ ]:

import torch
import diffusers

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)

from diffusers import UNet2DModel, DDPMScheduler

print("Diffusion imports successful!")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import torch
import diffusers
import os
import re
import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Device:", device)
print("Checkpoint exists:", os.path.exists(
    "/content/drive/MyDrive/handwriting_diffusion_level3/best_checkpoint.pt"
))

import torch
import diffusers
import os
import re
import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Device:", device)
print("Checkpoint exists:", os.path.exists(
    "/content/drive/MyDrive/handwriting_diffusion_level3/best_checkpoint.pt"
))


In [ ]:

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 128
BATCH_SIZE = 8

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

gt_files = {}
blur3_files = {}

for path in names:
    m = re.search(r"Ground_Truth \((\d+)\)\.jpg$", path)
    if m and "/Ground Truth/" in path:
        gt_files[int(m.group(1))] = path

    m = re.search(r"Blur_Level_3 \((\d+)\)\.jpg$", path, re.I)
    if m and "/Blur/Blur_Level_3/" in path:
        blur3_files[int(m.group(1))] = path

paired_ids = sorted(set(gt_files) & set(blur3_files))

print("Ground Truth:", len(gt_files))
print("Blur Level 3:", len(blur3_files))
print("Matched pairs:", len(paired_ids))

assert len(paired_ids) > 0, "No matched image pairs found!"

import random
random.seed(42)
random.shuffle(paired_ids)

split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

class BlurLevel3Dataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
        self.transform = transforms.Compose([
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        idx = self.ids[index]
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")
            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        return self.transform(blur), self.transform(clean)

train_ds = BlurLevel3Dataset(train_ids)
val_ds = BlurLevel3Dataset(val_ids)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)

print("Training pairs:", len(train_ds))
print("Validation pairs:", len(val_ds))


In [ ]:

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "handwriting_diffusion_level3/best_checkpoint.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model = UNet2DModel(
    sample_size=128,
    in_channels=6,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D"
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D"
    ),
    norm_num_groups=8
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully!")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])


In [ ]:

scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

blur, clean = val_ds[0]
blur_batch = blur.unsqueeze(0).to(device)

scheduler.set_timesteps(100, device=device)

generator = torch.Generator(device=device).manual_seed(42)

sample = torch.randn(
    (1, 3, 128, 128),
    device=device,
    generator=generator
)

with torch.no_grad():
    for t in scheduler.timesteps:
        model_input = torch.cat([sample, blur_batch], dim=1)
        t_batch = t.expand(1).to(device)

        noise_pred = model(model_input, t_batch).sample

        sample = scheduler.step(
            noise_pred,
            t,
            sample,
            generator=generator
        ).prev_sample

restored = sample[0].cpu().clamp(-1, 1)

def to_image(t):
    return (
        t.detach().cpu().permute(1, 2, 0).numpy()
        .clip(-1, 1) * 0.5 + 0.5
    )

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(to_image(blur))
plt.title("Blur Level 3 Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(to_image(restored))
plt.title("Diffusion Restored")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(to_image(clean))
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon"
)

# Same validation example
blur, clean = val_ds[0]
blur_batch = blur.unsqueeze(0).to(device)

# Use all 1000 reverse steps
scheduler.set_timesteps(1000, device=device)

generator = torch.Generator(device=device).manual_seed(42)
sample = torch.randn(
    (1, 3, 128, 128),
    device=device,
    generator=generator
)

with torch.inference_mode():
    for t in scheduler.timesteps:
        model_input = torch.cat([sample, blur_batch], dim=1)
        t_batch = t.expand(1).to(device)

        noise_pred = model(model_input, t_batch).sample
        sample = scheduler.step(
            noise_pred, t, sample, generator=generator
        ).prev_sample

restored = sample[0].cpu().clamp(-1, 1)

def to_image(t):
    return (
        t.detach().cpu().permute(1, 2, 0).numpy()
        .clip(-1, 1) * 0.5 + 0.5
    )

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(to_image(blur))
ax[0].set_title("Blur Level 3 Input")
ax[1].imshow(to_image(restored))
ax[1].set_title("Restored - 1000 Steps")
ax[2].imshow(to_image(clean))
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

# A simple U-Net for paired image restoration
class RestorationUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.enc1 = block(3, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = block(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = block(64, 32)

        self.out = nn.Conv2d(32, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.tanh(self.out(d1))


# Fresh model; no diffusion checkpoint is loaded
restoration_model = RestorationUNet().to(device)

optimizer = torch.optim.AdamW(
    restoration_model.parameters(),
    lr=1e-4
)

epochs = 30
best_val = float("inf")

SAVE_DIR = "/content/drive/MyDrive/handwriting_unet"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(epochs):
    restoration_model.train()
    train_sum = 0.0

    for blur, clean in train_loader:
        blur = blur.to(device)
        clean = clean.to(device)

        optimizer.zero_grad(set_to_none=True)

        prediction = restoration_model(blur)
        loss = F.l1_loss(prediction, clean)

        loss.backward()
        optimizer.step()

        train_sum += loss.item()

    train_loss = train_sum / len(train_loader)

    restoration_model.eval()
    val_sum = 0.0

    with torch.no_grad():
        for blur, clean in val_loader:
            blur = blur.to(device)
            clean = clean.to(device)

            prediction = restoration_model(blur)
            val_sum += F.l1_loss(prediction, clean).item()

    val_loss = val_sum / len(val_loader)

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train L1: {train_loss:.5f} | "
        f"Val L1: {val_loss:.5f}"
    )

    if val_loss < best_val:
        best_val = val_loss

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": restoration_model.state_dict(),
                "val_loss": val_loss
            },
            os.path.join(SAVE_DIR, "best_unet.pt")
        )

print("Training complete. Best validation L1:", best_val)


In [ ]:

import os
import re
import zipfile
import random
from io import BytesIO
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ZIP_PATH = "/content/drive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 96
BATCH_SIZE = 16

# Read image paths from ZIP
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

gt_files = {}
blur3_files = {}

for path in names:
    m = re.search(r"Ground_Truth \((\d+)\)\.jpg$", path)
    if m and "/Ground Truth/" in path:
        gt_files[int(m.group(1))] = path

    m = re.search(r"Blur_Level_3 \((\d+)\)\.jpg$", path, re.I)
    if m and "/Blur/Blur_Level_3/" in path:
        blur3_files[int(m.group(1))] = path

paired_ids = sorted(set(gt_files) & set(blur3_files))
print("Matched pairs:", len(paired_ids))

assert len(paired_ids) > 0, "No matching images found!"

random.seed(42)
random.shuffle(paired_ids)

split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

class BlurLevel3Dataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
        self.transform = transforms.Compose([
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        idx = self.ids[index]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")

            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        return self.transform(blur), self.transform(clean)

train_ds = BlurLevel3Dataset(train_ids)
val_ds = BlurLevel3Dataset(val_ids)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from google.colab import drive

drive.mount('/content/drive', force_remount=True)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

zip_path = "/content/drive/MyDrive/Koushik_HCCD.zip"

print("Drive accessible:", os.path.isdir("/content/drive/MyDrive"))
print("Dataset ZIP exists:", os.path.isfile(zip_path))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os

zip_path = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

print("Drive accessible:", os.path.isdir("/content/gdrive/MyDrive"))
print("Dataset ZIP exists:", os.path.isfile(zip_path))

In [ ]:

import os
import re
import zipfile
import random
from io import BytesIO
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 96
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

gt_files = {}
blur3_files = {}

for path in names:
    m = re.search(r"Ground_Truth \((\d+)\)\.jpg$", path)
    if m and "/Ground Truth/" in path:
        gt_files[int(m.group(1))] = path

    m = re.search(r"Blur_Level_3 \((\d+)\)\.jpg$", path, re.I)
    if m and "/Blur/Blur_Level_3/" in path:
        blur3_files[int(m.group(1))] = path

paired_ids = sorted(set(gt_files) & set(blur3_files))
print("Ground truth:", len(gt_files))
print("Blur Level 3:", len(blur3_files))
print("Matched pairs:", len(paired_ids))

assert len(paired_ids) > 0, "No matching image pairs found. Check ZIP paths."

random.seed(42)
random.shuffle(paired_ids)
split = int(0.8 * len(paired_ids))
train_ids = paired_ids[:split]
val_ids = paired_ids[split:]

class BlurLevel3Dataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
        self.transform = transforms.Compose([
            transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        idx = self.ids[index]
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")
            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        return self.transform(blur), self.transform(clean)

train_ds = BlurLevel3Dataset(train_ids)
val_ds = BlurLevel3Dataset(val_ids)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=True
)

print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Dataset ready!")


In [ ]:

class FastRestorationUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.e1 = block(3, 16)
        self.e2 = block(16, 32)
        self.e3 = block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(64, 128)

        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d3 = block(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = block(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = block(32, 16)

        self.out = nn.Conv2d(16, 3, 1)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        z = self.mid(self.pool(c))

        z = self.d3(torch.cat([self.u3(z), c], dim=1))
        z = self.d2(torch.cat([self.u2(z), b], dim=1))
        z = self.d1(torch.cat([self.u1(z), a], dim=1))

        return torch.tanh(self.out(z))

restoration_model = FastRestorationUNet().to(device)
optimizer = torch.optim.AdamW(
    restoration_model.parameters(), lr=1e-4
)

print("U-Net model ready!")


In [ ]:

epochs = 10
best_val = float("inf")

SAVE_DIR = "/content/gdrive/MyDrive/handwriting_unet_fast"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(epochs):
    restoration_model.train()
    train_sum = 0.0

    for blur, clean in train_loader:
        blur = blur.to(device)
        clean = clean.to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = restoration_model(blur)
        loss = F.l1_loss(pred, clean)
        loss.backward()
        optimizer.step()

        train_sum += loss.item()

    train_loss = train_sum / len(train_loader)

    restoration_model.eval()
    val_sum = 0.0

    with torch.no_grad():
        for blur, clean in val_loader:
            blur = blur.to(device)
            clean = clean.to(device)
            pred = restoration_model(blur)
            val_sum += F.l1_loss(pred, clean).item()

    val_loss = val_sum / len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": restoration_model.state_dict(),
            "val_loss": val_loss
        }, os.path.join(SAVE_DIR, "best_unet.pt"))

print("Training finished. Best validation loss:", best_val)


In [ ]:

import matplotlib.pyplot as plt
import torch

checkpoint_path = (
    "/content/gdrive/MyDrive/handwriting_unet_fast/best_unet.pt"
)

checkpoint = torch.load(checkpoint_path, map_location=device)
restoration_model.load_state_dict(checkpoint["model_state_dict"])
restoration_model.eval()

blur, clean = val_ds[0]

with torch.no_grad():
    prediction = restoration_model(blur.unsqueeze(0).to(device))
    prediction = prediction.squeeze(0).cpu()

def show_image(tensor):
    tensor = tensor.detach().cpu().clamp(-1, 1)
    tensor = (tensor + 1) / 2
    return tensor.permute(1, 2, 0).numpy()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(show_image(blur))
plt.title("Blurred Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(show_image(prediction))
plt.title("U-Net Restored")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(show_image(clean))
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(3, 2, figsize=(9, 10))

for row in range(3):
    idx = random.randrange(len(val_ds))
    blur, clean = val_ds[idx]

    axes[row, 0].imshow(show_image(blur))
    axes[row, 0].set_title("Blurred Input")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(show_image(clean))
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

path = "/content/gdrive/MyDrive/handwriting_unet_fast/best_unet.pt"
print("Checkpoint exists:", os.path.isfile(path))

In [ ]:

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = (
    "/content/gdrive/MyDrive/handwriting_unet_fast/best_unet.pt"
)

checkpoint = torch.load(checkpoint_path, map_location=device)

restoration_model = FastRestorationUNet().to(device)
restoration_model.load_state_dict(checkpoint["model_state_dict"])
restoration_model.eval()

print("Checkpoint loaded successfully!")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Device:", device)


In [ ]:

blur, clean = val_ds[0]

restoration_model.eval()

with torch.no_grad():
    pred = restoration_model(
        blur.unsqueeze(0).to(device)
    ).squeeze(0).cpu()

def to_image(x):
    return ((x.detach().cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(to_image(blur))
ax[0].set_title("Blurred Input")

ax[1].imshow(to_image(pred))
ax[1].set_title("Restored Output")

ax[2].imshow(to_image(clean))
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import zipfile
from PIL import Image
from io import BytesIO

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    idx = val_ids[0]

    blur_original = Image.open(
        BytesIO(z.read(blur3_files[idx]))
    )
    clean_original = Image.open(
        BytesIO(z.read(gt_files[idx]))
    )

print("Blur image size:", blur_original.size)
print("Ground truth size:", clean_original.size)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.imshow(blur_original)
plt.title("Original Blur Level 3")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(clean_original)
plt.title("Original Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

from PIL import Image
import matplotlib.pyplot as plt

# Original images থেকে aspect ratio বজায় রেখে preview
preview_width = 568
preview_height = round(
    clean_original.height * preview_width / clean_original.width
)

blur_preview = blur_original.resize(
    (preview_width, preview_height), Image.Resampling.LANCZOS
)
clean_preview = clean_original.resize(
    (preview_width, preview_height), Image.Resampling.LANCZOS
)

fig, ax = plt.subplots(1, 2, figsize=(10, 9))

ax[0].imshow(blur_preview)
ax[0].set_title("Blurred — Aspect Ratio Preserved")
ax[0].axis("off")

ax[1].imshow(clean_preview)
ax[1].set_title("Ground Truth — Aspect Ratio Preserved")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

# Original image-এর মাঝখানের একটি অংশ
w, h = blur_original.size
box = (w//4, h//4, 3*w//4, 3*h//4)

blur_crop = blur_original.crop(box)
clean_crop = clean_original.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(blur_crop)
ax[0].set_title("Blurred Crop")
ax[0].axis("off")

ax[1].imshow(clean_crop)
ax[1].set_title("Ground Truth Crop")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 7))

ax[0].imshow(blur_original)
ax[0].set_title("Original Blurred Image")
ax[0].axis("off")

ax[1].imshow(clean_original)
ax[1].set_title("Original Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

# আগের dataset-এর একই matched pair ব্যবহার
idx = val_ids[0]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_full = Image.open(
        BytesIO(z.read(blur3_files[idx]))
    ).convert("RGB")

    clean_full = Image.open(
        BytesIO(z.read(gt_files[idx]))
    ).convert("RGB")

print("Blur size:", blur_full.size)
print("Clean size:", clean_full.size)

# একই coordinates থেকে patch নেওয়া
patch_size = 256
left = (blur_full.width - patch_size) // 2
top = (blur_full.height - patch_size) // 2
crop_box = (left, top, left + patch_size, top + patch_size)

blur_patch = blur_full.crop(crop_box)
clean_patch = clean_full.crop(crop_box)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(blur_patch)
ax[0].set_title("Blurred Patch")
ax[0].axis("off")

ax[1].imshow(clean_patch)
ax[1].set_title("Ground Truth Patch")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(blur_patch)
ax[0].set_title("Blurred Patch")
ax[0].axis("off")

ax[1].imshow(clean_patch)
ax[1].set_title("Ground Truth Patch")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from PIL import Image
import matplotlib.pyplot as plt

# পুরো পৃষ্ঠার aspect ratio বজায় রেখে resize
preview_width = 568
preview_height = round(
    blur_full.height * preview_width / blur_full.width
)

blur_preview = blur_full.resize(
    (preview_width, preview_height),
    Image.Resampling.LANCZOS
)
clean_preview = clean_full.resize(
    (preview_width, preview_height),
    Image.Resampling.LANCZOS
)

fig, ax = plt.subplots(1, 2, figsize=(10, 7))

ax[0].imshow(blur_preview)
ax[0].set_title("Full Blurred Page")
ax[0].axis("off")

ax[1].imshow(clean_preview)
ax[1].set_title("Full Ground Truth Page")
ax[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import random
import zipfile
from io import BytesIO
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

PATCH_SIZE = 128
BATCH_SIZE = 2

patch_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

class PatchRestorationDataset(Dataset):
    def __init__(self, ids, patches_per_image=2):
        self.ids = ids
        self.patches_per_image = patches_per_image

    def __len__(self):
        return len(self.ids) * self.patches_per_image

    def __getitem__(self, index):
        idx = self.ids[index // self.patches_per_image]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")
            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        # Ensure paired images have matching dimensions
        if blur.size != clean.size:
            clean = clean.resize(blur.size, Image.Resampling.LANCZOS)

        w, h = blur.size
        x = random.randint(0, w - PATCH_SIZE)
        y = random.randint(0, h - PATCH_SIZE)
        box = (x, y, x + PATCH_SIZE, y + PATCH_SIZE)

        blur_patch = blur.crop(box)
        clean_patch = clean.crop(box)

        return (
            patch_transform(blur_patch),
            patch_transform(clean_patch)
        )

patch_train_ds = PatchRestorationDataset(train_ids)
patch_val_ds = PatchRestorationDataset(val_ids, patches_per_image=1)

patch_train_loader = DataLoader(
    patch_train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0
)
patch_val_loader = DataLoader(
    patch_val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)

print("Training patches:", len(patch_train_ds))
print("Validation patches:", len(patch_val_ds))
print("Patch dataset ready!")


In [ ]:

import os
import torch
import torch.nn.functional as F

device = torch.device("cpu")
restoration_model = FastRestorationUNet().to(device)

checkpoint_path = (
    "/content/gdrive/MyDrive/handwriting_unet_fast/best_unet.pt"
)
checkpoint = torch.load(checkpoint_path, map_location=device)
restoration_model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    restoration_model.parameters(), lr=2e-5
)

SAVE_DIR = "/content/gdrive/MyDrive/handwriting_unet_patches"
os.makedirs(SAVE_DIR, exist_ok=True)

best_val = float("inf")
epochs = 3

for epoch in range(epochs):
    restoration_model.train()
    train_sum = 0.0

    for blur, clean in patch_train_loader:
        blur, clean = blur.to(device), clean.to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = restoration_model(blur)
        loss = F.l1_loss(pred, clean)
        loss.backward()
        optimizer.step()

        train_sum += loss.item()

    train_loss = train_sum / len(patch_train_loader)

    restoration_model.eval()
    val_sum = 0.0

    with torch.no_grad():
        for blur, clean in patch_val_loader:
            pred = restoration_model(blur.to(device))
            val_sum += F.l1_loss(
                pred, clean.to(device)
            ).item()

    val_loss = val_sum / len(patch_val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train: {train_loss:.5f} | Val: {val_loss:.5f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": restoration_model.state_dict(),
            "val_loss": val_loss
        }, os.path.join(SAVE_DIR, "best_patch_unet.pt"))

print("Fine-tuning complete. Best validation loss:", best_val)


In [ ]:
MODEL_PATH = "/content/gdrive/MyDrive/handwriting_unet_patches/best_patch_unet.pt"

In [ ]:
!pip -q install gradio

import os
import numpy as np
import torch
import torch.nn as nn
import gradio as gr
from PIL import Image

# ---------- Model ----------
class FastRestorationUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.e1 = block(3, 16)
        self.e2 = block(16, 32)
        self.e3 = block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(64, 128)

        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d3 = block(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = block(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = block(32, 16)
        self.out = nn.Conv2d(16, 3, 1)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        z = self.mid(self.pool(c))
        z = self.d3(torch.cat([self.u3(z), c], dim=1))
        z = self.d2(torch.cat([self.u2(z), b], dim=1))
        z = self.d1(torch.cat([self.u1(z), a], dim=1))
        return torch.tanh(self.out(z))


# ---------- Load trained checkpoint ----------
device = torch.device("cpu")

MODEL_PATH = (
    "/content/gdrive/MyDrive/"
    "handwriting_unet_patches/best_patch_unet.pt"
)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found: {MODEL_PATH}"
    )

model = FastRestorationUNet().to(device)

checkpoint = torch.load(
    MODEL_PATH, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded. Best validation loss:",
      checkpoint.get("val_loss", "Not recorded"))


# ---------- Full-page tiled restoration ----------
def restore_image(image):
    if image is None:
        raise gr.Error("Please upload an image first.")

    image = image.convert("RGB")
    original = np.asarray(image).astype(np.float32) / 255.0

    h, w, _ = original.shape
    patch = 128
    stride = 96

    # Pad small images if necessary
    pad_h = max(0, patch - h)
    pad_w = max(0, patch - w)

    padded = np.pad(
        original,
        ((0, pad_h), (0, pad_w), (0, 0)),
        mode="reflect"
    )
    H, W, _ = padded.shape

    ys = list(range(0, max(1, H - patch + 1), stride))
    xs = list(range(0, max(1, W - patch + 1), stride))

    if ys[-1] != H - patch:
        ys.append(H - patch)
    if xs[-1] != W - patch:
        xs.append(W - patch)

    result = np.zeros_like(padded, dtype=np.float32)
    count = np.zeros((H, W, 1), dtype=np.float32)

    with torch.inference_mode():
        for y in ys:
            for x in xs:
                tile = padded[y:y+patch, x:x+patch, :]

                tensor = torch.from_numpy(
                    tile.transpose(2, 0, 1).copy()
                ).float().unsqueeze(0)

                tensor = tensor * 2 - 1
                prediction = model(tensor.to(device))
                prediction = (
                    prediction[0].cpu().numpy().transpose(1, 2, 0)
                    + 1
                ) / 2

                prediction = np.clip(prediction, 0, 1)
                result[y:y+patch, x:x+patch] += prediction
                count[y:y+patch, x:x+patch] += 1

    result /= np.maximum(count, 1)
    result = result[:h, :w, :]
    output = Image.fromarray(
        (result * 255).round().astype(np.uint8)
    )

    return output


# ---------- Web interface ----------
demo = gr.Interface(
    fn=restore_image,
    inputs=gr.Image(type="pil", label="Upload Blurred Handwriting"),
    outputs=gr.Image(type="pil", label="Restored Image"),
    title="English Handwriting Image Restoration",
    description=(
        "Upload a blurred handwriting image and restore it "
        "using a trained U-Net model. Full-page processing "
        "uses overlapping patches."
    ),
    submit_btn="Restore Image",
    examples=None
)

demo.launch(share=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from io import BytesIO
import zipfile

# Dataset থেকে একটি blurred image নেওয়া
ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = [
        f for f in z.namelist()
        if "Blur_Level_3" in f
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    test_image = Image.open(
        BytesIO(z.read(files[0]))
    ).convert("RGB")

# Model input তৈরি
small = test_image.resize((128, 128))
arr = np.asarray(small).astype(np.float32) / 255.0
x = torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0)
x = x * 2 - 1

model.eval()
with torch.no_grad():
    pred = model(x.to(device))[0].cpu().numpy()

print("Input range:", x.min().item(), x.max().item())
print("Prediction range:", pred.min(), pred.max())
print("Prediction mean:", pred.mean())

# Prediction-কে image-এ convert
pred_img = np.clip(
    (pred.transpose(1, 2, 0) + 1) / 2, 0, 1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(test_image)
plt.title("Blurred Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(small)
plt.title("128x128 Input")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_img)
plt.title("Raw Model Output")
plt.axis("off")
plt.show()

In [ ]:
import torch

p = "/content/gdrive/MyDrive/handwriting_unet_patches/best_patch_unet.pt"
ckpt = torch.load(p, map_location="cpu", weights_only=False)

print("Checkpoint epoch:", ckpt["epoch"])
print("Validation loss:", ckpt["val_loss"])

print("Output layer weights:")
print(model.out.weight.data.min().item(),
      model.out.weight.data.max().item())

print("Output layer bias:")
print(model.out.bias.data)

In [ ]:
blur, clean = patch_val_ds[0]

print("Blur:", blur.min().item(), blur.max().item(),
      blur.mean().item())
print("Ground Truth:", clean.min().item(),
      clean.max().item(), clean.mean().item())

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cpu")
model.eval()

blur, clean = patch_val_ds[0]

with torch.no_grad():
    pred = model(blur.unsqueeze(0).to(device))[0].cpu()

def show_stats(name, x):
    img = ((x.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
    print(
        f"{name}: min={img.min():.4f}, "
        f"max={img.max():.4f}, mean={img.mean():.4f}, "
        f"std={img.std():.4f}"
    )
    return img

blur_img = show_stats("Blur", blur)
clean_img = show_stats("Ground Truth", clean)
pred_img = show_stats("Prediction", pred)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, img, title in zip(
    ax,
    [blur_img, pred_img, clean_img],
    ["Blur Input", "Model Prediction", "Ground Truth"]
):
    a.imshow(img)
    a.set_title(title)
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cpu")

# আগের baseline checkpoint load
baseline_path = (
    "/content/gdrive/MyDrive/"
    "handwriting_unet_fast/best_unet.pt"
)

ckpt = torch.load(
    baseline_path, map_location=device, weights_only=False
)

baseline_model = FastRestorationUNet().to(device)
baseline_model.load_state_dict(ckpt["model_state_dict"])
baseline_model.eval()

# 96x96 input: baseline model যেভাবে train হয়েছিল
blur, clean = val_ds[0]

with torch.no_grad():
    pred = baseline_model(blur.unsqueeze(0))[0].cpu()

def to_image(t):
    return ((t.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

blur_img = to_image(blur)
pred_img = to_image(pred)
clean_img = to_image(clean)

print("Prediction min:", pred.min().item())
print("Prediction max:", pred.max().item())
print("Prediction std:", pred_img.std())

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, img, title in zip(
    ax,
    [blur_img, pred_img, clean_img],
    ["Blur Input", "Baseline U-Net", "Ground Truth"]
):
    a.imshow(img)
    a.set_title(title)
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# একই normalization function দুই image-এর জন্য
def normalize_image(pil_image):
    arr = np.asarray(pil_image.convert("RGB"), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr.transpose(2, 0, 1).copy())
    return (tensor - 0.5) / 0.5

# ZIP থেকে একই ID-এর pair load
from PIL import Image
from io import BytesIO
import zipfile

test_id = val_ids[0]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_pil = Image.open(
        BytesIO(z.read(blur3_files[test_id]))
    ).convert("RGB")
    clean_pil = Image.open(
        BytesIO(z.read(gt_files[test_id]))
    ).convert("RGB")

# একই size ও একই crop
if blur_pil.size != clean_pil.size:
    clean_pil = clean_pil.resize(
        blur_pil.size, Image.Resampling.LANCZOS
    )

w, h = blur_pil.size
PATCH_SIZE = 128
x0 = max(0, (w - PATCH_SIZE) // 2)
y0 = max(0, (h - PATCH_SIZE) // 2)
box = (x0, y0, x0 + PATCH_SIZE, y0 + PATCH_SIZE)

blur = normalize_image(blur_pil.crop(box))
clean = normalize_image(clean_pil.crop(box))

print("Blur range:", blur.min().item(), blur.max().item())
print("GT range:", clean.min().item(), clean.max().item())
print("Blur mean/std:", blur.mean().item(), blur.std().item())
print("GT mean/std:", clean.mean().item(), clean.std().item())

def display(t):
    return ((t.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(display(blur))
ax[0].set_title("Blur Level 3")
ax[1].imshow(display(clean))
ax[1].set_title("Ground Truth")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(display(blur))
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(display(clean))
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# পুরো image ছোট করে দেখানো; crop করা হবে না
preview_size = (320, 430)

blur_preview = blur_pil.copy()
blur_preview.thumbnail(preview_size)

gt_preview = clean_pil.copy()
gt_preview.thumbnail(preview_size)

fig, ax = plt.subplots(1, 2, figsize=(8, 5))

ax[0].imshow(blur_preview)
ax[0].set_title("Blur Level 3 - Full Page")
ax[0].axis("off")

ax[1].imshow(gt_preview)
ax[1].set_title("Ground Truth - Full Page")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display

# Original Ground Truth image — no resizing, no crop
display(clean_pil)

In [ ]:
from PIL import Image, ImageChops
from io import BytesIO
import zipfile
import matplotlib.pyplot as plt

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

# Validation set থেকে একটি image ID
test_id = val_ids[0]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_img = Image.open(
        BytesIO(z.read(blur3_files[test_id]))
    ).convert("RGB")

    gt_img = Image.open(
        BytesIO(z.read(gt_files[test_id]))
    ).convert("RGB")

print("Image ID:", test_id)
print("Blur size:", blur_img.size)
print("Ground Truth size:", gt_img.size)
print("Blur mode:", blur_img.mode)
print("Ground Truth mode:", gt_img.mode)

if blur_img.size == gt_img.size:
    print("Size check: PASS")
else:
    print("Size check: Different sizes — investigate before training")

# পুরো পৃষ্ঠা একই আকারে preview
blur_preview = blur_img.copy()
gt_preview = gt_img.copy()

blur_preview.thumbnail((380, 500))
gt_preview.thumbnail((380, 500))

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
ax[0].imshow(blur_preview)
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_preview)
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import random
import numpy as np
import torch
from PIL import Image

PATCH_SIZE = 128

def get_aligned_patch(blur_img, gt_img, patch_size=128):
    if blur_img.size != gt_img.size:
        raise ValueError("Blur and GT dimensions do not match.")

    w, h = blur_img.size
    x = random.randint(0, w - patch_size)
    y = random.randint(0, h - patch_size)
    box = (x, y, x + patch_size, y + patch_size)

    blur_patch = blur_img.crop(box)
    gt_patch = gt_img.crop(box)

    def to_tensor(img):
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(
            arr.transpose(2, 0, 1).copy()
        ) * 2 - 1

    return to_tensor(blur_patch), to_tensor(gt_patch)

blur_patch, gt_patch = get_aligned_patch(blur_img, gt_img)

print("Blur patch:", blur_patch.shape)
print("GT patch:", gt_patch.shape)
print("Blur range:", blur_patch.min().item(), blur_patch.max().item())
print("GT range:", gt_patch.min().item(), gt_patch.max().item())

In [ ]:
import matplotlib.pyplot as plt

def show_tensor(t):
    return ((t.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].imshow(show_tensor(blur_patch))
ax[0].set_title("Blur Patch")
ax[0].axis("off")

ax[1].imshow(show_tensor(gt_patch))
ax[1].set_title("Ground Truth Patch")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 6))

ax[0].imshow(blur_img)
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

print("Blur image size:", blur_img.size)
print("Ground Truth size:", gt_img.size)

fig, ax = plt.subplots(1, 2, figsize=(12, 8))

ax[0].imshow(blur_img)
ax[0].set_title("Blur Level 3 - Input")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Ground Truth - Target")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 8))

ax[0].imshow(blur_img)
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

blur_tensor, gt_tensor = patch_val_ds[0]

device = next(model.parameters()).device

with torch.no_grad():
    pred = model(blur_tensor.unsqueeze(0).to(device))

pred_img = (
    pred.squeeze(0).cpu()
    .clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

input_img = (
    blur_tensor.cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

target_img = (
    gt_tensor.cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

fig, ax = plt.subplots(1, 3, figsize=(12, 5))

ax[0].imshow(input_img)
ax[0].set_title("Blur Input")
ax[1].imshow(pred_img)
ax[1].set_title("Model Output")
ax[2].imshow(target_img)
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Prediction range:", pred.min().item(), pred.max().item())
print("Prediction standard deviation:", pred.std().item())

In [ ]:
print("Input min/max:",
      blur_tensor.min().item(),
      blur_tensor.max().item())

print("Ground Truth min/max:",
      gt_tensor.min().item(),
      gt_tensor.max().item())

print("Input mean:", blur_tensor.mean().item())
print("Ground Truth mean:", gt_tensor.mean().item())

print("Model output min/max:",
      pred.min().item(),
      pred.max().item())

In [ ]:
import torch

print("Model device:",
      next(model.parameters()).device)

print("Model output layer:")
print(model.out)

print("\nOutput activation range:")
print("Min:", pred.min().item())
print("Max:", pred.max().item())
print("Std:", pred.std().item())

print("\nInput statistics:")
print("Mean:", blur_tensor.mean().item())
print("Std:", blur_tensor.std().item())

print("\nGround Truth statistics:")
print("Mean:", gt_tensor.mean().item())
print("Std:", gt_tensor.std().item())

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cpu")
model = model.to(device)
model.train()

# একটি fixed batch তৈরি
x = torch.stack([patch_train_ds[i][0] for i in range(4)]).to(device)
y = torch.stack([patch_train_ds[i][1] for i in range(4)]).to(device)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

for step in range(1, 31):
    optimizer.zero_grad()

    output = model(x)
    loss = criterion(output, y)

    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(
            f"Step {step:02d} | "
            f"Loss: {loss.item():.5f} | "
            f"Output std: {output.std().item():.5f}"
        )

model.eval()
with torch.no_grad():
    test_output = model(x[:1])

print("Final prediction range:",
      test_output.min().item(),
      test_output.max().item())
print("Final prediction std:", test_output.std().item())

In [ ]:
import torch
import torch.nn as nn

model.train()

x = torch.stack([patch_train_ds[i][0] for i in range(4)]).to(device)
y = torch.stack([patch_train_ds[i][1] for i in range(4)]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

# Forward pass
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)

# Backpropagation
loss.backward()

print("Loss:", loss.item())
print("Output layer gradient:",
      model.out.weight.grad.abs().mean().item())
print("Output bias gradient:",
      model.out.bias.grad.abs().mean().item())

# Check whether weights actually update
old_weight = model.out.weight.detach().clone()
optimizer.step()

print("Output layer weight change:",
      (model.out.weight.detach() - old_weight).abs().mean().item())

In [ ]:
import torch
import torch.nn as nn
import copy

# বর্তমান model-এর আলাদা copy
test_model = copy.deepcopy(model).to("cpu")
test_model.train()

x = torch.stack([
    patch_train_ds[i][0] for i in range(4)
]).to("cpu")

y = torch.stack([
    patch_train_ds[i][1] for i in range(4)
]).to("cpu")

optimizer = torch.optim.Adam(test_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

for step in range(1, 101):
    optimizer.zero_grad()

    output = test_model(x)
    loss = criterion(output, y)

    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        with torch.no_grad():
            current_output = test_model(x)

        print(
            f"Step {step} | "
            f"Loss: {loss.item():.5f} | "
            f"Output std: {current_output.std().item():.5f}"
        )

In [ ]:
import torch

test_model.eval()

x = torch.stack([patch_train_ds[i][0] for i in range(4)])
y = torch.stack([patch_train_ds[i][1] for i in range(4)])

with torch.no_grad():
    output = test_model(x)

print("Input range:", x.min().item(), x.max().item())
print("Target range:", y.min().item(), y.max().item())
print("Output range:", output.min().item(), output.max().item())
print("Output std:", output.std().item())

print("\nOutput layer bias:")
print(test_model.out.bias.detach())

print("\nOutput layer weights:")
print("Min:", test_model.out.weight.min().item())
print("Max:", test_model.out.weight.max().item())
print("Std:", test_model.out.weight.std().item())

In [ ]:
import torch

test_model.eval()

x = torch.stack([patch_train_ds[i][0] for i in range(4)])

with torch.no_grad():
    a = test_model.e1(x)
    b = test_model.e2(test_model.pool(a))
    c = test_model.e3(test_model.pool(b))
    z = test_model.mid(test_model.pool(c))

    z = test_model.d3(torch.cat([test_model.u3(z), c], dim=1))
    z = test_model.d2(torch.cat([test_model.u2(z), b], dim=1))
    z = test_model.d1(torch.cat([test_model.u1(z), a], dim=1))

    raw = test_model.out(z)
    output = torch.tanh(raw)

print("Raw output min:", raw.min().item())
print("Raw output max:", raw.max().item())
print("Raw output mean:", raw.mean().item())
print("Raw output std:", raw.std().item())

print("Tanh output min:", output.min().item())
print("Tanh output max:", output.max().item())
print("Tanh output std:", output.std().item())

In [ ]:
import torch
import torch.nn as nn

class StableUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(a, b):
            return nn.Sequential(
                nn.Conv2d(a, b, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(b, b, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.e1 = block(3, 16)
        self.e2 = block(16, 32)
        self.e3 = block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(64, 128)

        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d3 = block(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = block(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = block(32, 16)

        self.out = nn.Conv2d(16, 3, 1)

        # ছোট weight দিয়ে শুরু, যাতে output শুরুতেই saturate না করে
        nn.init.normal_(self.out.weight, mean=0.0, std=0.001)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        z = self.mid(self.pool(c))

        z = self.d3(torch.cat([self.u3(z), c], dim=1))
        z = self.d2(torch.cat([self.u2(z), b], dim=1))
        z = self.d1(torch.cat([self.u1(z), a], dim=1))

        # আপাতত tanh ছাড়া output
        return self.out(z)

device = torch.device("cpu")
stable_model = StableUNet().to(device)

print("New model created.")

In [ ]:
import torch.nn.functional as F

x = torch.stack([patch_train_ds[i][0] for i in range(2)]).to(device)
y = torch.stack([patch_train_ds[i][1] for i in range(2)]).to(device)

optimizer = torch.optim.Adam(stable_model.parameters(), lr=1e-4)

for step in range(1, 21):
    stable_model.train()
    optimizer.zero_grad()

    output = stable_model(x)
    loss = F.l1_loss(output, y)

    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        with torch.no_grad():
            check = stable_model(x)

        print(
            f"Step {step} | Loss: {loss.item():.5f} | "
            f"Output range: {check.min().item():.3f} to "
            f"{check.max().item():.3f} | "
            f"Std: {check.std().item():.4f}"
        )

In [ ]:
import torch.nn.functional as F

for step in range(21, 121):
    stable_model.train()
    optimizer.zero_grad()

    output = stable_model(x)
    loss = F.l1_loss(output, y)

    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        stable_model.eval()
        with torch.no_grad():
            check = stable_model(x)

        print(
            f"Step {step} | Loss: {loss.item():.5f} | "
            f"Min: {check.min().item():.3f} | "
            f"Max: {check.max().item():.3f} | "
            f"Std: {check.std().item():.4f}"
        )

In [ ]:
import matplotlib.pyplot as plt
import torch

stable_model.eval()

with torch.no_grad():
    pred = stable_model(x[:1])[0].cpu()

input_img = (
    x[0].cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

target_img = (
    y[0].cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

pred_img = (
    pred.clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

fig, ax = plt.subplots(1, 3, figsize=(12, 5))

ax[0].imshow(input_img)
ax[0].set_title("Blur Input")

ax[1].imshow(pred_img)
ax[1].set_title("Restored Prediction")

ax[2].imshow(target_img)
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(input_img.mean(axis=2), cmap="gray", vmin=0, vmax=1)
plt.title("Blur Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_img.mean(axis=2), cmap="gray", vmin=0, vmax=1)
plt.title("Model Prediction")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(target_img.mean(axis=2), cmap="gray", vmin=0, vmax=1)
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F

# একই 4টি paired patch
x_test = torch.stack([patch_train_ds[i][0] for i in range(4)])
y_test = torch.stack([patch_train_ds[i][1] for i in range(4)])

# Input যদি [0,1]-এ থাকে, তাকে [-1,1]-এ রূপান্তর
if x_test.min() >= 0 and x_test.max() <= 1:
    x_test = x_test * 2 - 1

# Ground Truth ইতিমধ্যে [-1,1]-এ আছে
x_test = x_test.to("cpu")
y_test = y_test.to("cpu")

print("Input range:", x_test.min().item(), x_test.max().item())
print("Target range:", y_test.min().item(), y_test.max().item())

# একেবারে নতুন model
stable_model = StableUNet().to("cpu")
optimizer = torch.optim.Adam(stable_model.parameters(), lr=1e-4)

for step in range(1, 101):
    stable_model.train()
    optimizer.zero_grad()

    output = stable_model(x_test)
    loss = F.l1_loss(output, y_test)

    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step} | Loss: {loss.item():.5f}")

In [ ]:
import matplotlib.pyplot as plt
import torch

stable_model.eval()

with torch.no_grad():
    pred = stable_model(x_test[:1])[0].cpu()

input_img = (
    x_test[0].cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

target_img = (
    y_test[0].cpu().clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

pred_img = (
    pred.clamp(-1, 1).add(1).div(2)
    .permute(1, 2, 0).numpy()
)

fig, ax = plt.subplots(1, 3, figsize=(12, 5))

ax[0].imshow(input_img)
ax[0].set_title("Blur Input")
ax[1].imshow(pred_img)
ax[1].set_title("Restored Output")
ax[2].imshow(target_img)
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
stable_model.eval()

with torch.no_grad():
    pred = stable_model(x_test[:1])

print("Prediction min:", pred.min().item())
print("Prediction max:", pred.max().item())
print("Prediction std:", pred.std().item())
print("Target min:", y_test[:1].min().item())
print("Target max:", y_test[:1].max().item())

In [ ]:
import matplotlib.pyplot as plt
import torch

stable_model.eval()

with torch.no_grad():
    pred = stable_model(x_test[:1])[0].cpu()

input_img = x_test[0].cpu().clamp(-1, 1).add(1).div(2)
target_img = y_test[0].cpu().clamp(-1, 1).add(1).div(2)
pred_img = pred.clamp(-1, 1).add(1).div(2)

def to_image(t):
    return t.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(to_image(input_img))
ax[0].set_title("Blur Input")
ax[1].imshow(to_image(pred_img))
ax[1].set_title("Restored Prediction")
ax[2].imshow(to_image(target_img))
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x = patch_train_ds[0][0]
y = patch_train_ds[0][1]

fig, ax = plt.subplots(1, 2, figsize=(8, 5))

ax[0].imshow(x.clamp(-1, 1).add(1).div(2).permute(1, 2, 0))
ax[0].set_title("Blur Input")

ax[1].imshow(y.clamp(-1, 1).add(1).div(2).permute(1, 2, 0))
ax[1].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.show()

In [ ]:
from PIL import Image
from io import BytesIO
import zipfile
import matplotlib.pyplot as plt

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_img = Image.open(
        BytesIO(z.read(blur3_files[test_id]))
    ).convert("RGB")

    gt_img = Image.open(
        BytesIO(z.read(gt_files[test_id]))
    ).convert("RGB")

print("Blur size:", blur_img.size)
print("Ground Truth size:", gt_img.size)

fig, ax = plt.subplots(1, 2, figsize=(12, 8))

ax[0].imshow(blur_img)
ax[0].set_title("Original Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Original Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

blur_np = np.array(blur_img)
gt_np = np.array(gt_img)

blur_gray = cv2.cvtColor(blur_np, cv2.COLOR_RGB2GRAY)
gt_gray = cv2.cvtColor(gt_np, cv2.COLOR_RGB2GRAY)

print("Blur sharpness:", cv2.Laplacian(
    blur_gray, cv2.CV_64F
).var())

print("Ground Truth sharpness:", cv2.Laplacian(
    gt_gray, cv2.CV_64F
).var())

fig, ax = plt.subplots(1, 2, figsize=(12, 7))

ax[0].imshow(blur_img)
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# একই জায়গা থেকে crop
box = (200, 300, 600, 500)

blur_crop = blur_img.crop(box)
gt_crop = gt_img.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].imshow(blur_crop)
ax[0].set_title("Blur Level 3 Crop")
ax[0].axis("off")

ax[1].imshow(gt_crop)
ax[1].set_title("Ground Truth Crop")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(blur_img)
ax[0].set_title("Original Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_img)
ax[1].set_title("Original Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
stable_model.eval()

with torch.no_grad():
    x, y = patch_val_ds[0]

    x = x.unsqueeze(0).to(device)
    pred = stable_model(x)

    print("Input range:", x.min().item(), x.max().item())
    print("Target range:", y.min().item(), y.max().item())
    print("Prediction range:", pred.min().item(), pred.max().item())
    print("Prediction std:", pred.std().item())

In [ ]:
import torch
import matplotlib.pyplot as plt

stable_model.eval()

x, y = patch_val_ds[0]

with torch.no_grad():
    pred = stable_model(x.unsqueeze(0).to(device))[0].cpu()

# Model output-কে display করার জন্য [0, 1]-এ সীমাবদ্ধ করা
x_show = ((x.cpu() + 1) / 2).clamp(0, 1)
y_show = ((y.cpu() + 1) / 2).clamp(0, 1)
pred_show = ((pred + 1) / 2).clamp(0, 1)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(x_show.permute(1, 2, 0))
ax[0].set_title("Input: Blur Level 3")

ax[1].imshow(y_show.permute(1, 2, 0))
ax[1].set_title("Ground Truth")

ax[2].imshow(pred_show.permute(1, 2, 0))
ax[2].set_title("Model Prediction")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch

stable_model.eval()

# Original full-size blurred image
w, h = blur_img.size

# Full image-ke model input-er jonno resize
input_small = blur_img.resize((256, 256), Image.Resampling.LANCZOS)
arr = np.asarray(input_small).astype(np.float32) / 255.0

# Model-er expected input range [-1, 1]
tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
tensor = (tensor * 2 - 1).to(device)

with torch.no_grad():
    pred = stable_model(tensor)[0].cpu()

# Output display range [0, 1]
pred = ((pred + 1) / 2).clamp(0, 1)
pred_img = (pred.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

# Restore original aspect ratio and dimensions
restored_img = Image.fromarray(pred_img).resize(
    (w, h), Image.Resampling.LANCZOS
)

fig, ax = plt.subplots(1, 2, figsize=(12, 8))
ax[0].imshow(blur_img)
ax[0].set_title("Original Blur")
ax[1].imshow(restored_img)
ax[1].set_title("Restored Full Image")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

def restore_full_page(model, image, device, patch_size=128, overlap=32):
    model.eval()

    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    arr = arr * 2.0 - 1.0
    h, w, _ = arr.shape

    step = patch_size - overlap
    ys = list(range(0, max(h - patch_size, 0) + 1, step))
    xs = list(range(0, max(w - patch_size, 0) + 1, step))

    if not ys or ys[-1] + patch_size < h:
        ys.append(max(h - patch_size, 0))
    if not xs or xs[-1] + patch_size < w:
        xs.append(max(w - patch_size, 0))

    total = np.zeros((h, w, 3), dtype=np.float32)
    weights = np.zeros((h, w, 1), dtype=np.float32)

    # Smooth blending window to reduce patch seams
    win = np.hanning(patch_size).astype(np.float32)
    win = np.maximum(np.outer(win, win), 0.05)[..., None]

    with torch.no_grad():
        for y in ys:
            for x in xs:
                patch = arr[y:y+patch_size, x:x+patch_size, :]
                if patch.shape[:2] != (patch_size, patch_size):
                    continue

                t = torch.from_numpy(patch.copy()).permute(2, 0, 1)
                t = t.unsqueeze(0).to(device)

                out = model(t)[0].detach().cpu().permute(1, 2, 0).numpy()
                out = np.clip(out, -1, 1)

                total[y:y+patch_size, x:x+patch_size] += out * win
                weights[y:y+patch_size, x:x+patch_size] += win

    restored = total / np.maximum(weights, 1e-6)
    restored = ((restored + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(restored)

restored_img = restore_full_page(stable_model, blur_img, device)

fig, ax = plt.subplots(1, 2, figsize=(12, 8))
ax[0].imshow(blur_img)
ax[0].set_title("Original Blur")
ax[1].imshow(restored_img)
ax[1].set_title("Patch-Based Restoration")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt

stable_model.eval()

x, y = patch_val_ds[0]

with torch.no_grad():
    pred = stable_model(x.unsqueeze(0).to(device))[0].cpu()

fig, ax = plt.subplots(1, 3, figsize=(14, 5))

ax[0].imshow(((x.cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[0].set_title("Blur Input")

ax[1].imshow(((y.cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[1].set_title("Ground Truth")

ax[2].imshow(((pred + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[2].set_title("Model Prediction")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
from io import BytesIO
import zipfile
import matplotlib.pyplot as plt

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_original = Image.open(
        BytesIO(z.read(blur3_files[test_id]))
    ).convert("RGB")

    gt_original = Image.open(
        BytesIO(z.read(gt_files[test_id]))
    ).convert("RGB")

fig, ax = plt.subplots(1, 2, figsize=(12, 8))

ax[0].imshow(blur_original)
ax[0].set_title("Original Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_original)
ax[1].set_title("Original Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

blur_arr = np.asarray(blur_original).astype(np.float32) / 255.0
gt_arr = np.asarray(gt_original).astype(np.float32) / 255.0

print("Blur shape:", blur_arr.shape)
print("Ground Truth shape:", gt_arr.shape)

print("Blur min/max/mean:",
      blur_arr.min(), blur_arr.max(), blur_arr.mean())

print("Ground Truth min/max/mean:",
      gt_arr.min(), gt_arr.max(), gt_arr.mean())

In [ ]:
import matplotlib.pyplot as plt
import torch

x, y = patch_train_ds[0]

print("Input range:", x.min().item(), x.max().item())
print("Target range:", y.min().item(), y.max().item())

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(((x + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[0].set_title("Blur Input Patch")
ax[0].axis("off")

ax[1].imshow(((y + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[1].set_title("Ground Truth Patch")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Original image theke eki area crop
box = (100, 250, 1000, 1250)

blur_crop = blur_original.crop(box)
gt_crop = gt_original.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(blur_crop)
ax[0].set_title("Blur Level 3 - Crop")
ax[0].axis("off")

ax[1].imshow(gt_crop)
ax[1].set_title("Ground Truth - Crop")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
x, y = patch_train_ds[0]

print("Input shape:", x.shape)
print("Target shape:", y.shape)

print("Input range:", x.min().item(), x.max().item())
print("Target range:", y.min().item(), y.max().item())

In [ ]:
import inspect

print(inspect.getsource(PatchRestorationDataset))

In [ ]:
import torch

x, y = patch_train_ds[0]

print("Input:")
print("Min:", x.min().item())
print("Max:", x.max().item())
print("Mean:", x.mean().item())

print("\nGround Truth:")
print("Min:", y.min().item())
print("Max:", y.max().item())
print("Mean:", y.mean().item())

In [ ]:
import matplotlib.pyplot as plt

x, y = patch_train_ds[0]

# দুটিকে [0, 1] display range-এ রূপান্তর
x_display = x.clamp(0, 1)
y_display = ((y + 1) / 2).clamp(0, 1)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(x_display.permute(1, 2, 0))
ax[0].set_title("Input Patch")

ax[1].imshow(y_display.permute(1, 2, 0))
ax[1].set_title("Ground Truth Patch")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Original full-size images theke eki handwriting area crop
box = (100, 250, 1000, 1250)

blur_crop = blur_original.crop(box)
gt_crop = gt_original.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(blur_crop)
ax[0].set_title("Blur Level 3")
ax[0].axis("off")

ax[1].imshow(gt_crop)
ax[1].set_title("Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Same coordinates from the original paired images
box = (100, 250, 1000, 1250)

blur_crop = blur_original.crop(box).resize((256, 256))
gt_crop = gt_original.crop(box).resize((256, 256))

blur_arr = np.asarray(blur_crop, dtype=np.float32) / 255.0
gt_arr = np.asarray(gt_crop, dtype=np.float32) / 255.0

# Normalize BOTH images consistently to [-1, 1]
blur_tensor = torch.from_numpy(blur_arr).permute(2, 0, 1) * 2 - 1
gt_tensor = torch.from_numpy(gt_arr).permute(2, 0, 1) * 2 - 1

print("Input range:", blur_tensor.min().item(), blur_tensor.max().item())
print("Target range:", gt_tensor.min().item(), gt_tensor.max().item())

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(((blur_tensor + 1) / 2).permute(1, 2, 0))
ax[0].set_title("Blur Input")
ax[1].imshow(((gt_tensor + 1) / 2).permute(1, 2, 0))
ax[1].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch

# Original crops থেকে নতুন করে তৈরি করো
blur_arr = np.asarray(blur_crop.convert("RGB"), dtype=np.float32) / 255.0
gt_arr = np.asarray(gt_crop.convert("RGB"), dtype=np.float32) / 255.0

# একই normalization: [0, 1] -> [-1, 1]
blur_tensor = torch.from_numpy(blur_arr).permute(2, 0, 1).mul(2).sub(1)
gt_tensor = torch.from_numpy(gt_arr).permute(2, 0, 1).mul(2).sub(1)

print("Input range:", blur_tensor.min().item(), blur_tensor.max().item())
print("Target range:", gt_tensor.min().item(), gt_tensor.max().item())

In [ ]:
import numpy as np
import torch

# Original images থেকে সরাসরি pixel data নেওয়া
blur_raw = np.asarray(blur_original.convert("RGB"), dtype=np.float32) / 255.0
gt_raw = np.asarray(gt_original.convert("RGB"), dtype=np.float32) / 255.0

print("Before normalization:")
print("Blur:", blur_raw.min(), blur_raw.max())
print("GT:", gt_raw.min(), gt_raw.max())

# Normalize to [-1, 1]
blur_norm = torch.from_numpy(blur_raw).permute(2, 0, 1) * 2.0 - 1.0
gt_norm = torch.from_numpy(gt_raw).permute(2, 0, 1) * 2.0 - 1.0

print("\nAfter normalization:")
print("Blur:", blur_norm.min().item(), blur_norm.max().item())
print("GT:", gt_norm.min().item(), gt_norm.max().item())

In [ ]:
print("Training samples:", len(patch_train_ds))
print("Validation samples:", len(patch_val_ds))

print("Train dataset type:", type(patch_train_ds))
print("Validation dataset type:", type(patch_val_ds))

In [ ]:
import dis

dis.dis(PatchRestorationDataset.__getitem__)

In [ ]:
import dis

print("patch_transform type:", type(patch_transform))
dis.dis(patch_transform)

In [ ]:
print(patch_transform)

In [ ]:
import random
import zipfile
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset
from torchvision import transforms

PATCH_SIZE = 128

class HandwritingPatchDataset(Dataset):
    def __init__(self, ids, patches_per_image=8):
        self.ids = ids
        self.patches_per_image = patches_per_image

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]
            )
        ])

    def __len__(self):
        return len(self.ids) * self.patches_per_image

    def __getitem__(self, index):
        idx = self.ids[index // self.patches_per_image]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur3_files[idx]))
            ).convert("RGB")
            clean = Image.open(
                BytesIO(z.read(gt_files[idx]))
            ).convert("RGB")

        if clean.size != blur.size:
            clean = clean.resize(
                blur.size, Image.Resampling.LANCZOS
            )

        w, h = blur.size
        best_box = None
        best_score = -1

        # Choose a crop containing more dark handwriting pixels
        for _ in range(20):
            x = random.randint(0, w - PATCH_SIZE)
            y = random.randint(0, h - PATCH_SIZE)
            box = (x, y, x + PATCH_SIZE, y + PATCH_SIZE)

            candidate = clean.crop(box).convert("L")
            gray = np.asarray(candidate, dtype=np.float32) / 255.0

            score = np.mean(gray < 0.80)

            if score > best_score:
                best_score = score
                best_box = box

        blur_patch = blur.crop(best_box)
        clean_patch = clean.crop(best_box)

        return self.transform(blur_patch), self.transform(clean_patch)

In [ ]:
patch_train_ds = HandwritingPatchDataset(train_ids, patches_per_image=8)
patch_val_ds = HandwritingPatchDataset(val_ids, patches_per_image=2)

x, y = patch_train_ds[0]

print("Input range:", x.min().item(), x.max().item())
print("Target range:", y.min().item(), y.max().item())

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(((x + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[0].set_title("Blur Patch")
ax[1].imshow(((y + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[1].set_title("Ground Truth Patch")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x, y = patch_train_ds[0]

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(((x + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[0].set_title("Blur Input Patch")
ax[0].axis("off")

ax[1].imshow(((y + 1) / 2).clamp(0, 1).permute(1, 2, 0))
ax[1].set_title("Ground Truth Patch")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 8))

ax[0].imshow(blur_original)
ax[0].set_title("Full Blur Image")
ax[0].axis("off")

ax[1].imshow(gt_original)
ax[1].set_title("Full Ground Truth")
ax[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:

import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stable_model = stable_model.to(device)

# Create datasets
patch_train_ds = HandwritingPatchDataset(
    train_ids, patches_per_image=8
)
patch_val_ds = HandwritingPatchDataset(
    val_ids, patches_per_image=2
)

train_loader = DataLoader(
    patch_train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    patch_val_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Loss and optimizer
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(
    stable_model.parameters(),
    lr=0.0001,
    weight_decay=0.00001
)

# Save checkpoints in Google Drive
SAVE_DIR = "/content/gdrive/MyDrive/handwriting_restoration"
os.makedirs(SAVE_DIR, exist_ok=True)
BEST_PATH = os.path.join(SAVE_DIR, "best_stable_model.pt")

best_val_loss = float("inf")
EPOCHS = 3

for epoch in range(EPOCHS):
    stable_model.train()
    train_loss = 0.0

    for blur, clean in train_loader:
        blur = blur.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        output = stable_model(blur)
        loss = criterion(output, clean)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            stable_model.parameters(), max_norm=1.0
        )
        optimizer.step()

        train_loss += loss.item()

    stable_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for blur, clean in val_loader:
            blur = blur.to(device)
            clean = clean.to(device)

            output = stable_model(blur)
            val_loss += criterion(output, clean).item()

    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Validation Loss: {val_loss:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(stable_model.state_dict(), BEST_PATH)
        print("Best model saved!")

print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", BEST_PATH)


In [ ]:

import torch
import matplotlib.pyplot as plt
import numpy as np

# Load the best checkpoint (Epoch 2)
stable_model.load_state_dict(
    torch.load(BEST_PATH, map_location=device)
)
stable_model.eval()

# Restore the complete original page
restored_image = restore_full_page(
    stable_model,
    blur_original
)

# Compare Blur, Ground Truth and Restored
fig, ax = plt.subplots(1, 3, figsize=(18, 9))

ax[0].imshow(blur_original)
ax[0].set_title("Input: Blur Level 3")

ax[1].imshow(gt_original)
ax[1].set_title("Ground Truth")

ax[2].imshow(restored_image)
ax[2].set_title("Model Restored Output")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Input size:", blur_original.size)
print("Ground Truth size:", gt_original.size)
print("Restored size:", restored_image.size)


In [ ]:

import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BEST_PATH = "/content/gdrive/MyDrive/handwriting_restoration/best_stable_model.pt"

assert os.path.exists(BEST_PATH), f"Checkpoint not found: {BEST_PATH}"

# Requires the original StableUNet class definition
stable_model = StableUNet().to(device)

stable_model.load_state_dict(
    torch.load(BEST_PATH, map_location=device)
)

stable_model.eval()
print("Best model loaded successfully!")
print("Device:", device)


In [ ]:

from google.colab import drive
drive.mount('/content/gdrive')

import os

# Check the expected folder
folder = "/content/gdrive/MyDrive/handwriting_restoration"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    print("Files in folder:", os.listdir(folder))

# Search for model checkpoints in MyDrive
root = "/content/gdrive/MyDrive"
found = []

for current_root, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]
    for name in files:
        if name.endswith((".pt", ".pth")) and (
            "stable" in name.lower()
            or "restor" in name.lower()
            or "unet" in name.lower()
        ):
            found.append(os.path.join(current_root, name))

print("\nCheckpoint files found:")
for path in found:
    print(path)


In [ ]:

import os
import torch

BEST_PATH = "/content/gdrive/MyDrive/handwriting_restoration/best_stable_model.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Checkpoint exists:", os.path.isfile(BEST_PATH))
print("Device:", device)
print("StableUNet defined:", "StableUNet" in globals())


In [ ]:

import glob
import os

# Find notebook files in Google Drive
notebooks = glob.glob(
    "/content/gdrive/MyDrive/**/*.ipynb",
    recursive=True
)

print("Notebook files found:", len(notebooks))

for path in notebooks:
    print(path)


In [ ]:

import json
import glob
import os

notebook_paths = glob.glob(
    "/content/gdrive/MyDrive/Colab Notebooks/*.ipynb"
)

keywords = ["class StableUNet", "StableUNet", "def restore_full_page"]

for path in notebook_paths:
    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    matches = []

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))

        if any(word in source for word in keywords):
            matches.append((i, source))

    if matches:
        print("\n" + "=" * 65)
        print("NOTEBOOK:", os.path.basename(path))

        for cell_no, source in matches:
            print(f"\n--- Cell {cell_no} ---")
            print(source[:8000])


In [ ]:

import os
import torch

device = torch.device("cpu")
BEST_PATH = "/content/gdrive/MyDrive/handwriting_restoration/best_stable_model.pt"

stable_model = StableUNet().to(device)

checkpoint = torch.load(BEST_PATH, map_location=device)
stable_model.load_state_dict(checkpoint)
stable_model.eval()

print("Best checkpoint loaded successfully!")
print("Device:", device)


In [ ]:

import os
import torch
import torch.nn as nn

# 1. Define the exact StableUNet architecture
class StableUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(a, b):
            return nn.Sequential(
                nn.Conv2d(a, b, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(b, b, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.e1 = block(3, 16)
        self.e2 = block(16, 32)
        self.e3 = block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(64, 128)

        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d3 = block(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = block(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = block(32, 16)

        self.out = nn.Conv2d(16, 3, 1)

        nn.init.normal_(self.out.weight, mean=0.0, std=0.001)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        z = self.mid(self.pool(c))

        z = self.d3(torch.cat([self.u3(z), c], dim=1))
        z = self.d2(torch.cat([self.u2(z), b], dim=1))
        z = self.d1(torch.cat([self.u1(z), a], dim=1))

        return self.out(z)


# 2. Load the saved best checkpoint
device = torch.device("cpu")
BEST_PATH = "/content/gdrive/MyDrive/handwriting_restoration/best_stable_model.pt"

assert os.path.isfile(BEST_PATH), "Checkpoint file not found!"

stable_model = StableUNet().to(device)
state = torch.load(BEST_PATH, map_location=device, weights_only=True)
stable_model.load_state_dict(state)
stable_model.eval()

print("Success: StableUNet created and checkpoint loaded.")


In [ ]:

import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

test_id = 220

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_original = Image.open(
        BytesIO(z.read(blur3_files[test_id]))
    ).convert("RGB")

    gt_original = Image.open(
        BytesIO(z.read(gt_files[test_id]))
    ).convert("RGB")

print("Blur image size:", blur_original.size)
print("Ground truth size:", gt_original.size)


In [ ]:

import os
import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

# Google Drive mount kora na thakle age ei line run koro
from google.colab import drive
drive.mount('/content/gdrive')

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

assert os.path.exists(ZIP_PATH), f"ZIP file paoa jayni: {ZIP_PATH}"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_files = z.namelist()

blur3_files = sorted([
    f for f in all_files
    if "/Synthetic/Blur/Blur_Level_3/" in f
    and f.lower().endswith((".jpg", ".jpeg", ".png"))
])

gt_files = sorted([
    f for f in all_files
    if "/Ground Truth/" in f
    and f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print("Blur Level 3 images:", len(blur3_files))
print("Ground truth images:", len(gt_files))
print("ZIP path verified:", ZIP_PATH)


In [ ]:

test_id = 220

# Match by image number, not list index, to avoid wrong pairs
def image_number(path):
    import re
    match = re.search(r"\((\d+)\)", os.path.basename(path))
    return int(match.group(1)) if match else None

blur_map = {image_number(f): f for f in blur3_files}
gt_map = {image_number(f): f for f in gt_files}

common_ids = sorted(set(blur_map) & set(gt_map))
assert common_ids, "Matching image pairs paoa jayni."

if test_id not in common_ids:
    test_id = common_ids[0]
    print("220 paoa jayni; use korchi:", test_id)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_original = Image.open(
        BytesIO(z.read(blur_map[test_id]))
    ).convert("RGB")

    gt_original = Image.open(
        BytesIO(z.read(gt_map[test_id]))
    ).convert("RGB")

print("Selected pair:", test_id)
print("Image size:", blur_original.size, gt_original.size)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(blur_original)
plt.title("Blur Level 3")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gt_original)
plt.title("Ground Truth")
plt.axis("off")
plt.show()


In [ ]:

print("Model loaded:", "stable_model" in globals())
print("Restoration function loaded:", "restore_full_page" in globals())
print("Input image loaded:", "blur_original" in globals())


In [ ]:

import numpy as np
import torch
from PIL import Image

def restore_full_page(model, image, device, patch_size=128, overlap=32):
    model.eval()

    # Original image ke RGB numpy array te convert
    img = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    h, w, _ = img.shape
    stride = patch_size - overlap

    # Image boundary cover korar jonno patch positions
    ys = list(range(0, max(h - patch_size, 0) + 1, stride))
    xs = list(range(0, max(w - patch_size, 0) + 1, stride))

    if not ys or ys[-1] + patch_size < h:
        ys.append(max(0, h - patch_size))
    if not xs or xs[-1] + patch_size < w:
        xs.append(max(0, w - patch_size))

    # Overlap blending window
    window_1d = np.hanning(patch_size).astype(np.float32)
    window_1d = np.maximum(window_1d, 0.05)
    window = np.outer(window_1d, window_1d)[..., None]

    output_sum = np.zeros_like(img, dtype=np.float32)
    weight_sum = np.zeros((h, w, 1), dtype=np.float32)

    with torch.no_grad():
        for y in ys:
            for x in xs:
                patch = img[y:y+patch_size, x:x+patch_size]

                # Edge patch choto hole padding
                ph, pw, _ = patch.shape
                if ph != patch_size or pw != patch_size:
                    padded = np.ones(
                        (patch_size, patch_size, 3), dtype=np.float32
                    )
                    padded[:ph, :pw] = patch
                    patch = padded

                tensor = torch.from_numpy(
                    patch.transpose(2, 0, 1).copy()
                ).unsqueeze(0).to(device)

                # Training normalization: [0,1] -> [-1,1]
                tensor = tensor * 2.0 - 1.0
                pred = model(tensor)

                # Model output [-1,1] theke [0,1]
                pred = ((pred.clamp(-1, 1) + 1) / 2)
                pred = pred[0].cpu().numpy().transpose(1, 2, 0)
                pred = pred[:ph, :pw]

                win = window[:ph, :pw]
                output_sum[y:y+ph, x:x+pw] += pred * win
                weight_sum[y:y+ph, x:x+pw] += win

    result = output_sum / np.maximum(weight_sum, 1e-8)
    result = (np.clip(result, 0, 1) * 255).astype(np.uint8)

    return Image.fromarray(result)

print("Restoration function ready!")


In [ ]:

restored = restore_full_page(
    stable_model,
    blur_original,
    device,
    patch_size=128,
    overlap=32
)

import matplotlib.pyplot as plt

plt.figure(figsize=(18, 7))

plt.subplot(1, 3, 1)
plt.imshow(blur_original)
plt.title("Input: Blur Level 3")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(restored)
plt.title("StableUNet Restored")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(gt_original)
plt.title("Ground Truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

print("Blur image:", blur_original.size)
print("Ground truth:", gt_original.size)
print("Restored image:", restored.size)

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.imshow(blur_original)
plt.title("Blur Level 3 - Input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(restored)
plt.title("StableUNet - Restored")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt

# Input image ke model-er training normalization-e convert
arr = np.asarray(blur_original.convert("RGB"), dtype=np.float32) / 255.0

# Choto patch diye raw model output check
patch = arr[:128, :128, :].copy()

x = torch.from_numpy(
    patch.transpose(2, 0, 1).copy()
).unsqueeze(0).to(device)

x = x * 2.0 - 1.0

stable_model.eval()
with torch.no_grad():
    raw = stable_model(x)

print("Input tensor range:", x.min().item(), x.max().item())
print("Raw output min:", raw.min().item())
print("Raw output max:", raw.max().item())
print("Raw output mean:", raw.mean().item())
print("Raw output std:", raw.std().item())

# Duita conversion compare koro
out1 = ((raw.clamp(-1, 1) + 1) / 2)[0].cpu()
out2 = torch.sigmoid(raw)[0].cpu()

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(out1.permute(1, 2, 0).numpy())
ax[0].set_title("Conversion: (output + 1) / 2")
ax[1].imshow(out2.permute(1, 2, 0).numpy())
ax[1].set_title("Conversion: sigmoid(output)")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

img = np.asarray(blur_original.convert("RGB"), dtype=np.float32) / 255.0

# Original image-er dark pixel percentage
gray = img.mean(axis=2)
print("Overall mean brightness:", round(float(gray.mean()), 4))
print("Dark pixels (< 0.80):", round(float((gray < 0.80).mean() * 100), 2), "%")
print("Very dark pixels (< 0.50):", round(float((gray < 0.50).mean() * 100), 2), "%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(blur_original)
plt.title("Full Blur Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(blur_original.crop((0, 0, 512, 512)))
plt.title("Top-left Crop")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile
from PIL import Image
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
import os, re

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    all_names = z.namelist()

blur_paths = [
    p for p in all_names
    if "Blur_Level_3" in p
    and p.lower().endswith((".jpg", ".jpeg", ".png"))
]
gt_paths = [
    p for p in all_names
    if "Ground Truth" in p
    and p.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Blur files found:", len(blur_paths))
print("Ground-truth files found:", len(gt_paths))
print("Example blur path:", blur_paths[0] if blur_paths else "NONE")
print("Example ground-truth path:", gt_paths[0] if gt_paths else "NONE")


In [ ]:

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    sample = Image.open(BytesIO(z.read(blur_paths[0]))).convert("RGB")

sample_arr = np.asarray(sample, dtype=np.float32) / 255.0

print("Sample size:", sample.size)
print("Sample mean brightness:", round(float(sample_arr.mean()), 4))
print("Sample dark-pixel percentage:",
      round(float((sample_arr.mean(axis=2) < 0.80).mean() * 100), 2), "%")

plt.figure(figsize=(7, 8))
plt.imshow(sample)
plt.title("Original ZIP: Blur Level 3")
plt.axis("off")
plt.show()


In [ ]:

import zipfile
from io import BytesIO
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur1 = Image.open(BytesIO(z.read(
        "Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3/Blur_Level_3 (1).jpg"
    ))).convert("RGB")

    gt1 = Image.open(BytesIO(z.read(
        "Koushik_HCCD/HCCD DATASET/Ground Truth/Ground_Truth (1).jpg"
    ))).convert("RGB")

print("Blur 1 size:", blur1.size)
print("GT 1 size:", gt1.size)
print("Blur 1 brightness:", np.asarray(blur1).mean() / 255)
print("GT 1 brightness:", np.asarray(gt1).mean() / 255)

plt.figure(figsize=(12, 7))
plt.subplot(1, 2, 1)
plt.imshow(blur1)
plt.title("Blur Level 3 (1)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gt1)
plt.title("Ground Truth (1)")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
from PIL import Image

blur_arr = np.asarray(blur1, dtype=np.float32) / 255.0
gt_arr = np.asarray(gt1, dtype=np.float32) / 255.0

print("Mean Absolute Difference:", np.abs(blur_arr - gt_arr).mean())
print("Blur standard deviation:", blur_arr.std())
print("Ground Truth standard deviation:", gt_arr.std())


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Same-number pair-er image use korchi
blur_arr = np.asarray(blur1.convert("RGB"), dtype=np.float32) / 255.0
gt_arr = np.asarray(gt1.convert("RGB"), dtype=np.float32) / 255.0

# Ground truth-e handwriting beshi ache emon crop khunji
gray_gt = gt_arr.mean(axis=2)
patch_size = 128
best_score = -1
best_xy = (0, 0)

for y in range(0, gt_arr.shape[0] - patch_size, 64):
    for x in range(0, gt_arr.shape[1] - patch_size, 64):
        crop = gray_gt[y:y+patch_size, x:x+patch_size]
        score = np.mean(crop < 0.90)
        if score > best_score:
            best_score = score
            best_xy = (x, y)

x, y = best_xy
blur_patch = blur_arr[y:y+patch_size, x:x+patch_size]
gt_patch = gt_arr[y:y+patch_size, x:x+patch_size]

print("Selected crop:", best_xy)
print("Ground truth dark pixel ratio:",
      round(float((gt_patch.mean(axis=2) < 0.90).mean() * 100), 2), "%")
print("Blur patch standard deviation:", round(float(blur_patch.std()), 4))
print("GT patch standard deviation:", round(float(gt_patch.std()), 4))

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(blur_patch)
plt.title("Blur patch")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gt_patch)
plt.title("Ground Truth patch")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt

# Previously selected 128x128 paired patches
blur_tensor = torch.from_numpy(
    blur_patch.transpose(2, 0, 1).copy()
).float() * 2 - 1

gt_tensor = torch.from_numpy(
    gt_patch.transpose(2, 0, 1).copy()
).float() * 2 - 1

print("Blur tensor range:",
      round(blur_tensor.min().item(), 3),
      round(blur_tensor.max().item(), 3))
print("GT tensor range:",
      round(gt_tensor.min().item(), 3),
      round(gt_tensor.max().item(), 3))

print("Blur tensor mean:", round(blur_tensor.mean().item(), 3))
print("GT tensor mean:", round(gt_tensor.mean().item(), 3))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(((blur_tensor + 1) / 2).permute(1, 2, 0).numpy())
ax[0].set_title("Normalized Blur")
ax[1].imshow(((gt_tensor + 1) / 2).permute(1, 2, 0).numpy())
ax[1].set_title("Normalized Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Existing model-er copy; saved checkpoint change hobe na
test_model = copy.deepcopy(stable_model).to(device)
test_model.train()

optimizer = torch.optim.Adam(test_model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

x = blur_tensor.unsqueeze(0).to(device)
y = gt_tensor.unsqueeze(0).to(device)

losses = []

for step in range(100):
    optimizer.zero_grad()
    pred = test_model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("Initial loss:", round(losses[0], 4))
print("Final loss:", round(losses[-1], 4))

test_model.eval()
with torch.no_grad():
    pred = test_model(x)[0].clamp(-1, 1)
    pred_img = ((pred + 1) / 2).cpu().permute(1, 2, 0).numpy()

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(((x[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Blur input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_img)
plt.title("After 100 steps")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(((y[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Ground truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

test_model.eval()

with torch.no_grad():
    pred = test_model(x)

print("Prediction min:", round(pred.min().item(), 4))
print("Prediction max:", round(pred.max().item(), 4))
print("Prediction mean:", round(pred.mean().item(), 4))
print("Prediction std:", round(pred.std().item(), 4))

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(((x[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Blur input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(((pred[0].clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Model prediction")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(((y[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Ground truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

test_model.train()

optimizer = torch.optim.Adam(test_model.parameters(), lr=2e-4)
criterion = nn.L1Loss()

losses = []

for step in range(1000):
    optimizer.zero_grad()
    pred = test_model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (step + 1) % 200 == 0:
        print(f"Step {step+1}: loss = {loss.item():.4f}")

test_model.eval()
with torch.no_grad():
    pred = test_model(x)

print("Final prediction range:",
      round(pred.min().item(), 4),
      round(pred.max().item(), 4))
print("Final prediction mean:", round(pred.mean().item(), 4))
print("Final prediction std:", round(pred.std().item(), 4))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(((x[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Blur input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(((pred[0].clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("After patch training")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(((y[0] + 1) / 2).cpu().permute(1, 2, 0).numpy())
plt.title("Ground truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import torch

test_model.eval()
with torch.no_grad():
    pred = test_model(x)[0].clamp(-1, 1)

pred_img = ((pred + 1) / 2).cpu().permute(1, 2, 0).numpy()
input_img = ((x[0] + 1) / 2).cpu().permute(1, 2, 0).numpy()
target_img = ((y[0] + 1) / 2).cpu().permute(1, 2, 0).numpy()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(input_img)
plt.title("Blur input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_img)
plt.title("Restored patch")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(target_img)
plt.title("Ground truth")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
import matplotlib.pyplot as plt

test_model.eval()

with torch.no_grad():
    pred = test_model(x)

def stats(name, tensor):
    print(
        name,
        "| min:", round(tensor.min().item(), 3),
        "| max:", round(tensor.max().item(), 3),
        "| mean:", round(tensor.mean().item(), 3),
        "| std:", round(tensor.std().item(), 3)
    )

stats("Blur input", x)
stats("Ground truth", y)
stats("Prediction raw", pred)

# Compare images using the SAME [-1,1] -> [0,1] conversion
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(((x[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy())
ax[0].set_title("Blur input")

ax[1].imshow(((pred[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy())
ax[1].set_title("Prediction")

ax[2].imshow(((y[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy())
ax[2].set_title("Ground truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

TEST_PATH = "/content/gdrive/MyDrive/handwriting_restoration/test_patch_model.pt"

torch.save(test_model.state_dict(), TEST_PATH)
print("Test model saved:", TEST_PATH)


In [ ]:

test_model.eval()

with torch.no_grad():
    pred = test_model(x)[0].clamp(-1, 1)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(((x[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy())
ax[0].set_title("Blur Input")

ax[1].imshow(((pred + 1) / 2).cpu().permute(1, 2, 0).numpy())
ax[1].set_title("Prediction after 1000 steps")

ax[2].imshow(((y[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy())
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Existing 128x128 patch-ke aro boro kore dekhano
blur_view = ((x[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
gt_view = ((y[0] + 1) / 2).clamp(0, 1).cpu().permute(1, 2, 0).numpy()

test_model.eval()
with torch.no_grad():
    pred_view = ((test_model(x)[0] + 1) / 2).clamp(0, 1)
    pred_view = pred_view.cpu().permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(blur_view, interpolation="nearest")
ax[0].set_title("Blur Input — Zoomed")

ax[1].imshow(pred_view, interpolation="nearest")
ax[1].set_title("Restored — Zoomed")

ax[2].imshow(gt_view, interpolation="nearest")
ax[2].set_title("Ground Truth — Zoomed")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from PIL import Image
import matplotlib.pyplot as plt

# Ground Truth crop — lekhar dark area theke
img = gt1.convert("RGB")

# Age je crop-e lekha select korechhile, oi area
crop = img.crop((640, 512, 768, 640))

# 4x zoom
zoom = crop.resize(
    (crop.width * 4, crop.height * 4),
    Image.Resampling.NEAREST
)

plt.figure(figsize=(12, 12))
plt.imshow(zoom)
plt.title("Handwriting — 4x Zoom")
plt.axis("off")
plt.show()


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))  # Age 12x12 chilo, ekhon chhoto
plt.imshow(pred_view)
plt.title("Restored Handwriting — Zoom Out")
plt.axis("off")
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Full original image — crop kora hobe na
img = gt1.convert("RGB")

# Image-ke chhoto kore display-er jonno resize
img.thumbnail((500, 650), Image.Resampling.LANCZOS)

plt.figure(figsize=(4, 5))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch

stable_model.eval()

# Full input image
input_img = blur1.convert("RGB")

# Ground truth
target_img = gt1.convert("RGB")

# Model output-er jonno 128x128 patch
with torch.no_grad():
    output = test_model(x)[0]
    output = ((output + 1) / 2).clamp(0, 1)
    output = output.cpu().permute(1, 2, 0).numpy()

# Shob image-ke same display size-e ana
input_img.thumbnail((350, 470))
target_img.thumbnail((350, 470))

fig, ax = plt.subplots(1, 3, figsize=(12, 5))
ax[0].imshow(input_img)
ax[0].set_title("Blur Input — Full Page")
ax[1].imshow(output)
ax[1].set_title("Restored — 128px Patch")
ax[2].imshow(target_img)
ax[2].set_title("Ground Truth — Full Page")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

fig, ax = plt.subplots(1, 3, figsize=(9, 3))

ax[0].imshow(blur_patch)
ax[0].set_title("Blur Input")

ax[1].imshow(output)
ax[1].set_title("Restored")

ax[2].imshow(gt_patch)
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile, re
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

blur_name = next(
    n for n in names
    if n.endswith("Blur_Level_3 (1).jpg")
)
gt_name = next(
    n for n in names
    if n.endswith("Ground_Truth (1).jpg")
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur_img = Image.open(BytesIO(z.read(blur_name))).convert("RGB")
    gt_img = Image.open(BytesIO(z.read(gt_name))).convert("RGB")

print("Blur:", blur_img.size)
print("Ground Truth:", gt_img.size)
print("Blur file:", blur_name)
print("GT file:", gt_name)

# Same crop coordinates
box = (640, 512, 768, 640)
blur_crop = blur_img.crop(box)
gt_crop = gt_img.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(blur_crop)
ax[0].set_title("Blur Level 3")
ax[1].imshow(gt_crop)
ax[1].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

# Lekha ache emon area-r crop
box = (640, 512, 768, 640)

blur_crop = blur_img.crop(box)
gt_crop = gt_img.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].imshow(blur_crop.resize((384, 384)))
ax[0].set_title("Blur Level 3 — 3x")

ax[1].imshow(gt_crop.resize((384, 384)))
ax[1].set_title("Ground Truth — 3x")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Same crop-er image duita grayscale-e convert
blur_gray = np.array(blur_crop.convert("L"), dtype=np.float32) / 255.0
gt_gray = np.array(gt_crop.convert("L"), dtype=np.float32) / 255.0

# Image-er size ar difference check
print("Blur shape:", blur_gray.shape)
print("Ground Truth shape:", gt_gray.shape)
print("Mean absolute difference:", np.mean(np.abs(blur_gray - gt_gray)))

fig, ax = plt.subplots(1, 3, figsize=(10, 4))
ax[0].imshow(blur_gray, cmap="gray", vmin=0, vmax=1)
ax[0].set_title("Blur Input")
ax[1].imshow(gt_gray, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("Ground Truth")
ax[2].imshow(np.abs(blur_gray - gt_gray), cmap="hot")
ax[2].set_title("Pixel Difference")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

test_model.eval()

with torch.no_grad():
    pred = test_model(x)

print("Input min/max:", x.min().item(), x.max().item())
print("Ground Truth min/max:", y.min().item(), y.max().item())
print("Prediction min/max:", pred.min().item(), pred.max().item())
print("Prediction mean:", pred.mean().item())
print("Prediction std:", pred.std().item())


In [ ]:

import torch
import matplotlib.pyplot as plt

test_model.eval()

with torch.no_grad():
    pred_raw = test_model(x)

    # Output-ke valid normalized range-e rakha
    pred_fixed = pred_raw.clamp(-1, 1)

    # [-1, 1] theke [0, 1]
    blur_show = ((x[0] + 1) / 2).clamp(0, 1)
    gt_show = ((y[0] + 1) / 2).clamp(0, 1)
    pred_show = ((pred_fixed[0] + 1) / 2).clamp(0, 1)

fig, ax = plt.subplots(1, 3, figsize=(9, 4))

ax[0].imshow(blur_show.cpu().permute(1, 2, 0).numpy())
ax[0].set_title("Blur Input")

ax[1].imshow(pred_show.cpu().permute(1, 2, 0).numpy())
ax[1].set_title("Restored (Clamped)")

ax[2].imshow(gt_show.cpu().permute(1, 2, 0).numpy())
ax[2].set_title("Ground Truth")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Raw prediction range:",
      pred_raw.min().item(), pred_raw.max().item())
print("Clamped prediction range:",
      pred_fixed.min().item(), pred_fixed.max().item())


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(9, 4))

images = [
    ((x[0] + 1) / 2).clamp(0, 1),
    ((pred_fixed[0] + 1) / 2).clamp(0, 1),
    ((y[0] + 1) / 2).clamp(0, 1)
]
titles = ["Blur Input", "Restored Output", "Ground Truth"]

for a, image, title in zip(ax, images, titles):
    a.imshow(image.detach().cpu().permute(1, 2, 0).numpy())
    a.set_title(title)
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

blur_path = next(
    n for n in names
    if n.endswith("Blur_Level_3 (1).jpg")
)
gt_path = next(
    n for n in names
    if n.endswith("Ground_Truth (1).jpg")
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    blur = Image.open(BytesIO(z.read(blur_path))).convert("RGB")
    gt = Image.open(BytesIO(z.read(gt_path))).convert("RGB")

# Same location-er crop
box = (640, 512, 768, 640)
blur_crop = blur.crop(box)
gt_crop = gt.crop(box)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(blur_crop)
ax[0].set_title("Original Blur Pair")
ax[1].imshow(gt_crop)
ax[1].set_title("Original Ground Truth Pair")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

gray = np.array(gt.convert("L"), dtype=np.float32) / 255.0
blur_gray = np.array(blur.convert("L"), dtype=np.float32) / 255.0

patch_size = 128
candidates = []

# Page-er bibhinno jayga check
for yy in range(0, gt.height - patch_size, 64):
    for xx in range(0, gt.width - patch_size, 64):
        patch = gray[yy:yy+patch_size, xx:xx+patch_size]
        dark_ratio = np.mean(patch < 0.75)
        if dark_ratio > 0.08:
            candidates.append((dark_ratio, xx, yy))

# Lekha beshi ache emon 6 ta crop
candidates.sort(reverse=True)
selected = []
for score, xx, yy in candidates:
    if all(abs(xx-sx) > 100 or abs(yy-sy) > 100
           for _, sx, sy in selected):
        selected.append((score, xx, yy))
    if len(selected) == 6:
        break

fig, ax = plt.subplots(len(selected), 2,
                       figsize=(7, 3 * len(selected)))
for i, (score, xx, yy) in enumerate(selected):
    ax[i, 0].imshow(blur.crop(
        (xx, yy, xx+patch_size, yy+patch_size)))
    ax[i, 0].set_title(f"Blur ({xx}, {yy})")
    ax[i, 1].imshow(gt.crop(
        (xx, yy, xx+patch_size, yy+patch_size)))
    ax[i, 1].set_title(f"Ground Truth ({xx}, {yy})")
    ax[i, 0].axis("off")
    ax[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
from PIL import Image

PATCH_DIR = "/content/gdrive/MyDrive/handwriting_restoration/paired_patches"
os.makedirs(PATCH_DIR, exist_ok=True)

patch_size = 128

for i, (score, xx, yy) in enumerate(selected):
    blur_patch_img = blur.crop(
        (xx, yy, xx + patch_size, yy + patch_size)
    )
    gt_patch_img = gt.crop(
        (xx, yy, xx + patch_size, yy + patch_size)
    )

    blur_patch_img.save(f"{PATCH_DIR}/blur_{i:02d}.png")
    gt_patch_img.save(f"{PATCH_DIR}/gt_{i:02d}.png")

print("Saved paired crops:", len(selected))
print("Folder:", PATCH_DIR)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

i = 0
blur_check = Image.open(f"{PATCH_DIR}/blur_{i:02d}.png")
gt_check = Image.open(f"{PATCH_DIR}/gt_{i:02d}.png")

fig, ax = plt.subplots(1, 2, figsize=(7, 4))
ax[0].imshow(blur_check)
ax[0].set_title("Blur Patch")
ax[1].imshow(gt_check)
ax[1].set_title("Ground Truth Patch")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import os

PATCH_DIR = "/content/gdrive/MyDrive/handwriting_restoration/paired_patches"

fig, ax = plt.subplots(6, 2, figsize=(8, 15))

for i in range(6):
    blur = Image.open(f"{PATCH_DIR}/blur_{i:02d}.png")
    gt = Image.open(f"{PATCH_DIR}/gt_{i:02d}.png")

    ax[i, 0].imshow(blur)
    ax[i, 0].set_title(f"Blur {i}")

    ax[i, 1].imshow(gt)
    ax[i, 1].set_title(f"Ground Truth {i}")

    ax[i, 0].axis("off")
    ax[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import zipfile
import re
from collections import defaultdict

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

def get_number(path):
    match = re.search(r"\((\d+)\)\.jpg$", path)
    return int(match.group(1)) if match else None

blur_map = {
    get_number(n): n for n in names
    if "/Blur_Level_3/" in n and get_number(n) is not None
}
gt_map = {
    get_number(n): n for n in names
    if "/Ground Truth/" in n and get_number(n) is not None
}

common_ids = sorted(set(blur_map) & set(gt_map))

print("Blur images:", len(blur_map))
print("Ground Truth images:", len(gt_map))
print("Matched pairs:", len(common_ids))
print("First 10 IDs:", common_ids[:10])
print("Missing Ground Truth IDs:",
      sorted(set(blur_map) - set(gt_map))[:20])
print("Missing Blur IDs:",
      sorted(set(gt_map) - set(blur_map))[:20])


In [ ]:

import zipfile, re, random
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducible split
random.seed(42)
torch.manual_seed(42)

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"
PATCH_SIZE = 128
PATCHES_PER_IMAGE = 8

# Number-based pairing maps age toiri kora blur_map, gt_map, common_ids use korchi
all_ids = list(common_ids)
random.shuffle(all_ids)

split_idx = int(0.9 * len(all_ids))
train_ids = sorted(all_ids[:split_idx])
val_ids = sorted(all_ids[split_idx:])

print("Train images:", len(train_ids))
print("Validation images:", len(val_ids))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

class PairedPatchDataset(Dataset):
    def __init__(self, ids, patches_per_image=8):
        self.ids = ids
        self.patches_per_image = patches_per_image

    def __len__(self):
        return len(self.ids) * self.patches_per_image

    def __getitem__(self, index):
        img_id = self.ids[index // self.patches_per_image]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur_map[img_id]))
            ).convert("RGB")
            gt = Image.open(
                BytesIO(z.read(gt_map[img_id]))
            ).convert("RGB")

        if blur.size != gt.size:
            gt = gt.resize(blur.size, Image.Resampling.LANCZOS)

        w, h = blur.size
        best_box, best_score = None, -1

        # Ground Truth-e handwriting ache emon crop select kora
        for _ in range(20):
            xx = random.randint(0, w - PATCH_SIZE)
            yy = random.randint(0, h - PATCH_SIZE)
            box = (xx, yy, xx + PATCH_SIZE, yy + PATCH_SIZE)

            gray = np.asarray(
                gt.crop(box).convert("L"), dtype=np.float32
            ) / 255.0
            score = np.mean(gray < 0.80)

            if score > best_score:
                best_score = score
                best_box = box

        blur_patch = blur.crop(best_box)
        gt_patch = gt.crop(best_box)

        return transform(blur_patch), transform(gt_patch)

train_ds = PairedPatchDataset(train_ids, PATCHES_PER_IMAGE)
val_ds = PairedPatchDataset(val_ids, 2)

train_loader = DataLoader(
    train_ds, batch_size=8, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=8, shuffle=False, num_workers=0
)

# Batch shape verify
xb, yb = next(iter(train_loader))
print("Input batch:", xb.shape)
print("Target batch:", yb.shape)
print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())


In [ ]:

from torchvision import transforms
import torch

# PIL image [0,1] tensor-e convert kore [-1,1] e normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# Dataset class-er __getitem__ method-e ei return line thakbe:
# return transform(blur_patch), transform(gt_patch)

# Dataset object abar create koro
train_ds = PairedPatchDataset(train_ids, PATCHES_PER_IMAGE)
val_ds = PairedPatchDataset(val_ids, 2)

train_loader = DataLoader(
    train_ds, batch_size=8, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=8, shuffle=False, num_workers=0
)

xb, yb = next(iter(train_loader))

print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())
print("Shapes:", xb.shape, yb.shape)


In [ ]:

class PairedPatchDataset(Dataset):
    def __init__(self, ids, patches_per_image=8):
        self.ids = ids
        self.patches_per_image = patches_per_image

    def __len__(self):
        return len(self.ids) * self.patches_per_image

    def __getitem__(self, index):
        img_id = self.ids[index // self.patches_per_image]

        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            blur = Image.open(
                BytesIO(z.read(blur_map[img_id]))
            ).convert("RGB")
            gt = Image.open(
                BytesIO(z.read(gt_map[img_id]))
            ).convert("RGB")

        if blur.size != gt.size:
            gt = gt.resize(blur.size, Image.Resampling.LANCZOS)

        w, h = blur.size
        best_box, best_score = None, -1

        for _ in range(20):
            xx = random.randint(0, w - PATCH_SIZE)
            yy = random.randint(0, h - PATCH_SIZE)
            box = (xx, yy, xx + PATCH_SIZE, yy + PATCH_SIZE)

            gray = np.asarray(
                gt.crop(box).convert("L"), dtype=np.float32
            ) / 255.0
            score = np.mean(gray < 0.80)

            if score > best_score:
                best_score = score
                best_box = box

        blur_patch = blur.crop(best_box)
        gt_patch = gt.crop(best_box)

        # Duita image-er jonno eki normalization
        blur_tensor = transforms.ToTensor()(blur_patch)
        gt_tensor = transforms.ToTensor()(gt_patch)

        blur_tensor = (blur_tensor - 0.5) / 0.5
        gt_tensor = (gt_tensor - 0.5) / 0.5

        return blur_tensor, gt_tensor


train_ds = PairedPatchDataset(train_ids, PATCHES_PER_IMAGE)
val_ds = PairedPatchDataset(val_ids, 2)

train_loader = DataLoader(
    train_ds, batch_size=8, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=8, shuffle=False, num_workers=0
)

xb, yb = next(iter(train_loader))
print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())


In [ ]:

# Existing batch-ke direct normalize kora
xb_fixed = (xb - 0.5) / 0.5

print("Before:", xb.min().item(), xb.max().item())
print("After :", xb_fixed.min().item(), xb_fixed.max().item())
print("Target:", yb.min().item(), yb.max().item())

# Normalized input range verify
assert xb_fixed.min() >= -1.0
assert xb_fixed.max() <= 1.0
print("Normalization check passed!")


In [ ]:

import torch
import torch.nn as nn
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Scratch theke notun model — old checkpoint load korbe na
model = StableUNet().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

BEST_PATH = (
    "/content/gdrive/MyDrive/handwriting_restoration/"
    "best_normalized_model.pt"
)

best_val = float("inf")
epochs = 3  # Prothome 3 epoch test

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        # Loader input [0,1] range-e dey; model-er jonno [-1,1]
        xb = ((xb - 0.5) / 0.5).to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = ((xb - 0.5) / 0.5).to(device)
            yb = yb.to(device)

            pred = model(xb)
            val_loss += criterion(pred, yb).item()

    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train L1: {train_loss:.4f} | "
        f"Validation L1: {val_loss:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), BEST_PATH)
        print("Best model saved.")

print("Training complete.")
print("Best validation loss:", best_val)
print("Checkpoint:", BEST_PATH)


In [ ]:

import torch
import torch.nn as nn

class StableUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(a, b):
            return nn.Sequential(
                nn.Conv2d(a, b, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(b, b, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.e1 = block(3, 16)
        self.e2 = block(16, 32)
        self.e3 = block(32, 64)

        self.pool = nn.MaxPool2d(2)
        self.mid = block(64, 128)

        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.d3 = block(128, 64)

        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d2 = block(64, 32)

        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.d1 = block(32, 16)

        self.out = nn.Conv2d(16, 3, 1)
        nn.init.normal_(self.out.weight, mean=0.0, std=0.001)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        z = self.mid(self.pool(c))

        z = self.d3(torch.cat([self.u3(z), c], dim=1))
        z = self.d2(torch.cat([self.u2(z), b], dim=1))
        z = self.d1(torch.cat([self.u1(z), a], dim=1))

        return self.out(z)

# Model test
model = StableUNet()
test_input = torch.randn(1, 3, 128, 128)
test_output = model(test_input)

print("Model created successfully!")
print("Output shape:", test_output.shape)


In [ ]:

import torch
import torch.nn as nn
import os

device = torch.device("cpu")
model = StableUNet().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

BEST_PATH = (
    "/content/gdrive/MyDrive/handwriting_restoration/"
    "best_normalized_model.pt"
)
os.makedirs(os.path.dirname(BEST_PATH), exist_ok=True)

best_val = float("inf")
epochs = 3

for epoch in range(epochs):
    model.train()
    train_total = 0.0

    for xb, yb in train_loader:
        # Input [0,1] theke [-1,1] e convert
        xb = ((xb - 0.5) / 0.5).to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_total += loss.item()

    train_loss = train_total / len(train_loader)

    model.eval()
    val_total = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = ((xb - 0.5) / 0.5).to(device)
            yb = yb.to(device)

            pred = model(xb)
            val_total += criterion(pred, yb).item()

    val_loss = val_total / len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train L1: {train_loss:.4f} | "
        f"Validation L1: {val_loss:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), BEST_PATH)
        print("Best checkpoint saved!")

print("Training complete.")
print("Best validation loss:", best_val)
print("Saved at:", BEST_PATH)


In [ ]:

from torch.utils.data import DataLoader

# Age dataset class PairedPatchDataset define kora thakte hobe
train_dataset = PairedPatchDataset(
    train_ids, blur_map, gt_map,
    patch_size=PATCH_SIZE, patches_per_image=8
)

val_dataset = PairedPatchDataset(
    val_ids, blur_map, gt_map,
    patch_size=PATCH_SIZE, patches_per_image=2
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

xb, yb = next(iter(train_loader))
print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())


In [ ]:

import random
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

class PairedPatchDataset(Dataset):
    def __init__(
        self, ids, blur_map, gt_map,
        patch_size=128, patches_per_image=8
    ):
        self.ids = list(ids)
        self.blur_map = blur_map
        self.gt_map = gt_map
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image

    def __len__(self):
        return len(self.ids) * self.patches_per_image

    def __getitem__(self, idx):
        image_id = self.ids[idx // self.patches_per_image]

        blur = Image.open(
            self.blur_map[image_id]
        ).convert("RGB")

        gt = Image.open(
            self.gt_map[image_id]
        ).convert("RGB")

        if blur.size != gt.size:
            gt = gt.resize(blur.size, Image.Resampling.BICUBIC)

        w, h = blur.size
        p = self.patch_size

        if w < p or h < p:
            blur = blur.resize(
                (max(w, p), max(h, p)),
                Image.Resampling.BICUBIC
            )
            gt = gt.resize(blur.size, Image.Resampling.BICUBIC)
            w, h = blur.size

        # Lekha ache emon region-er patch select
        best_box = None
        best_score = -1

        for _ in range(20):
            x = random.randint(0, w - p)
            y = random.randint(0, h - p)

            gt_crop = gt.crop((x, y, x+p, y+p))
            gray = np.asarray(gt_crop.convert("L"))
            score = np.mean(gray < 200)

            if score > best_score:
                best_score = score
                best_box = (x, y, x+p, y+p)

        blur = blur.crop(best_box)
        gt = gt.crop(best_box)

        blur = np.asarray(blur, dtype=np.float32) / 255.0
        gt = np.asarray(gt, dtype=np.float32) / 255.0

        blur = torch.from_numpy(blur).permute(2, 0, 1)
        gt = torch.from_numpy(gt).permute(2, 0, 1)

        # Input ebong target dutoi [-1, 1]
        blur = (blur - 0.5) / 0.5
        gt = (gt - 0.5) / 0.5

        return blur, gt

print("PairedPatchDataset class ready!")


In [ ]:

from torch.utils.data import DataLoader

train_dataset = PairedPatchDataset(
    train_ids,
    blur_map,
    gt_map,
    patch_size=128,
    patches_per_image=8
)

val_dataset = PairedPatchDataset(
    val_ids,
    blur_map,
    gt_map,
    patch_size=128,
    patches_per_image=2
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

xb, yb = next(iter(train_loader))
print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())
print("Input shape:", xb.shape)
print("Target shape:", yb.shape)


In [ ]:

import random

# Blur ebong Ground Truth-er common image ID ber kora
common_ids = sorted(
    set(blur_map.keys()) & set(gt_map.keys()),
    key=int
)

print("Total paired images:", len(common_ids))

# Reproducible random split
random.seed(42)
random.shuffle(common_ids)

split_index = int(len(common_ids) * 0.90)

train_ids = common_ids[:split_index]
val_ids = common_ids[split_index:]

print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))
print("First training IDs:", train_ids[:5])


In [ ]:

import os
import glob
import re

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

# Extracted dataset folder khোঁজা
BASE_DIR = "/content/gdrive/MyDrive/Koushik_HCCD/HCCD DATASET"

if not os.path.isdir(BASE_DIR):
    BASE_DIR = "/content/Koushik_HCCD/HCCD DATASET"

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        "Dataset folder paoa jayni. ZIP extract kora ache kina check koro."
    )

GT_DIR = os.path.join(BASE_DIR, "Ground Truth")
BLUR_DIR = os.path.join(
    BASE_DIR, "Synthetic", "Blur", "Blur_Level_3"
)

def build_map(folder, prefix):
    result = {}
    for path in glob.glob(os.path.join(folder, "*.jpg")):
        name = os.path.basename(path)
        match = re.fullmatch(
            rf"{re.escape(prefix)}\s*\((\d+)\)\.jpg",
            name,
            flags=re.IGNORECASE
        )
        if match:
            result[int(match.group(1))] = path
    return result

gt_map = build_map(GT_DIR, "Ground_Truth")
blur_map = build_map(BLUR_DIR, "Blur_Level_3")

print("Ground Truth images:", len(gt_map))
print("Blur Level 3 images:", len(blur_map))
print("Ground Truth folder:", GT_DIR)
print("Blur folder:", BLUR_DIR)


In [ ]:

from google.colab import drive
drive.mount('/content/gdrive')

import os
import zipfile

ZIP_PATH = "/content/gdrive/MyDrive/Koushik_HCCD.zip"

if not os.path.isfile(ZIP_PATH):
    print("ZIP file ei path-e nei.")
    print("MyDrive-er moddhe ZIP files:")
    for root, dirs, files in os.walk("/content/gdrive/MyDrive"):
        for file in files:
            if file.lower().endswith(".zip") and "hccd" in file.lower():
                print(os.path.join(root, file))
else:
    EXTRACT_DIR = "/content/gdrive/MyDrive/Koushik_HCCD"

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    print("Extraction complete!")
    print("Extract folder:", EXTRACT_DIR)

    for root, dirs, files in os.walk(EXTRACT_DIR):
        if "Ground Truth" in dirs:
            print("Dataset folder found:", root)


In [ ]:

from google.colab import drive
import os

MOUNT_PATH = "/content/drive2"

os.makedirs(MOUNT_PATH, exist_ok=True)
drive.mount(MOUNT_PATH)

print("Drive mounted:", os.path.isdir(MOUNT_PATH))
print("MyDrive exists:",
      os.path.isdir(MOUNT_PATH + "/MyDrive"))


In [ ]:

import os

MYDRIVE = "/content/drive2/MyDrive"

found = []

for root, dirs, files in os.walk(MYDRIVE):
    for file in files:
        if file.lower().endswith(".zip") and "hccd" in file.lower():
            path = os.path.join(root, file)
            found.append(path)
            print("ZIP found:", path)

if not found:
    print("HCCD ZIP paoa jayni. MyDrive-er first-level files:")
    print(os.listdir(MYDRIVE)[:50])


In [ ]:

import os
import zipfile
import glob
import re

ZIP_PATH = "/content/drive2/MyDrive/Koushik_HCCD.zip"
EXTRACT_DIR = "/content/drive2/MyDrive/Koushik_HCCD"

# ZIP extract
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extraction complete!")

# Ground Truth folder khোঁজা
gt_dirs = glob.glob(
    EXTRACT_DIR + "/**/Ground Truth",
    recursive=True
)

if not gt_dirs:
    raise FileNotFoundError(
        "Ground Truth folder paoa jayni. ZIP-er folder structure check korte hobe."
    )

GT_DIR = gt_dirs[0]
BASE_DIR = os.path.dirname(GT_DIR)

# Blur Level 3 folder khোঁজা
blur_dirs = glob.glob(
    BASE_DIR + "/**/Blur_Level_3",
    recursive=True
)

if not blur_dirs:
    raise FileNotFoundError("Blur_Level_3 folder paoa jayni.")

BLUR_DIR = blur_dirs[0]

def build_map(folder, prefix):
    result = {}
    for path in glob.glob(os.path.join(folder, "*.jpg")):
        name = os.path.basename(path)
        match = re.fullmatch(
            rf"{re.escape(prefix)}\s*\((\d+)\)\.jpg",
            name,
            flags=re.IGNORECASE
        )
        if match:
            result[int(match.group(1))] = path
    return result

gt_map = build_map(GT_DIR, "Ground_Truth")
blur_map = build_map(BLUR_DIR, "Blur_Level_3")

common_ids = sorted(
    set(gt_map) & set(blur_map)
)

print("Ground Truth:", len(gt_map))
print("Blur Level 3:", len(blur_map))
print("Matched pairs:", len(common_ids))
print("GT folder:", GT_DIR)
print("Blur folder:", BLUR_DIR)


In [ ]:

import random

random.seed(42)
random.shuffle(common_ids)

split_index = int(len(common_ids) * 0.90)
train_ids = common_ids[:split_index]
val_ids = common_ids[split_index:]

print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))


In [ ]:

from torch.utils.data import DataLoader

PATCH_SIZE = 128

train_dataset = PairedPatchDataset(
    train_ids,
    blur_map,
    gt_map,
    patch_size=PATCH_SIZE,
    patches_per_image=8
)

val_dataset = PairedPatchDataset(
    val_ids,
    blur_map,
    gt_map,
    patch_size=PATCH_SIZE,
    patches_per_image=2
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

xb, yb = next(iter(train_loader))
print("Input shape:", xb.shape)
print("Target shape:", yb.shape)
print("Input range:", xb.min().item(), xb.max().item())
print("Target range:", yb.min().item(), yb.max().item())


In [ ]:

import torch
import torch.nn as nn
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = StableUNet().to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

BEST_PATH = (
    "/content/drive2/MyDrive/handwriting_restoration/"
    "best_normalized_model.pt"
)
os.makedirs(os.path.dirname(BEST_PATH), exist_ok=True)

best_val = float("inf")
epochs = 3

for epoch in range(epochs):
    model.train()
    train_total = 0.0

    for xb, yb in train_loader:
        # Input 0-1 theke -1 to 1 normalize
        xb = ((xb - 0.5) / 0.5).to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_total += loss.item()

    train_loss = train_total / len(train_loader)

    model.eval()
    val_total = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = ((xb - 0.5) / 0.5).to(device)
            yb = yb.to(device)

            pred = model(xb)
            val_total += criterion(pred, yb).item()

    val_loss = val_total / len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train L1: {train_loss:.4f} | "
        f"Validation L1: {val_loss:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), BEST_PATH)
        print("Best model saved!")

print("Training complete!")
print("Best validation loss:", best_val)
print("Model path:", BEST_PATH)


In [ ]:
class DeblurDataset(Dataset):
    def __init__(self, blur_dir, sharp_dir, transform=None):
        """
        blur_dir: ব্লার ইমেজের ফোল্ডার পাথ
        sharp_dir: পরিষ্কার (Ground Truth) ইমেজের ফোল্ডার পাথ
        """
        self.blur_dir = blur_dir
        self.sharp_dir = sharp_dir
        self.transform = transform
        # ফাইলের নামের তালিকা তৈরি (ধরে নিচ্ছি blur এবং sharp ইমেজের ফাইলনেম একই)
        self.image_names = sorted(os.listdir(blur_dir))

        # যাচাই করুন যে প্রতিটি ব্লার ইমেজের জন্য একটি পরিষ্কার ইমেজ আছে
        # (ঐচ্ছিক, তবে ডেটা সঠিকতা নিশ্চিত করার জন্য ভালো)
        valid_names = []
        for name in self.image_names:
            if os.path.exists(os.path.join(sharp_dir, name)):
                valid_names.append(name)
        self.image_names = valid_names
        print(f"মোট {len(self.image_names)} জোড়া ইমেজ পাওয়া গেছে।")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        blur_path = os.path.join(self.blur_dir, img_name)
        sharp_path = os.path.join(self.sharp_dir, img_name)

        blur_img = Image.open(blur_path).convert("RGB")
        sharp_img = Image.open(sharp_path).convert("RGB")

        if self.transform:
            blur_img = self.transform(blur_img)
            sharp_img = self.transform(sharp_img)

        return blur_img, sharp_img

# ট্রান্সফর্ম ডিফাইন করুন (ইমেজকে টেনসরে রূপান্তর এবং সাধারণত [0,1] বা [-1,1] রেঞ্জে নরমালাইজ)
# এখানে আমরা [0,1] ব্যবহার করছি, কারণ আউটপুটে Sigmoid ব্যবহার করব।
transform = transforms.Compose([
    transforms.Resize((256, 256)), # মেমরি সাশ্রয়ের জন্য ইমেজ রিসাইজ (প্রয়োজন অনুযায়ী পরিবর্তন করুন)
    transforms.ToTensor(), # [0, 255] -> [0.0, 1.0]
])

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader   # ← এটাই missing ছিল
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import os

# আপনার আনজিপ করা ডেটাসেটের মূল পাথ (প্রয়োজন অনুযায়ী বদলান)
BASE = '/content/HCCD_Dataset'

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/  ({len(files)} files)')

In [ ]:
from torch.utils.data import DataLoader, random_split

# ==== আপনার আসল পাথ এখানে বসান (ধাপ ১-এর আউটপুট দেখে) ====
blur_folder  = '/content/HCCD_Dataset/Blur'
sharp_folder = '/content/HCCD_Dataset/Ground_Truth'
# ========================================================

from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(blur_folder, sharp_folder, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

batch_size = 8
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ============================================================
# STEP A: সব import + Dataset class + DataLoader একসাথে
# ============================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class DeblurDataset(Dataset):
    def __init__(self, blur_dir, sharp_dir, transform=None):
        self.blur_dir  = blur_dir
        self.sharp_dir = sharp_dir
        self.transform = transform

        blur_files  = set(os.listdir(blur_dir))
        sharp_files = set(os.listdir(sharp_dir))
        common = sorted(blur_files & sharp_files)

        # ---- debug info ----
        print("Blur files :", len(blur_files))
        print("Sharp files:", len(sharp_files))
        print("Common     :", len(common))
        print("Blur sample :", list(blur_files)[:3])
        print("Sharp sample:", list(sharp_files)[:3])
        # --------------------

        self.image_names = common
        if len(common) == 0:
            print("⚠️ কোনো ফাইলনেম মেলেনি! blur_folder / sharp_folder চেক করুন।")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        name = self.image_names[idx]
        blur  = Image.open(os.path.join(self.blur_dir,  name)).convert("RGB")
        sharp = Image.open(os.path.join(self.sharp_dir, name)).convert("RGB")
        if self.transform:
            blur  = self.transform(blur)
            sharp = self.transform(sharp)
        return blur, sharp


# ==== আপনার আসল পাথ ====
blur_folder  = '/content/HCCD_Dataset/Blur'
sharp_folder = '/content/HCCD_Dataset/Ground_Truth'
# =======================

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(blur_folder, sharp_folder, transform=transform)

if len(full_dataset) > 0:
    train_size = int(0.8 * len(full_dataset))
    val_size   = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

    batch_size = 8
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import os

print("=" * 50)
print("/content এর ভেতরে:")
print("=" * 50)
for f in os.listdir('/content'):
    print(" ", f)

print("\n" + "=" * 50)
print("Google Drive আছে কিনা:")
print("=" * 50)
if os.path.exists('/content/drive/MyDrive'):
    for f in os.listdir('/content/drive/MyDrive'):
        print(" ", f)
else:
    print("  Drive মাউন্ট করা নেই — drive.mount() রান করুন")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE = '/content/drive/MyDrive'
print("Drive-এর মূল ফোল্ডারে যা আছে:")
for f in os.listdir(BASE):
    full = os.path.join(BASE, f)
    tag = "📁" if os.path.isdir(full) else "📄"
    print(f"  {tag} {f}")

In [ ]:
import os

BASE = '/content/drive/MyDrive/Koushik_HCCD'

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/   ({len(files)} files)')
    if level == 2 and files:
        print(f'{indent}  sample: {files[:3]}')

In [ ]:
import os

BASE = '/content/drive/MyDrive/Koushik_HCCD'

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    if level > 5:          # আরও গভীরে যাই
        continue
    indent = '  ' * level
    n = len(files)
    print(f'{indent}{os.path.basename(root)}/   ({n} files)')
    if files and level >= 3:
        print(f'{indent}  sample: {files[:3]}')

In [ ]:
# ============================================================
# STEP A: Imports + Dataset (number-matching) + DataLoader
# ============================================================
import os, re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    """ফাইলনেম থেকে নাম্বার বের করে: 'Blur_level_1 (104).jpg' -> 104"""
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

# GT: {id: full_path}
gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None:
            gt_map[i] = os.path.join(GT_DIR, f)

# Blur: [(blur_path, gt_path), ...]
pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir):
        print("⚠️ নেই:", bdir); continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        i = extract_id(f)
        if i in gt_map:
            pairs.append((os.path.join(bdir, f), gt_map[i]))

print(f"GT ইমেজ       : {len(gt_map)}")
print(f"Blur↔GT জোড়া  : {len(pairs)}")
print("Sample জোড়া   :")
for b, g in pairs[:3]:
    print("   blur:", os.path.basename(b), "  ↔  gt:", os.path.basename(g))


class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp


transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
print("\nTotal pair:", len(full_dataset))

if len(full_dataset) > 0:
    train_size = int(0.8 * len(full_dataset))
    val_size   = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                    generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ============================================================
# STEP B: Model + Training Loop (এক সেলে)
# ============================================================
import torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

model     = UNetDeblur().to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

num_epochs = 20
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # ---- Val ----
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train L1: {tr:.4f} | Val L1: {va:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/best_deblur.pth')
        print(f"   ✅ Best saved (val={va:.4f})")

# লস গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('L1 Loss')
plt.title('Training vs Validation Loss')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('/content/best_deblur.pth'))
model.eval()

blur, sharp = next(iter(val_loader))
blur, sharp = blur.to(device), sharp.to(device)
with torch.no_grad():
    out = model(blur)

fig, ax = plt.subplots(3, 3, figsize=(11, 11))
for i in range(3):
    ax[i,0].imshow(blur[i].cpu().permute(1,2,0).clamp(0,1))
    ax[i,0].set_title("Blurry Input")
    ax[i,1].imshow(out[i].cpu().permute(1,2,0).clamp(0,1))
    ax[i,1].set_title("Model Deblurred")
    ax[i,2].imshow(sharp[i].cpu().permute(1,2,0).clamp(0,1))
    ax[i,2].set_title("Ground Truth")
    for j in range(3): ax[i,j].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
!cp /content/best_deblur.pth /content/drive/MyDrive/hccd_deblur_unet.pth
print("✅ Drive-এ সেভ হয়েছে: /content/drive/MyDrive/hccd_deblur_unet.pth")

In [ ]:
!pip install -q scikit-image
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

model.eval()
psnr_list, ssim_list = [], []
with torch.no_grad():
    for blur, sharp in tqdm(val_loader, desc="Evaluating"):
        blur, sharp = blur.to(device), sharp.to(device)
        out = model(blur).clamp(0,1).cpu().numpy()
        gt  = sharp.cpu().numpy()
        for i in range(out.shape[0]):
            o = np.transpose(out[i], (1,2,0))
            g = np.transpose(gt[i],  (1,2,0))
            psnr_list.append(psnr_fn(g, o, data_range=1.0))
            ssim_list.append(ssim_fn(g, o, data_range=1.0, channel_axis=2))

print(f"PSNR : {np.mean(psnr_list):.2f} dB  (বেশি = ভালো, >25 dB ভালো)")
print(f"SSIM : {np.mean(ssim_list):.4f}      (বেশি = ভালো, >0.85 খুব ভালো)")

In [ ]:
# ============================================================
# নিজের ছবিতে ডিব্লার (inference)
# ============================================================
from google.colab import files
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('/content/best_deblur.pth'))
model.eval()

print("একটি ব্লার ছবি আপলোড করুন:")
uploaded = files.upload()

for fname in uploaded:
    img = Image.open(fname).convert("RGB")
    orig_size = img.size  # (W, H)

    tf = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    inp = tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(inp).clamp(0, 1).cpu().squeeze(0).permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(img);       ax[0].set_title("Input (Blurry)");  ax[0].axis('off')
    ax[1].imshow(out);       ax[1].set_title("Deblurred Output"); ax[1].axis('off')
    plt.tight_layout(); plt.show()

    # ফলাফল সেভ
    Image.fromarray((out * 255).astype('uint8')).save(f"deblurred_{fname}")
    print(f"✅ সেভ: deblurred_{fname}")

In [ ]:
!cp /content/best_deblur.pth /content/drive/MyDrive/hccd_deblur_unet.pth
print("✅ /content/drive/MyDrive/hccd_deblur_unet.pth")

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np

# মডেলটি লোড করা (নিশ্চিত করুন best_deblur.pth সঠিক পাথে আছে)
model.load_state_dict(torch.load('/content/best_deblur.pth'))
model.eval()

# ইমেজ প্রসেসিং ফাংশন
def deblur_image(input_image):
    if input_image is None:
        return None

    # ইমেজকে টেনসরে রূপান্তর
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor()
    ])

    # মডেলে ইনপুট দেওয়ার জন্য প্রস্তুত করা
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    # মডেল রান
    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুটকে ইমেজে রূপান্তর
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_image = (output_image * 255).astype(np.uint8)

    return Image.fromarray(output_image)

# Gradio ইন্টারফেস তৈরি
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD Handwriting Deblurring",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। আমাদের U-Net ডিপ লার্নিং মডেল সেটিকে পরিষ্কার করে দেবে। (Blur Level 1, 2, 3 এর জন্য কাজ করে)",
    examples=[
        ["Blur_Level_3 (1).jpg"] # যদি আপনার Colab-এ এই ছবিটি থাকে, তাহলে উদাহরণ হিসেবে দেখাবে
    ],
    theme="soft" # সুন্দর একটি থিম
)

# অ্যাপটি লঞ্চ করুন (share=True দিলে পাবলিক লিংক পাবেন)
interface.launch(share=True, debug=False)

In [ ]:
import gradio as gr
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np

# মডেল লোড (যদি আগে না করা থাকে)
model.load_state_dict(torch.load('/content/best_deblur.pth'))
model.eval()

def deblur_image(input_image):
    if input_image is None:
        return None

    # ইনপুট ছবির আসল সাইজ সংরক্ষণ করা
    original_size = input_image.size  # (Width, Height)

    # মডেলে দেওয়ার জন্য 256x256 এ রিসাইজ
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor()
    ])

    input_tensor = transform(input_image).unsqueeze(0).to(device)

    # মডেল রান
    with torch.no_grad():
        output_tensor = model(input_tensor)

    # টেনসর থেকে ইমেজ তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_image = (output_image * 255).astype(np.uint8)
    output_pil = Image.fromarray(output_image)

    # আউটপুটটিকে আবার ইনপুট ছবির আসল সাইজে রিসাইজ করা
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    return output_pil

# Gradio ইন্টারফেস
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD Handwriting Deblurring",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। আমাদের U-Net ডিপ লার্নিং মডেল সেটিকে পরিষ্কার করে দেবে।",
    examples=[
        ["Blur_Level_3 (1).jpg"]
    ],
    theme="soft"
)

# share=True দিলে পাবলিক লিংক পাবেন
interface.launch(share=True, debug=True)

In [ ]:
from PIL import ImageEnhance, ImageFilter

def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # মডেলের ইনপুট ট্রান্সফর্ম
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor()
    ])

    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # টেনসর থেকে ইমেজ তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_image = (output_image * 255).astype(np.uint8)
    output_pil = Image.fromarray(output_image)

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ নতুন সংযোজন: Sharpness এবং Contrast একটু বাড়িয়ে দেওয়া
    enhancer = ImageEnhance.Sharpness(output_pil)
    output_pil = enhancer.enhance(2.0) # ২.০ মানে ডাবল শার্পনেস (প্রয়োজনে ১.৫ থেকে ৩.০ করা যাবে)

    contrast = ImageEnhance.Contrast(output_pil)
    output_pil = contrast.enhance(1.2) # কনট্রাস্ট একটু বাড়ানো হলো যাতে কালো লেখা ফুটে ওঠে

    return output_pil

In [ ]:
from PIL import ImageEnhance, ImageFilter
import numpy as np

def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # মডেলের জন্য ইনপুট রেডি
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor()
    ])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    # মডেল রান
    with torch.no_grad():
        output_tensor = model(input_tensor)

    # মডেল আউটপুটকে ছবিতে রূপান্তর
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # ১. আসল সাইজে রিসাইজ করা
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ২. মডেলের আউটপুটকে ইনপুট ছবির সাথে ২০% ব্লেন্ড করা (যাতে টেক্সট স্ট্রাকচার ঠিক থাকে)
    output_pil = Image.blend(input_image, output_pil, alpha=0.8)

    # ৩. শার্পনেস এবং কনট্রাস্ট অ্যাডজাস্ট করা
    enhancer_sharp = ImageEnhance.Sharpness(output_pil)
    output_pil = enhancer_sharp.enhance(2.5) # ২.৫ মানে জোরালো শার্পনেস

    enhancer_contrast = ImageEnhance.Contrast(output_pil)
    output_pil = enhancer_contrast.enhance(1.3)

    return output_pil

In [ ]:
!pip install -q pytorch-msssim
from pytorch_msssim import ssim

class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # ৮৫% L1 Loss (পিক্সেল ম্যাচিং) + ১৫% SSIM (স্ট্রাকচার ম্যাচিং)
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.85 * l1_loss + 0.15 * ssim_loss

criterion = CombinedLoss()

In [ ]:
!pip install -q pytorch-msssim

In [ ]:
# ============================================================
# STEP B2: Model + SSIM Loss + Training (এক সেলে)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim

# --- ১. মডেল ডিফাইন ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. Combined Loss (L1 + SSIM) ---
class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # ৮০% পিক্সেল ম্যাচিং + ২০% স্ট্রাকচার ম্যাচিং
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.8 * l1_loss + 0.2 * ssim_loss

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4) # একটু কম লার্নিং রেট
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready with SSIM Loss. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 20 # চাইলে বাড়াতে পারেন
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    # Train
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # Val
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # নতুন নামে সেভ করছি যাতে আগেরটা নষ্ট না হয়
        torch.save(model.state_dict(), '/content/best_deblur_ssim.pth')
        print(f"   ✅ Best saved (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1+SSIM)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
# ============================================================
# STEP B2 (Fixed): Device + Model + SSIM Loss + Training
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim

# --- ০. Device সেটআপ ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল ডিফাইন ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. Combined Loss (L1 + SSIM) ---
class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.8 * l1_loss + 0.2 * ssim_loss

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready with SSIM Loss. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 20
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    # Train
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # Val
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/best_deblur_ssim.pth')
        print(f"   ✅ Best saved (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1+SSIM)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# পাথ সেটআপ
BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None:
            gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map:
            pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ ডেটা লোড সফল! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# পাথ সেটআপ
BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None:
            gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map:
            pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ ডেটা লোড সফল! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
!pip install -q pytorch-msssim

In [ ]:
# ============================================================
# STEP B2 (Fixed): Device + Model + SSIM Loss + Training
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim

# --- ০. Device সেটআপ ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল ডিফাইন ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. Combined Loss (L1 + SSIM) ---
class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # ৮০% পিক্সেল ম্যাচিং + ২০% স্ট্রাকচার (Sharpness) ম্যাচিং
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.8 * l1_loss + 0.2 * ssim_loss

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready with SSIM Loss. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 20
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    # Train
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # Val
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/best_deblur_ssim.pth')
        print(f"   ✅ Best saved (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1+SSIM)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
!pip install pytorch-msssim

In [ ]:
import gradio as gr
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- মডেল লোড (আগের মডেল) ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/best_deblur.pth'))
model.eval()

def deblur_image(input_image):
    if input_image is None:
        return None
    original_size = input_image.size
    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ শার্পনেস ২.৫ থেকে বাড়িয়ে ৩.৫ করা হলো
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(3.5)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.3)

    return output_pil

# --- Gradio Interface ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD Handwriting Deblurring",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন।",
    theme="soft"
)
interface.launch(share=True, debug=False)

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেলের ক্লাস ডিফাইন ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. মডেল লোড ---
model = UNetDeblur().to(device)
# আপনার আগের ট্রেইন করা মডেলের পাথ
model.load_state_dict(torch.load('/content/best_deblur.pth', map_location=device))
model.eval()

# --- ৩. ডিব্লার ফাংশন (শার্পনেস সহ) ---
def deblur_image(input_image):
    if input_image is None:
        return None
    original_size = input_image.size
    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ শার্পনেস এবং কনট্রাস্ট অ্যাডজাস্ট করা
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(3.5)  # ৩.৫ মানে জোরালো শার্পনেস
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.3)   # কনট্রাস্ট একটু বাড়ানো

    return output_pil

# --- ৪. Gradio Interface লঞ্চ ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD Handwriting Deblurring",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। (Sharpness Enhanced)",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
# ============================================================
# SSIM ট্রেইনিং (Colab বন্ধ হলে আবার রান করতে হবে)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.8 * l1_loss + 0.2 * ssim_loss

model = UNetDeblur().to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

num_epochs = 20
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # Drive-এ সরাসরি সেভ করা হচ্ছে যাতে হারিয়ে না যায়!
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_ssim.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1+SSIM)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
# ============================================================
# STEP A: Drive Mount + Dataset + DataLoader
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None:
            gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map:
            pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ============================================================
# SSIM Loss + Training (Direct Save to Drive)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. Combined Loss (L1 + SSIM) ---
class CombinedLoss(nn.Module):
    def __init__(self):
        super(CombinedLoss, self).__init__()
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        l1_loss = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        return 0.8 * l1_loss + 0.2 * ssim_loss

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 20
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    # Train
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # Val
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # 🌟 সরাসরি Drive-এ সেভ করা হচ্ছে
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_ssim.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1+SSIM)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. নতুন SSIM মডেল লোড (Drive থেকে) ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()

# --- ৩. ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # মডেলে দেওয়ার জন্য প্রসেসিং
    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুট ইমেজ তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ হালকা শার্পনেস (SSIM মডেল নিজেই শার্প করে, তাই বেশি ফিল্টার দরকার নেই)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

    return output_pil

# --- ৪. Gradio Interface লঞ্চ ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output - SSIM Model)"),
    title="📝 HCCD Handwriting Deblurring (SSIM Enhanced)",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। (নতুন শার্প মডেল)",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import torch

# ১. টেস্ট করার জন্য একটি ছবি লোড করা (আপনার ডেটাসেট থেকে)
# আপনি চাইলে এখানে নিজের পিসি থেকে ছবি আপলোড করেও টেস্ট করতে পারেন
test_img_path = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3/Blur_Level_3 (1).jpg'
img = Image.open(test_img_path).convert("RGB")
original_size = img.size

# ২. মডেলে দেওয়ার জন্য প্রসেসিং
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

# ৩. মডেল রান
with torch.no_grad():
    output_tensor = model(input_tensor)

# ৪. আউটপুট তৈরি
output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ৫. পাশাপাশি দেখানো
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry Level 3)")
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output (SSIM Model)")
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import torch

# ১. ফোল্ডার থেকে একটি ফাইল খুঁজে নিন
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = os.listdir(blur_dir)
test_file = files[0]  # প্রথম ফাইলটি নিলাম
test_img_path = os.path.join(blur_dir, test_file)
print("টেস্ট ফাইল:", test_img_path)

# ২. ছবি লোড
img = Image.open(test_img_path).convert("RGB")
original_size = img.size

# ৩. মডেলে দেওয়ার জন্য প্রসেসিং
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

# ৪. মডেল রান
with torch.no_grad():
    output_tensor = model(input_tensor)

# ৫. আউটপুট তৈরি
output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ৬. পাশাপাশি দেখানো
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry Level 3)")
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output (SSIM Model)")
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ধাপে ধাপে ভেতরে যাওয়া
print(os.listdir('/content/drive/MyDrive/Koushik_HCCD'))
# যদি উপরে এরর দেয়, তবে /content/drive/MyDrive এ কী আছে দেখুন

In [ ]:
import os

# পুরো ডেটাসেট ফোল্ডারের ভেতরে খোঁজা
base = '/content/drive/MyDrive/Koushik_HCCD'

# ফোল্ডারের নাম চেক
for root, dirs, files in os.walk(base):
    if 'Blur' in root and 'Level_3' in root or 'level_3' in root.lower():
        print("✅ ফোল্ডার পাওয়া গেছে:", root)
        print("📁 ফাইল সংখ্যা:", len(files))
        print("📄 প্রথম ৩টি ফাইল:", files[:3])
        break

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import torch

# ⚠️ এখানে আপনার আসল পাথ বসান (ধাপ ২ থেকে পাওয়া)
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_level_3'

# ফাইল লিস্ট
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_file = files[0]
test_img_path = os.path.join(blur_dir, test_file)
print("টেস্ট ফাইল:", test_img_path)

# মডেল রান
img = Image.open(test_img_path).convert("RGB")
original_size = img.size
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ছবি দেখানো
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry)")
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output")
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os

# ড্রাইভের মূল ফোল্ডারটি দেখি
base_path = '/content/drive/MyDrive/Koushik_HCCD'

if os.path.exists(base_path):
    print("✅ Koushik_HCCD ফোল্ডার পাওয়া গেছে। এর ভেতরে যা আছে:")
    for item in os.listdir(base_path):
        print(" 📁 ", item)
else:
    print("❌ /content/drive/MyDrive/Koushik_HCCD পাওয়া যায়নি!")
    print("সম্ভবত ড্রাইভ মাউন্ট করা নেই। নিচের কমান্ডটি আগে রান করুন:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")

In [ ]:
import os

base = '/content/drive/MyDrive/Koushik_HCCD'
print("🔎 খোঁজা হচ্ছে...")

for root, dirs, files in os.walk(base):
    if 'Blur' in root and ('level_3' in root.lower() or 'Level_3' in root):
        print("\n✅ সঠিক ব্লার ফোল্ডার পাওয়া গেছে:")
        print("👉", root)
        print("📁 ফাইল সংখ্যা:", len(files))

        # পাথটি একটি ভেরিয়েবলে সেভ করে রাখি
        correct_blur_dir = root
        break

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import torch

# 👇 ধাপ ১ থেকে পাওয়া পাথটি এখানে বসান (কোটেশন মার্ক সহ)
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_level_3'

if os.path.exists(blur_dir):
    files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    test_file = files[0]
    test_img_path = os.path.join(blur_dir, test_file)
    print("✅ টেস্ট ফাইল:", test_file)

    # ছবি লোড ও প্রসেস
    img = Image.open(test_img_path).convert("RGB")
    original_size = img.size

    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(img).unsqueeze(0).to(device)

    # মডেল রান
    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুট তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ছবি দেখানো
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    ax[0].imshow(img)
    ax[0].set_title("Input (Blurry)")
    ax[0].axis('off')

    ax[1].imshow(output_pil)
    ax[1].set_title("Deblurred Output (SSIM Model)")
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("❌ পাথটি এখনো ভুল! দয়া করে ধাপ ১-এর আউটপুট থেকে সঠিক পাথটি কপি করুন।")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import torchvision.transforms as T
import torch

# ১. সঠিক পাথ
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'

# ২. ফাইল লিস্ট
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_file = files[0]
test_img_path = os.path.join(blur_dir, test_file)
print("✅ টেস্ট ফাইল:", test_file)

# ৩. ছবি লোড
img = Image.open(test_img_path).convert("RGB")
original_size = img.size

# ৪. প্রসেসিং
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

# ৫. মডেল রান
with torch.no_grad():
    output_tensor = model(input_tensor)

# ৬. আউটপুট তৈরি
output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ✨ হালকা শার্পনেস (SSIM মডেলের জন্য)
output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)

# ৭. ছবি দেখানো
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry Level 3)", fontsize=14)
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output (SSIM Model)", fontsize=14)
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

# --- ১. Device সেটআপ ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ২. মডেল আর্কিটেকচার ডিফাইন ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ৩. নতুন SSIM মডেল লোড করা (Drive থেকে) ---
model = UNetDeblur().to(device)
# আমাদের নতুন মডেলটি এখানে সেভ করা আছে
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()
print("✅ SSIM Model লোড সফল!")

# --- ৪. ছবি টেস্ট করা ---
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_file = files[0]
test_img_path = os.path.join(blur_dir, test_file)
print("🖼️ টেস্ট ফাইল:", test_file)

img = Image.open(test_img_path).convert("RGB")
original_size = img.size

# প্রসেসিং
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)
output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5) # হালকা শার্পনেস

# প্লট
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry Level 3)", fontsize=14)
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output (SSIM Model)", fontsize=14)
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np  # 👈 এটি মিসিং ছিল
from PIL import Image, ImageEnhance

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- মডেল লোড ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()
print("✅ SSIM Model লোড সফল!")

# --- ছবি টেস্ট ---
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_file = files[0]
test_img_path = os.path.join(blur_dir, test_file)
print("🖼️ টেস্ট ফাইল:", test_file)

img = Image.open(test_img_path).convert("RGB")
original_size = img.size

transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)
output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img)
ax[0].set_title("Input (Blurry Level 3)", fontsize=14)
ax[0].axis('off')

ax[1].imshow(output_pil)
ax[1].set_title("Deblurred Output (SSIM Model)", fontsize=14)
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- মডেল লোড ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()

# --- ছবি টেস্ট ---
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
img = Image.open(os.path.join(blur_dir, files[0])).convert("RGB")
original_size = img.size

transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ✨ এখানে শার্পনেস এবং ফিল্টার অনেক বাড়ানো হলো
output_pil = ImageEnhance.Sharpness(output_pil).enhance(3.5)  # ১.৫ থেকে বাড়িয়ে ৩.৫
output_pil = ImageEnhance.Contrast(output_pil).enhance(1.5)   # কনট্রাস্ট বাড়ানো
output_pil = output_pil.filter(ImageFilter.UnsharpMask(radius=2, percent=150, threshold=3)) # আল্ট্রা শার্প

# প্লট
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img); ax[0].set_title("Input (Blurry Level 3)"); ax[0].axis('off')
ax[1].imshow(output_pil); ax[1].set_title("Ultra-Sharpened Output"); ax[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

# ✨ এখানে 256 এর বদলে 512 ব্যবহার করা হলো
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# 512x512 হওয়ায় batch_size কমিয়ে 4 করা হলো (GPU মেমরি বাঁচাতে)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ============================================================
# STEP B3: 512x512 Training with Perceptual Loss (VGG)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # VGG16 এর প্রথম ১৬টি লেয়ার নিচ্ছি (ফিচার এক্সট্র্যাকশনের জন্য)
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # ১. পিক্সেল লস (L1)
        loss_l1 = self.l1(pred, target)
        # ২. স্ট্রাকচার লস (SSIM)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        # ৩. পার্সেপচুয়াল লস (VGG ফিচার)
        pred_features = self.vgg(pred)
        target_features = self.vgg(target)
        loss_perc = self.l1(pred_features, target_features)

        # মোট লস (গুরুত্ব অনুযায়ী ওয়েট দেওয়া)
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)  # লার্নিং রেট একটু কম
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 15
train_losses, val_losses = [], []  # 👈 লিস্ট ডিফাইন করা হলো
best_val = float('inf')

for epoch in range(num_epochs):
    # Train
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    # Val
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # Drive-এ নতুন নামে সেভ (যাতে আগের মডেল না হারায়)
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_512.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# --- ৫. লস গ্রাফ ---
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Total Loss')
plt.title('Training vs Validation (512x512 VGG)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
!pip install pytorch-msssim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

# ২৫৬ সাইজ
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss (মডেলকে শার্প করতে শেখাবে) ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        # ৫০% পিক্সেল ম্যাচিং + ৩০% স্ট্রাকচার + ২০% পার্সেপচুয়াল
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 10  # CPU-তে দ্রুত শেষ করতে ১০ দিচ্ছি (চাইলে বাড়াবেন)
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # Drive-এ নতুন নামে সেভ
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_vgg.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (VGG Perceptual)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# পাথ সেটআপ
BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

# ৫১২x৫১২ রেজোলিউশন
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# ৫১২ হওয়ায় batch_size=4 (GPU মেমরি বাঁচাতে)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 15
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # Drive-এ সেভ (যাতে Colab ডিসকানেক্ট হলেও থাকে)
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_512_vgg.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (512x512 VGG)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pytorch-msssim -q


In [ ]:
# ... (মডেল, লস, অপটিমাইজার ডিফাইন করার পর)

# ✨ এই লাইনটি যোগ করুন (সেভ করা মডেল লোড করতে)
start_epoch = 0
checkpoint_path = '/content/drive/MyDrive/best_deblur_512_vgg.pth'

if os.path.exists(checkpoint_path):
    print("🔄 আগের সেভ করা মডেল পাওয়া গেছে, সেখান থেকে শুরু হচ্ছে...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    start_epoch = 5 # ধরুন আগে ৫ epoch শেষ হয়েছিল, আপনি চাইলে বাড়াতে পারেন

# ... (এরপর ট্রেইনিং লুপ)
for epoch in range(start_epoch, num_epochs):
    # ... বাকি ট্রেইনিং কোড ...

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# ==========================================================
# 🔄 আগের সেভ করা মডেল লোড করা (যদি থাকে)
# ==========================================================
checkpoint_path = '/content/drive/MyDrive/best_deblur_512_vgg.pth'
start_epoch = 0

if os.path.exists(checkpoint_path):
    print("✅ Drive-এ আগের সেভ করা মডেল পাওয়া গেছে, সেটি লোড করা হচ্ছে...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    # আগে কত epoch শেষ হয়েছিল তা আপনি জানলে এখানে বসাতে পারেন, নাহলে ০ থেকে গুনবে
    # start_epoch = 5  # উদাহরণস্বরূপ
else:
    print("⚠️ Drive-এ আগের কোনো মডেল পাওয়া যায়নি, নতুন করে শুরু হচ্ছে।")

num_epochs = 15
train_losses, val_losses = [], []
best_val = float('inf')

# --- ৪. ট্রেইনিং লুপ ---
for epoch in range(start_epoch, num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_512_vgg.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (512x512 VGG)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. নতুন SSIM মডেল লোড (Drive থেকে) ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()

# --- ৩. ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # মডেলে দেওয়ার জন্য প্রসেসিং
    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুট ইমেজ তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ হালকা শার্পনেস (SSIM মডেল নিজেই শার্প করে, তাই বেশি ফিল্টার দরকার নেই)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

    return output_pil

# --- ৪. Gradio Interface লঞ্চ ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output - SSIM Model)"),
    title="📝 HCCD Handwriting Deblurring (SSIM Enhanced)",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। (নতুন শার্প মডেল)",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. নতুন SSIM মডেল লোড (Drive থেকে) ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_ssim.pth', map_location=device))
model.eval()

# --- ৩. ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # মডেলে দেওয়ার জন্য প্রসেসিং
    transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুট ইমেজ তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ হালকা শার্পনেস (SSIM মডেল নিজেই শার্প করে, তাই বেশি ফিল্টার দরকার নেই)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

    return output_pil

# --- ৪. Gradio Interface লঞ্চ ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output - SSIM Model)"),
    title="📝 HCCD Handwriting Deblurring (SSIM Enhanced)",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। (নতুন শার্প মডেল)",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import os

# ড্রাইভের ভেতরে Koushik_HCCD ফোল্ডারটি খোঁজা
base_check = '/content/drive/MyDrive/Koushik_HCCD'

if os.path.exists(base_check):
    print("✅ Koushik_HCCD ফোল্ডার পাওয়া গেছে। এর ভেতরে:")
    for item in os.listdir(base_check):
        print(" 📁", item)

    # আরও গভীরে চেক করা (HCCD DATASET এর জন্য)
    for root, dirs, files in os.walk(base_check):
        if 'Ground Truth' in root:
            print("\n🎯 সঠিক Ground Truth ফোল্ডার পাওয়া গেছে এখানে:")
            print(root)
            break
else:
    print("❌ /content/drive/MyDrive/Koushik_HCCD পাওয়া যায়নি!")
    print("এটি কি ড্রাইভের রুটে আছে? নিচের লিস্ট দেখুন:")
    print(os.listdir('/content/drive/MyDrive'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# 🔄 আগের সেভ করা মডেল লোড করা (যদি থাকে)
checkpoint_path = '/content/drive/MyDrive/best_deblur_512_vgg.pth'
start_epoch = 0

if os.path.exists(checkpoint_path):
    print("✅ Drive-এ আগের সেভ করা মডেল পাওয়া গেছে, সেটি লোড করা হচ্ছে...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    start_epoch = 0
else:
    print("⚠️ Drive-এ আগের কোনো মডেল পাওয়া যায়নি, নতুন করে শুরু হচ্ছে।")

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 15
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(start_epoch, num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_512_vgg.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (512x512 VGG)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import torch.nn as nn

# --- মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_512_vgg.pth', map_location=device))
model.eval()

# --- ছবি টেস্ট ---
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_img_path = os.path.join(blur_dir, files[0])

img = Image.open(test_img_path).convert("RGB")
original_size = img.size

transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# হালকা শার্পনেস (VGG মডেল নিজেই শার্প করে, তাই বেশি দরকার নেই)
output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img); ax[0].set_title("Input (Blurry Level 3)"); ax[0].axis('off')
ax[1].imshow(output_pil); ax[1].set_title("Deblurred (512x512 VGG)"); ax[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import torch.nn as nn

# --- মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDeblur().to(device)
# নতুন ৫১২x৫১২ মডেল লোড
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_512_vgg.pth', map_location=device))
model.eval()

# --- ছবি টেস্ট ---
blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
files = [f for f in os.listdir(blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_img_path = os.path.join(blur_dir, files[0])

img = Image.open(test_img_path).convert("RGB")
original_size = img.size

transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
input_tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output_tensor = model(input_tensor)

output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

# ✨ হালকা শার্পনেস
output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(img); ax[0].set_title("Input (Blurry Level 3)"); ax[0].axis('off')
ax[1].imshow(output_pil); ax[1].set_title("Deblurred (512x512 VGG)"); ax[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- নতুন মডেল লোড ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_512_vgg.pth', map_location=device))
model.eval()

# --- ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None
    original_size = input_image.size
    # ৫১২x৫১২ রিসাইজ
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # হালকা শার্পনেস ও কনট্রাস্ট
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)
    return output_pil

# --- Gradio Interface ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output - 512x512 VGG)"),
    title="📝 HCCD Handwriting Deblurring",
    description="আপনার হাতের লেখা বা যেকোনো ব্লার ছবি আপলোড করুন। নতুন ৫১২x৫১২ মডেল এটি পরিষ্কার করে দেবে।",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

model_path = '/content/drive/MyDrive/best_deblur_512_vgg.pth'
if os.path.exists(model_path):
    print("✅ মডেল ফাইলটি ড্রাইভে পাওয়া গেছে!")
else:
    print("❌ মডেল ফাইলটি পাওয়া যায়নি!")

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# পাথ (আপনার ড্রাইভের সঠিক পাথ অনুযায়ী)
BASE = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR    = os.path.join(BASE, 'Ground Truth')
BLUR_DIRS = [
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

pairs = []
for bdir in BLUR_DIRS:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: pairs.append((os.path.join(bdir, f), gt_map[i]))

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

# ✨ ১০২৪x১০২৪ রেজোলিউশন
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# মেমরি বাঁচাতে batch_size=1 অথবা 2 দিন
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from pytorch_msssim import ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm

model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4) # লার্নিং রেট একটু কম
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

num_epochs = 10 # ১০২৪ সাইজে ১০টি epoch দিলেই ভালো ফলাফল আসবে (সময় বেশি লাগবে)
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        # ড্রাইভে নতুন নামে সেভ
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_1024_vgg.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss (1024x1024 VGG)')
plt.legend(); plt.grid(); plt.show()

In [ ]:
import os

# Drive মাউন্ট (যদি আগে না করা থাকে)
from google.colab import drive
drive.mount('/content/drive')

# আপনার ডেটাসেটের নাম
dataset_name = 'saipl_1_majid_02082020-20261104T124352Z-1-006'

# ড্রাইভের মূল ফোল্ডারগুলোতে খোঁজা
base_dir = '/content/drive/MyDrive'
found_path = None

for root, dirs, files in os.walk(base_dir):
    if dataset_name in root:
        found_path = root
        print(f"✅ ডেটাসেট পাওয়া গেছে: {root}")
        # ভেতরের ফাইলগুলো দেখা
        for i, f in enumerate(os.listdir(root)[:10]): # প্রথম ১০টি ফাইল
            print(f"   📄 {f}")
        break

if not found_path:
    print("❌ ডেটাসেটটি Drive-এ পাওয়া যায়নি।")
    print("সম্ভবত এটি /content/ ফোল্ডারে আছে অথবা আপলোড করা হয়নি।")

    # /content/ চেক করা
    for item in os.listdir('/content'):
        if dataset_name in item:
            print(f"✅ /content/ এ পাওয়া গেছে: /content/{item}")

In [ ]:
import os

# Drive-এর মূল ফোল্ডারের সব ফাইল/ফোল্ডার দেখুন
print("📁 /content/drive/MyDrive এর ভেতরে যা আছে:\n")
for item in os.listdir('/content/drive/MyDrive'):
    full_path = os.path.join('/content/drive/MyDrive', item)
    if os.path.isdir(full_path):
        print(f"📁 [FOLDER] {item}")
    else:
        print(f"📄 [FILE]   {item}")

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/saipl_1_majid_02082020-20261104T124352Z-1-006.zip"
EXTRACT_DIR = "/content/drive/MyDrive/bangla_dataset_extracted"

# আনজিপ করা
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ আনজিপ সম্পন্ন!")
else:
    print("✅ ডেটাসেট আগেই আনজিপ করা আছে।")

# ভেতরের স্ট্রাকচার দেখুন
print("\n🔎 ডেটাসেটের ফোল্ডার স্ট্রাকচার:")
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 2: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/   ({len(files)} files)')
    if files and level <= 2:
        print(f'{indent}  sample: {files[:3]}')

In [ ]:
import cv2
from tqdm import tqdm

# 👇 ধাপ ১-এর আউটপুট দেখে আপনার আসল ইমেজের ফোল্ডার পাথটি এখানে বসান
SOURCE_IMG_DIR = "/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020-20261104T124352Z-1-006" # উদাহরণস্বরূপ
BLUR_DIR = "/content/drive/MyDrive/bangla_synthetic_blur"

if not os.path.exists(BLUR_DIR):
    os.makedirs(BLUR_DIR)

# ব্লারের মাত্রা (বিজোড় সংখ্যা হতে হবে: 9, 15, 21 ইত্যাদি। সংখ্যা বড় হলে ব্লার বেশি হবে)
BLUR_STRENGTH = 15

files = os.listdir(SOURCE_IMG_DIR)
valid_exts = ('.jpg', '.jpeg', '.png')
count = 0

for file in tqdm(files, desc="ব্লার তৈরি হচ্ছে"):
    if file.lower().endswith(valid_exts):
        img_path = os.path.join(SOURCE_IMG_DIR, file)
        img = cv2.imread(img_path)

        if img is not None:
            # গাউসিয়ান ব্লার তৈরি
            blurred = cv2.GaussianBlur(img, (BLUR_STRENGTH, BLUR_STRENGTH), 0)
            save_path = os.path.join(BLUR_DIR, file)
            cv2.imwrite(save_path, blurred)
            count += 1

print(f"✅ সফলভাবে {count} টি কৃত্রিম ব্লার ইমেজ তৈরি হয়েছে!")

In [ ]:
import os

EXTRACT_DIR = "/content/drive/MyDrive/bangla_dataset_extracted"

print("🔎 ডেটাসেটের আসল ফোল্ডার স্ট্রাকচার:")
found_dir = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 3: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/   ({len(files)} files)')
    if files and level <= 3:
        # প্রথম কয়েকটি ফাইল দেখানো হলো
        print(f'{indent}  sample: {files[:3]}')
        # ইমেজ ফাইল খোঁজা
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files) and found_dir is None:
            found_dir = root

In [ ]:
import os
import cv2
from tqdm import tqdm

BASE = "/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020"
BLUR_BASE = "/content/drive/MyDrive/bangla_synthetic_blur"

# সব সাব-ফোল্ডারের লিস্ট
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]

BLUR_STRENGTH = 15  # ব্লারের মাত্রা
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

total_count = 0

for sub in sub_folders:
    src_dir = os.path.join(BASE, sub)
    blur_dir = os.path.join(BLUR_BASE, sub)

    if not os.path.isdir(src_dir):
        print(f"⚠️ পাওয়া যায়নি: {src_dir}")
        continue

    os.makedirs(blur_dir, exist_ok=True)
    files = os.listdir(src_dir)

    for file in tqdm(files, desc=f"Blurring {sub[:30]}"):
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(src_dir, file)
            img = cv2.imread(img_path)
            if img is not None:
                blurred = cv2.GaussianBlur(img, (BLUR_STRENGTH, BLUR_STRENGTH), 0)
                # সেভ করার সময় .tif হলে সেটা রাখব, নাহলে .jpg
                save_path = os.path.join(blur_dir, file)
                cv2.imwrite(save_path, blurred)
                total_count += 1

print(f"\n✅ মোট {total_count} টি কৃত্রিম ব্লার ইমেজ তৈরি হয়েছে!")

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ===== ১. HCCD ডেটাসেট =====
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))

print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# ===== ২. বাংলা ডেটাসেট =====
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE, sub)
    blur_dir = os.path.join(BLUR_BASE, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            blur_p = os.path.join(blur_dir, f)
            gt_p   = os.path.join(src_dir, f)
            if os.path.exists(blur_p):
                bangla_pairs.append((blur_p, gt_p))

print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

# ===== ৩. দুটো একসাথে =====
combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

# ===== ৪. ডেটাসেট ক্লাস =====
class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 15
train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_deblur_combined.pth')
        print(f"   ✅ Best saved to Drive (val={va:.4f})")

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Combined HCCD + Bangla Training')
plt.legend(); plt.grid(); plt.show()

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import torch.nn as nn

# --- মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined.pth', map_location=device))
model.eval()
print("✅ Combined Model লোড সফল!")

def deblur_pipeline(img):
    original_size = img.size
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output_tensor = model(input_tensor)
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    return output_pil

# ===== ১. HCCD ছবিতে টেস্ট =====
hccd_blur_dir = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
hccd_files = sorted([f for f in os.listdir(hccd_blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
hccd_img = Image.open(os.path.join(hccd_blur_dir, hccd_files[0])).convert("RGB")
hccd_out = deblur_pipeline(hccd_img)

# ===== ২. বাংলা ছবিতে টেস্ট =====
bangla_blur_dir = '/content/drive/MyDrive/bangla_synthetic_blur/1. Camera/1. Essay'
bangla_files = sorted([f for f in os.listdir(bangla_blur_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))])
bangla_img = Image.open(os.path.join(bangla_blur_dir, bangla_files[0])).convert("RGB")
bangla_out = deblur_pipeline(bangla_img)

# ===== প্লট =====
fig, ax = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: HCCD
ax[0, 0].imshow(hccd_img); ax[0, 0].set_title("HCCD: Blur Level 3 Input")
ax[0, 1].imshow(hccd_out); ax[0, 1].set_title("HCCD: Deblurred Output")
gt_hccd_path = hccd_img  # আপনি চাইলে GT যোগ করতে পারেন
ax[0, 2].axis('off')

# Row 2: Bangla
ax[1, 0].imshow(bangla_img); ax[1, 0].set_title("Bangla: Blur Input")
ax[1, 1].imshow(bangla_out); ax[1, 1].set_title("Bangla: Deblurred Output")
ax[1, 2].axis('off')

for a in ax.flat:
    a.axis('off')

plt.suptitle("Combined Model (HCCD + Bangla) Test Results", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
model_path = '/content/drive/MyDrive/best_deblur_combined.pth'
if os.path.exists(model_path):
    print("✅ মডেল ফাইল পাওয়া গেছে!")
    print(f"Size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")
else:
    print("❌ মডেল ফাইল পাওয়া যায়নি!")
    print("Drive-এ যা আছে:")
    for f in os.listdir('/content/drive/MyDrive'):
        if f.endswith('.pth'):
            print(f"  📄 {f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ===== ১. HCCD ডেটাসেট =====
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))
print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# ===== ২. বাংলা ডেটাসেট =====
BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BN = '/content/drive/MyDrive/bangla_synthetic_blur'
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BN, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            bp = os.path.join(blur_dir, f)
            gp = os.path.join(src_dir, f)
            if os.path.exists(bp):
                bangla_pairs.append((bp, gp))
print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

# ===== ৩. একসাথে =====
combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

# ===== ৪. ডেটাসেট =====
class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import os

BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]

print("🔎 বাংলা ডেটাসেট স্ট্রাকচার চেক:\n")
for sub in sub_folders:
    full = os.path.join(BASE_BN, sub)
    if os.path.exists(full):
        files = os.listdir(full)
        img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]
        print(f"✅ {sub} → {len(img_files)} ছবি")
    else:
        print(f"❌ পাওয়া যায়নি: {full}")

In [ ]:
import os

BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'

# 1. Camera ফোল্ডারের সব ফাইল দেখি
for sub in ["1. Camera/1. Essay", "1. Camera/2. Isolated Character"]:
    full = os.path.join(BASE_BN, sub)
    if os.path.exists(full):
        files = os.listdir(full)
        print(f"\n📁 {sub} → {len(files)} ফাইল")
        print(f"   প্রথম ৫টি: {files[:5]}")

In [ ]:
import os
import cv2
from tqdm import tqdm

BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BASE = '/content/drive/MyDrive/bangla_synthetic_blur'

sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]

BLUR_STRENGTH = 15
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
total = 0

for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BASE, sub)
    if not os.path.isdir(src_dir):
        continue
    os.makedirs(blur_dir, exist_ok=True)

    files = os.listdir(src_dir)
    for file in tqdm(files, desc=f"Blur {sub[:25]}"):
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(src_dir, file)
            img = cv2.imread(img_path)
            if img is not None:
                blurred = cv2.GaussianBlur(img, (BLUR_STRENGTH, BLUR_STRENGTH), 0)
                save_path = os.path.join(blur_dir, file)
                cv2.imwrite(save_path, blurred)
                total += 1

print(f"\n✅ মোট {total} টি কৃত্রিম ব্লার তৈরি হয়েছে!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 12
train_losses, val_losses = [], []
best_val = float('inf')
SAVE_PATH = '/content/drive/MyDrive/best_deblur_combined.pth'
BEST_PATH = '/content/drive/MyDrive/best_deblur_combined_best.pth'

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    # প্রতি epoch-এ সেভ (নিরাপত্তার জন্য)
    torch.save(model.state_dict(), SAVE_PATH)
    print(f"   💾 Saved → {SAVE_PATH}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), BEST_PATH)
        print(f"   ✅ Best saved (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Combined HCCD + Bangla Training')
plt.legend(); plt.grid(); plt.show()

print(f"\n🎉 Training Done! Best Val Loss: {best_val:.4f}")

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ===== HCCD =====
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))
print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# ===== বাংলা =====
BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BN = '/content/drive/MyDrive/bangla_synthetic_blur'
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BN, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            bp = os.path.join(blur_dir, f)
            gp = os.path.join(src_dir, f)
            if os.path.exists(bp):
                bangla_pairs.append((bp, gp))
print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ===== HCCD =====
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))
print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# ===== বাংলা =====
BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BN = '/content/drive/MyDrive/bangla_synthetic_blur'
sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BN, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            bp = os.path.join(blur_dir, f)
            gp = os.path.join(src_dir, f)
            if os.path.exists(bp):
                bangla_pairs.append((bp, gp))
print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
import os

# সম্ভাব্য সব drive path চেক করি
possible_bases = [
    '/content/drive/MyDrive',
    '/content/gdrive/MyDrive',
    '/content/drive2/MyDrive',
]

for base in possible_bases:
    if os.path.isdir(base):
        print(f"\n✅ পাওয়া গেছে: {base}")
        print("   ভেতরে:", os.listdir(base)[:10])
    else:
        print(f"\n❌ নেই: {base}")

In [ ]:
import os

# সব drive path-এ Koushik_HCCD খুঁজি
for base in possible_bases:
    if not os.path.isdir(base): continue
    for item in os.listdir(base):
        full = os.path.join(base, item)
        if os.path.isdir(full) and 'koushik' in item.lower():
            print(f"\n🎯 HCCD পাওয়া গেছে: {full}")
            for sub in os.listdir(full):
                print(f"    → {sub}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

if os.path.isdir('/content/drive/MyDrive'):
    print("✅ Drive ঠিকমতো মাউন্ট হয়েছে!")
    print("মূল ফোল্ডারে যা আছে:")
    for item in os.listdir('/content/drive/MyDrive')[:10]:
        print(f"  - {item}")
else:
    print("❌ Drive মাউন্ট হয়নি। আবার ধাপ ১ চেষ্টা করুন।")

In [ ]:
import os
print("Drive আছে?", os.path.isdir('/content/drive/MyDrive'))
if os.path.isdir('/content/drive/MyDrive'):
    print("ফাইলগুলো:")
    for item in os.listdir('/content/drive/MyDrive'):
        print(f"  - {item}")

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/saipl_1_majid_02082020-20261104T124352Z-1-006.zip"
EXTRACT_DIR = "/content/drive/MyDrive/bangla_dataset_extracted"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ আনজিপ সম্পন্ন!")
else:
    print("✅ আগেই আনজিপ করা আছে")

# কাঠামো দেখুন
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 2: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

In [ ]:
import os
import cv2
from tqdm import tqdm

BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BASE = '/content/drive/MyDrive/bangla_synthetic_blur'

sub_folders = [
    "1. Camera/1. Essay",
    "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay",
    "2. Scan/2. Isolated Character",
    "3. Conjunct",
]

BLUR_STRENGTH = 15
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
total = 0

for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BASE, sub)
    if not os.path.isdir(src_dir):
        print(f"⚠️ নেই: {src_dir}")
        continue
    os.makedirs(blur_dir, exist_ok=True)
    files = os.listdir(src_dir)
    for file in tqdm(files, desc=f"Blur {sub[:25]}"):
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(src_dir, file)
            img = cv2.imread(img_path)
            if img is not None:
                blurred = cv2.GaussianBlur(img, (BLUR_STRENGTH, BLUR_STRENGTH), 0)
                cv2.imwrite(os.path.join(blur_dir, file), blurred)
                total += 1

print(f"\n✅ মোট {total} টি ব্লার তৈরি হয়েছে!")

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# HCCD
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))
print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# Bangla
BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BN = '/content/drive/MyDrive/bangla_synthetic_blur'
sub_folders = [
    "1. Camera/1. Essay", "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay", "2. Scan/2. Isolated Character",
    "3. Conjunct",
]
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BN, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            bp = os.path.join(blur_dir, f)
            gp = os.path.join(src_dir, f)
            if os.path.exists(bp):
                bangla_pairs.append((bp, gp))
print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# HCCD ফোল্ডার খোঁজা
base_drive = '/content/drive/MyDrive'
hccd_root = None

for root, dirs, files in os.walk(base_drive):
    if 'Ground Truth' in dirs:
        hccd_root = root
        print(f"✅ HCCD DATASET পাওয়া গেছে: {root}")
        break
    # খুব গভীরে না যাওয়া
    if root.count(os.sep) - base_drive.count(os.sep) > 4:
        dirs[:] = []
        continue

if hccd_root:
    gt_path = os.path.join(hccd_root, 'Ground Truth')
    print(f"📁 Ground Truth: {gt_path}")
    print(f"📄 ছবি সংখ্যা: {len([f for f in os.listdir(gt_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])}")
else:
    print("❌ HCCD DATASET পাওয়া যায়নি!")

In [ ]:
import os

base = '/content/drive/MyDrive'
print("Drive-এর ফাইল/ফোল্ডার:")
for item in os.listdir(base):
    full = os.path.join(base, item)
    tag = "📁" if os.path.isdir(full) else "📄"
    print(f"  {tag} {item}")

In [ ]:
import zipfile
import os

ZIP_PATH = '/content/drive/MyDrive/Koushik_HCCD.zip'
EXTRACT_DIR = '/content/drive/MyDrive/Koushik_HCCD_extracted'

if not os.path.exists(EXTRACT_DIR):
    print("আনজিপ হচ্ছে... (৫-১০ মিনিট লাগতে পারে)")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ আনজিপ সম্পন্ন!")
else:
    print("✅ আগেই আনজিপ করা আছে")

# স্ট্রাকচার দেখুন
print("\n🔎 HCCD স্ট্রাকচার:")
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 4: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

In [ ]:
import os, re
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ===== HCCD =====
BASE_HCCD = '/content/drive/MyDrive/Koushik_HCCD_extracted/Koushik_HCCD/HCCD DATASET'
GT_DIR = os.path.join(BASE_HCCD, 'Ground Truth')
BLUR_DIRS_HCCD = [
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_1'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_level_2'),
    os.path.join(BASE_HCCD, 'Synthetic', 'Blur', 'Blur_Level_3'),
]

print("GT_DIR আছে?", os.path.isdir(GT_DIR))

def extract_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else None

gt_map = {}
for f in os.listdir(GT_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        i = extract_id(f)
        if i is not None: gt_map[i] = os.path.join(GT_DIR, f)

hccd_pairs = []
for bdir in BLUR_DIRS_HCCD:
    if not os.path.isdir(bdir): continue
    for f in os.listdir(bdir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        i = extract_id(f)
        if i in gt_map: hccd_pairs.append((os.path.join(bdir, f), gt_map[i]))
print(f"✅ HCCD: {len(hccd_pairs)} পেয়ার")

# ===== বাংলা =====
BASE_BN = '/content/drive/MyDrive/bangla_dataset_extracted/saipl_1_majid_02082020'
BLUR_BN = '/content/drive/MyDrive/bangla_synthetic_blur'
sub_folders = [
    "1. Camera/1. Essay", "1. Camera/2. Isolated Character",
    "2. Scan/1. Essay", "2. Scan/2. Isolated Character",
    "3. Conjunct",
]
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')

bangla_pairs = []
for sub in sub_folders:
    src_dir = os.path.join(BASE_BN, sub)
    blur_dir = os.path.join(BLUR_BN, sub)
    if not os.path.isdir(src_dir): continue
    for f in os.listdir(src_dir):
        if f.lower().endswith(valid_exts):
            bp = os.path.join(blur_dir, f)
            gp = os.path.join(src_dir, f)
            if os.path.exists(bp):
                bangla_pairs.append((bp, gp))
print(f"✅ বাংলা: {len(bangla_pairs)} পেয়ার")

combined_pairs = hccd_pairs + bangla_pairs
print(f"🎯 মোট: {len(combined_pairs)} পেয়ার")

class DeblurDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        b, g = self.pairs[idx]
        blur  = Image.open(b).convert("RGB")
        sharp = Image.open(g).convert("RGB")
        if self.transform:
            blur, sharp = self.transform(blur), self.transform(sharp)
        return blur, sharp

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

full_dataset = DeblurDataset(combined_pairs, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ ডেটা রেডি! Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
!pip install pytorch-msssim -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. VGG Perceptual Loss ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.l1 = nn.L1Loss()
    def forward(self, pred, target):
        loss_l1 = self.l1(pred, target)
        loss_ssim = 1 - ssim(pred, target, data_range=1.0)
        loss_perc = self.l1(self.vgg(pred), self.vgg(target))
        return 0.5 * loss_l1 + 0.3 * loss_ssim + 0.2 * loss_perc

# --- ৩. সেটআপ ---
model = UNetDeblur().to(device)
criterion = VGGPerceptualLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Model ready. Params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

# --- ৪. ট্রেইনিং লুপ ---
num_epochs = 12
train_losses, val_losses = [], []
best_val = float('inf')
SAVE_PATH = '/content/drive/MyDrive/best_deblur_combined.pth'
BEST_PATH = '/content/drive/MyDrive/best_deblur_combined_best.pth'

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for blur, sharp in tqdm(train_loader, desc=f"Ep{epoch+1}/{num_epochs} Train"):
        blur, sharp = blur.to(device), sharp.to(device)
        optimizer.zero_grad()
        loss = criterion(model(blur), sharp)
        loss.backward()
        optimizer.step()
        running += loss.item()
    tr = running / len(train_loader)
    train_losses.append(tr)

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for blur, sharp in tqdm(val_loader, desc=f"Ep{epoch+1}/{num_epochs} Val  "):
            blur, sharp = blur.to(device), sharp.to(device)
            vl += criterion(model(blur), sharp).item()
    va = vl / len(val_loader)
    val_losses.append(va)
    scheduler.step(va)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr:.4f} | Val Loss: {va:.4f}")

    # প্রতি epoch-এ সেভ
    torch.save(model.state_dict(), SAVE_PATH)
    print(f"   💾 Saved → {SAVE_PATH}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), BEST_PATH)
        print(f"   ✅ Best saved (val={va:.4f})")

# গ্রাফ
plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Combined HCCD + Bangla Training')
plt.legend(); plt.grid(); plt.show()

print(f"\n🎉 Training Done! Best Val Loss: {best_val:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model loaded!")

def deblur_pipeline(img):
    original_size = img.size
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output_tensor = model(input_tensor)
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    return output_pil

# HCCD ছবি টেস্ট
hccd_dir = '/content/drive/MyDrive/Koushik_HCCD_extracted/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
hccd_files = sorted([f for f in os.listdir(hccd_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
hccd_img = Image.open(os.path.join(hccd_dir, hccd_files[0])).convert("RGB")
hccd_out = deblur_pipeline(hccd_img)

# বাংলা ছবি টেস্ট
bn_dir = '/content/drive/MyDrive/bangla_synthetic_blur/2. Scan/1. Essay'
bn_files = sorted([f for f in os.listdir(bn_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))])
bn_img = Image.open(os.path.join(bn_dir, bn_files[0])).convert("RGB")
bn_out = deblur_pipeline(bn_img)

# প্লট
fig, ax = plt.subplots(2, 2, figsize=(14, 10))
ax[0,0].imshow(hccd_img); ax[0,0].set_title("HCCD Blur Input")
ax[0,1].imshow(hccd_out); ax[0,1].set_title("HCCD Deblurred (Combined Model)")
ax[1,0].imshow(bn_img);   ax[1,0].set_title("Bangla Blur Input")
ax[1,1].imshow(bn_out);   ax[1,1].set_title("Bangla Deblurred (Combined Model)")
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import os
for f in ['best_deblur_combined.pth', 'best_deblur_combined_best.pth']:
    path = f'/content/drive/MyDrive/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"✅ {f} — {size:.1f} MB")
    else:
        print(f"❌ {f} — পাওয়া যায়নি")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model loaded!")

def deblur_pipeline(img):
    original_size = img.size
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output_tensor = model(input_tensor)
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.5)
    return output_pil

# HCCD ছবি টেস্ট
hccd_dir = '/content/drive/MyDrive/Koushik_HCCD_extracted/Koushik_HCCD/HCCD DATASET/Synthetic/Blur/Blur_Level_3'
hccd_files = sorted([f for f in os.listdir(hccd_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
hccd_img = Image.open(os.path.join(hccd_dir, hccd_files[0])).convert("RGB")
hccd_out = deblur_pipeline(hccd_img)

# বাংলা ছবি টেস্ট
bn_dir = '/content/drive/MyDrive/bangla_synthetic_blur/2. Scan/1. Essay'
bn_files = sorted([f for f in os.listdir(bn_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))])
bn_img = Image.open(os.path.join(bn_dir, bn_files[0])).convert("RGB")
bn_out = deblur_pipeline(bn_img)

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
ax[0,0].imshow(hccd_img); ax[0,0].set_title("HCCD: Blur Input")
ax[0,1].imshow(hccd_out); ax[0,1].set_title("HCCD: Deblurred")
ax[1,0].imshow(bn_img);   ax[1,0].set_title("Bangla: Blur Input")
ax[1,1].imshow(bn_out);   ax[1,1].set_title("Bangla: Deblurred")
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
!pip install gradio -q

import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. Combined মডেল লোড ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model লোড সফল!")

# --- ৩. ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None

    # ইনপুট ছবির আসল সাইজ সংরক্ষণ
    original_size = input_image.size

    # ৫১২x৫১২ রিসাইজ (মডেল যেভাবে ট্রেইন হয়েছে)
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # আউটপুট তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফেরানো
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ হালকা শার্পনেস (ছবি ক্রিস্টাল ক্লিয়ার দেখানোর জন্য)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(2.0)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

    return output_pil

# --- ৪. Gradio Interface ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output - HCCD + Bangla)"),
    title="📝 HCCD + Bangla Handwriting Deblurring",
    description="আপনার হাতের লেখা (ইংরেজি বা বাংলা) বা যেকোনো ব্লার ছবি আপলোড করুন। আমাদের Combined U-Net মডেল সেটিকে পরিষ্কার করে দেবে।",
    theme="soft"
)

# পাবলিক লিংক তৈরি (share=True)
interface.launch(share=True, debug=False)

In [ ]:
output_pil = ImageEnhance.Sharpness(output_pil).enhance(2.0)

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# Combined মডেল লোড
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model লোড সফল!")

def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ শার্পনেস ৩.০ করা হলো
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(3.0)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.2)

    return output_pil

interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD + Bangla Handwriting Deblurring",
    description="আপনার হাতের লেখা বা ব্লার ছবি আপলোড করুন। Combined মডেল এটি পরিষ্কার করে দেবে।",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. মডেল লোড ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model লোড সফল!")

# --- ৩. Tiling সহ ডিব্লার ফাংশন ---
def deblur_image(input_image):
    if input_image is None:
        return None

    img = input_image.convert("RGB")
    w, h = img.size
    patch_size = 512
    overlap = 64 # প্যাচগুলোর মাঝে ওভারল্যাপ, যাতে জোড়া লাগার দাগ না পড়ে

    # আউটপুট ক্যানভাস তৈরি
    output_img = Image.new("RGB", (w, h))
    transform = T.Compose([T.Resize((patch_size, patch_size)), T.ToTensor()])

    # ইমেজকে প্যাচে ভাগ করা
    for y in range(0, h, patch_size - overlap):
        for x in range(0, w, patch_size - overlap):
            right = min(x + patch_size, w)
            bottom = min(y + patch_size, h)
            left = max(right - patch_size, 0)
            top = max(bottom - patch_size, 0)

            patch = img.crop((left, top, right, bottom))

            # মডেলে ইনপুট
            input_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                output_tensor = model(input_tensor)

            # আউটপুট প্যাচ তৈরি
            out_np = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
            out_pil = Image.fromarray((out_np * 255).astype(np.uint8))
            out_pil = out_pil.resize((right - left, bottom - top), Image.Resampling.LANCZOS)

            # ক্যানভাসে বসানো
            output_img.paste(out_pil, (left, top))

    # ✨ পোস্ট-প্রসেসিং (শার্পনেস ও কনট্রাস্ট)
    output_img = ImageEnhance.Sharpness(output_img).enhance(2.5)
    output_img = ImageEnhance.Contrast(output_img).enhance(1.2)

    return output_img

# --- ৪. Gradio Interface ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="HD ডিব্লার করা ছবি (Tiling Applied)"),
    title="📝 HCCD + Bangla Handwriting Deblurring (HD Mode)",
    description="Tiling পদ্ধতি ব্যবহার করে ছবির প্রতিটি অংশ আলাদাভাবে প্রসেস করা হচ্ছে, যাতে HD মানের শার্পনেস পাওয়া যায়।",
    theme="soft"
)

interface.launch(share=True, debug=False)

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageEnhance
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ১. মডেল আর্কিটেকচার ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class UNetDeblur(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = DoubleConv(in_c, 64);   self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128);    self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256);   self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2); self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2); self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64,  2, 2); self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_c, 1); self.sig = nn.Sigmoid()
    def forward(self, x):
        d1 = self.d1(x); p1 = self.p1(d1)
        d2 = self.d2(p1); p2 = self.p2(d2)
        d3 = self.d3(p2); p3 = self.p3(d3)
        b  = self.bn(p3)
        c3 = self.c3(torch.cat([self.u3(b),  d3], 1))
        c2 = self.c2(torch.cat([self.u2(c3), d2], 1))
        c1 = self.c1(torch.cat([self.u1(c2), d1], 1))
        return self.sig(self.out(c1))

# --- ২. মডেল লোড ---
model = UNetDeblur().to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/best_deblur_combined_best.pth', map_location=device))
model.eval()
print("✅ Combined Model লোড সফল!")

# --- ৩. ডিব্লার ফাংশন (অরিজিনাল পদ্ধতি) ---
def deblur_image(input_image):
    if input_image is None:
        return None

    original_size = input_image.size

    # ছবিটিকে ৫১২x৫১২ তে রিসাইজ করা হচ্ছে (মডেল যেভাবে ট্রেইন হয়েছে)
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor()])
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    # মডেলের আউটপুট থেকে ছবি তৈরি
    output_image = output_tensor.squeeze(0).cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    output_pil = Image.fromarray((output_image * 255).astype(np.uint8))

    # আসল সাইজে ফিরিয়ে আনা
    output_pil = output_pil.resize(original_size, Image.Resampling.LANCZOS)

    # ✨ নিখুঁত শার্পনেস (ছবি ঘোলা না করে শুধু এজগুলো স্পষ্ট করবে)
    output_pil = ImageEnhance.Sharpness(output_pil).enhance(1.8)
    output_pil = ImageEnhance.Contrast(output_pil).enhance(1.1)

    return output_pil

# --- ৪. Gradio Interface ---
interface = gr.Interface(
    fn=deblur_image,
    inputs=gr.Image(type="pil", label="ব্লার ছবি আপলোড করুন (Upload Blurry Image)"),
    outputs=gr.Image(type="pil", label="ডিব্লার করা ছবি (Deblurred Output)"),
    title="📝 HCCD + Bangla Handwriting Deblurring",
    description="আপনার হাতের লেখা বা ব্লার ছবি আপলোড করুন। Combined মডেল এটি পরিষ্কার করে দেবে।",
    theme="soft"
)

interface.launch(share=True, debug=False)